# HashedNet95 KWS Lab

Fast experiment notebook for the HashedNets-inspired ESP32 KWS branch.

Main levers:
- exact firmware microfrontend;
- direct Speech Commands preparation under the runtime project data folder;
- persistent Google Drive cache for exact features and run artifacts;
- signed hashing;
- per-layer codebook budgets;
- virtual-width inflation up to the current ESP32 runtime limit;
- optional HashNetDK-style teacher distillation.


In [ ]:
# The next cell writes the minimal runtime files into /content first.
# If Colab is missing packages, run after bootstrap:
# !pip -q install -r /content/diploma_esp32_distributed_nn/code/training/requirements-kws-hash-exact-frontend.txt


In [ ]:
import importlib
import json
import os
import sys
from pathlib import Path

FORCE_SYNC_RUNTIME_FILES = True
USE_GOOGLE_DRIVE_CACHE = True
CACHE_SPEECHCOMMANDS_ON_DRIVE = False
DRIVE_CACHE_ROOT = Path("/content/drive/MyDrive/diploma_kws_cache/hashednet95")
SELECTED_RECIPE = "hn95_kd128_cached_schedule"
TEACHER_CHECKPOINT_PATH = ""  # optional: /content/.../teacher_best.pt
FORCE_TEACHER_RETRAIN = False
SMOKE_MODE = False  # keeps architecture intact, only limits data for syntax/debug runs

if USE_GOOGLE_DRIVE_CACHE and Path("/content").exists():
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    except Exception as exc:
        print("Drive mount skipped:", exc)
DRIVE_CACHE_ACTIVE = USE_GOOGLE_DRIVE_CACHE and (Path("/content/drive/MyDrive").exists())

BASE_RUNTIME_DIR = Path("/content") if Path("/content").exists() else Path.cwd()
PROJECT_ROOT = BASE_RUNTIME_DIR / "diploma_esp32_distributed_nn"

TRAINING_ROOT = PROJECT_ROOT / "code" / "training"
SCRIPTS_ROOT = PROJECT_ROOT / "code" / "scripts"
HASHEDNET95_ROOT = TRAINING_ROOT / "hashednet95"
FILE_PAYLOADS = json.loads('{"code/training/requirements-kws-hash.txt": "torch>=2.2\\ntorchaudio>=2.2\\nmatplotlib>=3.8,<4\\n", "code/training/requirements-kws-hash-exact-frontend.txt": "numpy<2\\ntorch>=2.2\\ntorchaudio>=2.2\\nmatplotlib>=3.8,<4\\ntensorflow>=2.15,<3\\n", "code/training/hash_kws_lab/__init__.py": "from .config import ExperimentConfig, make_experiment\\nfrom .recipes import build_recipe_book\\n\\n__all__ = [\\n    \\"ExperimentConfig\\",\\n    \\"build_recipe_book\\",\\n    \\"make_experiment\\",\\n]\\n", "code/training/hash_kws_lab/config.py": "from __future__ import annotations\\n\\nfrom dataclasses import asdict, dataclass, field\\nfrom typing import Any\\n\\n\\nVOCABULARY_PRESETS: dict[str, list[str]] = {\\n    \\"kws12\\": [\\n        \\"yes\\",\\n        \\"no\\",\\n        \\"up\\",\\n        \\"down\\",\\n        \\"left\\",\\n        \\"right\\",\\n        \\"on\\",\\n        \\"off\\",\\n        \\"stop\\",\\n        \\"go\\",\\n    ],\\n}\\n\\n\\n@dataclass(frozen=True)\\nclass FeatureConfig:\\n    sample_rate: int = 16_000\\n    clip_samples: int = 16_000\\n    window_ms: int = 30\\n    hop_ms: int = 20\\n    n_mels: int = 40\\n    center: bool = False\\n    frontend_name: str = \\"log_mel\\"\\n    normalize_mode: str = \\"instance\\"\\n    log_offset: float = 1e-6\\n    require_exact_microfrontend: bool = False\\n    lower_band_limit: float = 125.0\\n    upper_band_limit: float = 7_500.0\\n    smoothing_bits: int = 10\\n    even_smoothing: float = 0.025\\n    odd_smoothing: float = 0.06\\n    min_signal_remaining: float = 0.05\\n    enable_pcan: bool = True\\n    pcan_strength: float = 0.95\\n    pcan_offset: float = 80.0\\n    gain_bits: int = 21\\n    enable_log: bool = True\\n    scale_shift: int = 6\\n    exact_value_divisor: float = 665.6\\n    specaugment_prob: float = 0.0\\n    time_mask_max: int = 0\\n    freq_mask_max: int = 0\\n\\n    @property\\n    def window_samples(self) -> int:\\n        return (self.sample_rate * self.window_ms) // 1000\\n\\n    @property\\n    def hop_samples(self) -> int:\\n        return (self.sample_rate * self.hop_ms) // 1000\\n\\n    @property\\n    def frame_count(self) -> int:\\n        usable = max(0, self.clip_samples - self.window_samples)\\n        return 1 + (usable // self.hop_samples)\\n\\n    @property\\n    def feature_shape(self) -> tuple[int, int]:\\n        return (self.n_mels, self.frame_count)\\n\\n\\n@dataclass(frozen=True)\\nclass DatasetConfig:\\n    data_dir: str = \\"data\\"\\n    batch_size: int = 256\\n    num_workers: int = 0\\n    persistent_workers: bool = True\\n    prefetch_factor: int = 2\\n    seed: int = 13\\n    unknown_fraction: float = 1.0\\n    silence_fraction: float = 0.12\\n    silence_reference: str = \\"known\\"\\n    train_limit: int | None = None\\n    val_limit: int | None = None\\n    test_limit: int | None = None\\n    time_shift_ms: int = 0\\n    gain_min: float = 1.0\\n    gain_max: float = 1.0\\n    noise_stddev: float = 0.0\\n    cache_features: bool = False\\n    cache_dtype: str = \\"int8\\"\\n\\n\\n@dataclass(frozen=True)\\nclass ModelConfig:\\n    teacher_name: str = \\"\\"\\n    student_name: str = \\"hash_dscnn_deeper\\"\\n    channels: int = 64\\n    teacher_channels: int = 96\\n    num_blocks: int = 4\\n    teacher_num_blocks: int = 5\\n    codebook_size: int = 500\\n    stem_codebook_size: int = 500\\n    depthwise_codebook_size: int = 500\\n    pointwise_codebook_size: int = 500\\n    linear_codebook_size: int = 500\\n    depthwise_codebook_sizes: tuple[int, ...] = ()\\n    pointwise_codebook_sizes: tuple[int, ...] = ()\\n    signed_hash: bool = False\\n    hash_only_pointwise: bool = False\\n    use_residual: bool = False\\n    teacher_dropout: float = 0.10\\n    student_dropout: float = 0.0\\n\\n\\n@dataclass(frozen=True)\\nclass TrainConfig:\\n    seed: int = 13\\n    teacher_epochs: int = 18\\n    student_pretrain_epochs: int = 0\\n    student_epochs: int = 20\\n    student_polish_epochs: int = 0\\n    teacher_lr: float = 1e-3\\n    student_lr: float = 1e-3\\n    polish_lr: float = 2.5e-4\\n    weight_decay: float = 1e-4\\n    optimizer_name: str = \\"adamw\\"\\n    teacher_scheduler_name: str = \\"none\\"\\n    student_scheduler_name: str = \\"none\\"\\n    label_smoothing: float = 0.0\\n    teacher_label_smoothing: float = 0.0\\n    kd_alpha: float = 0.0\\n    kd_alpha_schedule: str = \\"constant\\"\\n    kd_alpha_final: float | None = None\\n    kd_temperature: float = 4.0\\n    kd_temperature_schedule: str = \\"constant\\"\\n    kd_temperature_final: float | None = None\\n    cache_teacher_logits: bool = False\\n    teacher_logits_cache_dir: str = \\"\\"\\n    teacher_logits_cache_dtype: str = \\"float16\\"\\n    teacher_logits_cache_rebuild: bool = False\\n    polish_label_smoothing: float | None = None\\n    grad_clip_norm: float = 0.0\\n    teacher_early_stopping_patience: int = 0\\n    student_early_stopping_patience: int = 0\\n    use_amp: bool = True\\n    use_ema: bool = False\\n    ema_decay: float = 0.999\\n    eval_with_ema: bool = True\\n    top_k: int = 3\\n\\n    @property\\n    def uses_distillation(self) -> bool:\\n        return self.kd_alpha > 0.0\\n\\n\\n@dataclass(frozen=True)\\nclass ExportConfig:\\n    artifacts_dir: str = \\"code/training/hash_artifacts\\"\\n    model_stem: str = \\"hash_kws_student\\"\\n\\n\\n@dataclass(frozen=True)\\nclass ExperimentConfig:\\n    tag: str\\n    vocabulary_preset: str\\n    teacher_reuse_tag: str = \\"\\"\\n    feature: FeatureConfig = field(default_factory=FeatureConfig)\\n    dataset: DatasetConfig = field(default_factory=DatasetConfig)\\n    model: ModelConfig = field(default_factory=ModelConfig)\\n    train: TrainConfig = field(default_factory=TrainConfig)\\n    export: ExportConfig = field(default_factory=ExportConfig)\\n\\n    @property\\n    def commands(self) -> list[str]:\\n        if self.vocabulary_preset not in VOCABULARY_PRESETS:\\n            raise KeyError(f\\"Unknown vocabulary preset: {self.vocabulary_preset}\\")\\n        return list(VOCABULARY_PRESETS[self.vocabulary_preset])\\n\\n    @property\\n    def all_labels(self) -> list[str]:\\n        return [*self.commands, \\"unknown\\", \\"silence\\"]\\n\\n    @property\\n    def label_to_index(self) -> dict[str, int]:\\n        return {label: index for index, label in enumerate(self.all_labels)}\\n\\n    @property\\n    def num_labels(self) -> int:\\n        return len(self.all_labels)\\n\\n    @property\\n    def feature_shape(self) -> tuple[int, int]:\\n        return self.feature.feature_shape\\n\\n    @property\\n    def model_input_shape(self) -> tuple[int, int, int]:\\n        mel_bins, frames = self.feature_shape\\n        return (1, mel_bins, frames)\\n\\n    @property\\n    def uses_teacher(self) -> bool:\\n        return bool(self.model.teacher_name)\\n\\n    def to_dict(self) -> dict[str, Any]:\\n        payload = asdict(self)\\n        payload[\\"commands\\"] = self.commands\\n        payload[\\"all_labels\\"] = self.all_labels\\n        payload[\\"num_labels\\"] = self.num_labels\\n        payload[\\"feature_shape\\"] = list(self.feature_shape)\\n        payload[\\"model_input_shape\\"] = list(self.model_input_shape)\\n        payload[\\"uses_teacher\\"] = self.uses_teacher\\n        payload[\\"uses_distillation\\"] = self.train.uses_distillation\\n        return payload\\n\\n\\ndef make_experiment(\\n    tag: str = \\"hash_kws12_iterlab_v1\\",\\n    vocabulary_preset: str = \\"kws12\\",\\n) -> ExperimentConfig:\\n    return ExperimentConfig(\\n        tag=tag,\\n        vocabulary_preset=vocabulary_preset,\\n    )\\n\\n\\ndef experiment_from_dict(payload: dict[str, Any]) -> ExperimentConfig:\\n    return ExperimentConfig(\\n        tag=str(payload[\\"tag\\"]),\\n        vocabulary_preset=str(payload[\\"vocabulary_preset\\"]),\\n        teacher_reuse_tag=str(payload.get(\\"teacher_reuse_tag\\", \\"\\")),\\n        feature=FeatureConfig(**payload.get(\\"feature\\", {})),\\n        dataset=DatasetConfig(**payload.get(\\"dataset\\", {})),\\n        model=ModelConfig(**payload.get(\\"model\\", {})),\\n        train=TrainConfig(**payload.get(\\"train\\", {})),\\n        export=ExportConfig(**payload.get(\\"export\\", {})),\\n    )\\n", "code/training/hash_kws_lab/recipes.py": "from __future__ import annotations\\n\\nfrom dataclasses import replace\\n\\nfrom .config import ExperimentConfig\\n\\n\\ndef build_recipe_book(base: ExperimentConfig) -> dict[str, ExperimentConfig]:\\n    hash_deeper_usermix_ce = replace(\\n        base,\\n        tag=f\\"{base.tag}_hash_deeper_usermix_ce\\",\\n        dataset=replace(\\n            base.dataset,\\n            unknown_fraction=0.10,\\n            silence_fraction=0.10,\\n            silence_reference=\\"known\\",\\n            time_shift_ms=0,\\n            gain_min=1.0,\\n            gain_max=1.0,\\n            noise_stddev=0.0,\\n        ),\\n        feature=replace(\\n            base.feature,\\n            specaugment_prob=0.0,\\n            time_mask_max=0,\\n            freq_mask_max=0,\\n        ),\\n        model=replace(\\n            base.model,\\n            teacher_name=\\"\\",\\n            student_name=\\"hash_dscnn_deeper\\",\\n            channels=64,\\n            num_blocks=4,\\n            codebook_size=500,\\n            stem_codebook_size=500,\\n            depthwise_codebook_size=500,\\n            pointwise_codebook_size=500,\\n            linear_codebook_size=500,\\n            signed_hash=False,\\n            hash_only_pointwise=False,\\n        ),\\n        train=replace(\\n            base.train,\\n            teacher_epochs=0,\\n            student_pretrain_epochs=0,\\n            student_epochs=20,\\n            student_polish_epochs=0,\\n            student_lr=1e-3,\\n            weight_decay=1e-4,\\n            student_scheduler_name=\\"none\\",\\n            label_smoothing=0.0,\\n            kd_alpha=0.0,\\n            grad_clip_norm=0.0,\\n            student_early_stopping_patience=0,\\n            use_ema=False,\\n        ),\\n    )\\n\\n    hash_deeper_fair_ce = replace(\\n        hash_deeper_usermix_ce,\\n        tag=f\\"{base.tag}_hash_deeper_fair_ce\\",\\n        dataset=replace(\\n            hash_deeper_usermix_ce.dataset,\\n            unknown_fraction=1.0,\\n            silence_fraction=0.12,\\n            silence_reference=\\"known\\",\\n        ),\\n    )\\n\\n    hash_deeper_fair_augmented = replace(\\n        hash_deeper_fair_ce,\\n        tag=f\\"{base.tag}_hash_deeper_fair_augmented\\",\\n        dataset=replace(\\n            hash_deeper_fair_ce.dataset,\\n            time_shift_ms=100,\\n            gain_min=0.8,\\n            gain_max=1.2,\\n            noise_stddev=0.004,\\n        ),\\n        feature=replace(\\n            hash_deeper_fair_ce.feature,\\n            specaugment_prob=0.35,\\n            time_mask_max=4,\\n            freq_mask_max=3,\\n        ),\\n        train=replace(\\n            hash_deeper_fair_ce.train,\\n            student_epochs=24,\\n            student_scheduler_name=\\"cosine\\",\\n            label_smoothing=0.05,\\n            grad_clip_norm=1.0,\\n            student_early_stopping_patience=6,\\n            use_ema=True,\\n            ema_decay=0.995,\\n            eval_with_ema=True,\\n        ),\\n    )\\n\\n    hash_deeper_fair_signed = replace(\\n        hash_deeper_fair_augmented,\\n        tag=f\\"{base.tag}_hash_deeper_fair_signed\\",\\n        model=replace(\\n            hash_deeper_fair_augmented.model,\\n            signed_hash=True,\\n        ),\\n    )\\n\\n    hash_deeper_fair_ce_exact_microfrontend = replace(\\n        hash_deeper_fair_ce,\\n        tag=f\\"{base.tag}_hash_deeper_fair_ce_exact_microfrontend\\",\\n        dataset=replace(\\n            hash_deeper_fair_ce.dataset,\\n            num_workers=2,\\n            cache_features=True,\\n        ),\\n        feature=replace(\\n            hash_deeper_fair_ce.feature,\\n            frontend_name=\\"exact_microfrontend\\",\\n            normalize_mode=\\"none\\",\\n            require_exact_microfrontend=True,\\n            specaugment_prob=0.0,\\n            time_mask_max=0,\\n            freq_mask_max=0,\\n        ),\\n    )\\n\\n    hash_deeper_fair_ce_exact_microfrontend_tuned = replace(\\n        hash_deeper_fair_ce_exact_microfrontend,\\n        tag=f\\"{base.tag}_hash_deeper_fair_ce_exact_microfrontend_tuned\\",\\n        dataset=replace(\\n            hash_deeper_fair_ce_exact_microfrontend.dataset,\\n            batch_size=128,\\n            num_workers=2,\\n            cache_features=True,\\n        ),\\n        train=replace(\\n            hash_deeper_fair_ce_exact_microfrontend.train,\\n            student_epochs=28,\\n            student_scheduler_name=\\"cosine\\",\\n            label_smoothing=0.02,\\n            grad_clip_norm=1.0,\\n            student_early_stopping_patience=6,\\n            use_ema=False,\\n            eval_with_ema=False,\\n        ),\\n    )\\n\\n    hash_deeper_fair_specaug_exact_microfrontend = replace(\\n        hash_deeper_fair_ce_exact_microfrontend_tuned,\\n        tag=f\\"{base.tag}_hash_deeper_fair_specaug_exact_microfrontend\\",\\n        feature=replace(\\n            hash_deeper_fair_ce_exact_microfrontend_tuned.feature,\\n            specaugment_prob=0.35,\\n            time_mask_max=6,\\n            freq_mask_max=4,\\n        ),\\n    )\\n\\n    hash_deeper_fair_pointwise_budget_exact_microfrontend = replace(\\n        hash_deeper_fair_specaug_exact_microfrontend,\\n        tag=f\\"{base.tag}_hash_deeper_fair_pointwise_budget_exact_microfrontend\\",\\n        model=replace(\\n            hash_deeper_fair_specaug_exact_microfrontend.model,\\n            stem_codebook_size=192,\\n            depthwise_codebook_size=96,\\n            pointwise_codebook_size=1024,\\n            linear_codebook_size=256,\\n        ),\\n    )\\n\\n    hash_deeper_fair_balanced_budget_exact_microfrontend = replace(\\n        hash_deeper_fair_ce_exact_microfrontend_tuned,\\n        tag=f\\"{base.tag}_hash_deeper_fair_balanced_budget_exact_microfrontend\\",\\n        model=replace(\\n            hash_deeper_fair_ce_exact_microfrontend_tuned.model,\\n            stem_codebook_size=256,\\n            depthwise_codebook_size=192,\\n            pointwise_codebook_size=864,\\n            linear_codebook_size=520,\\n        ),\\n        train=replace(\\n            hash_deeper_fair_ce_exact_microfrontend_tuned.train,\\n            student_epochs=60,\\n        ),\\n    )\\n\\n    hash_deeper_fair_3block_big_pointwise_exact_microfrontend = replace(\\n        hash_deeper_fair_ce_exact_microfrontend_tuned,\\n        tag=f\\"{base.tag}_hash_deeper_fair_3block_big_pointwise_exact_microfrontend\\",\\n        model=replace(\\n            hash_deeper_fair_ce_exact_microfrontend_tuned.model,\\n            num_blocks=3,\\n            stem_codebook_size=500,\\n            depthwise_codebook_size=500,\\n            pointwise_codebook_size=1000,\\n            linear_codebook_size=384,\\n        ),\\n        train=replace(\\n            hash_deeper_fair_ce_exact_microfrontend_tuned.train,\\n            student_epochs=60,\\n        ),\\n    )\\n\\n    hash_deeper_fair_augmented_exact_microfrontend = replace(\\n        hash_deeper_fair_augmented,\\n        tag=f\\"{base.tag}_hash_deeper_fair_augmented_exact_microfrontend\\",\\n        dataset=replace(\\n            hash_deeper_fair_augmented.dataset,\\n            num_workers=2,\\n            cache_features=False,\\n        ),\\n        feature=replace(\\n            hash_deeper_fair_augmented.feature,\\n            frontend_name=\\"exact_microfrontend\\",\\n            normalize_mode=\\"none\\",\\n            require_exact_microfrontend=True,\\n        ),\\n    )\\n\\n    hash_deeper_fair_signed_exact_microfrontend = replace(\\n        hash_deeper_fair_augmented_exact_microfrontend,\\n        tag=f\\"{base.tag}_hash_deeper_fair_signed_exact_microfrontend\\",\\n        model=replace(\\n            hash_deeper_fair_augmented_exact_microfrontend.model,\\n            signed_hash=True,\\n        ),\\n    )\\n\\n    hash_deeper_fair_signed_cached_exact_microfrontend = replace(\\n        hash_deeper_fair_specaug_exact_microfrontend,\\n        tag=f\\"{base.tag}_hash_deeper_fair_signed_cached_exact_microfrontend\\",\\n        model=replace(\\n            hash_deeper_fair_specaug_exact_microfrontend.model,\\n            signed_hash=True,\\n        ),\\n    )\\n\\n    hash_deeper_fair_residual_exact_microfrontend = replace(\\n        hash_deeper_fair_ce_exact_microfrontend_tuned,\\n        tag=f\\"{base.tag}_hash_deeper_fair_residual_exact_microfrontend\\",\\n        model=replace(\\n            hash_deeper_fair_ce_exact_microfrontend_tuned.model,\\n            use_residual=True,\\n        ),\\n        train=replace(\\n            hash_deeper_fair_ce_exact_microfrontend_tuned.train,\\n            student_epochs=60,\\n        ),\\n    )\\n\\n    hash_deeper_fair_residual_specaug_exact_microfrontend = replace(\\n        hash_deeper_fair_specaug_exact_microfrontend,\\n        tag=f\\"{base.tag}_hash_deeper_fair_residual_specaug_exact_microfrontend\\",\\n        model=replace(\\n            hash_deeper_fair_specaug_exact_microfrontend.model,\\n            use_residual=True,\\n        ),\\n        train=replace(\\n            hash_deeper_fair_specaug_exact_microfrontend.train,\\n            student_epochs=60,\\n        ),\\n    )\\n\\n    hash_pointwise_only_fair = replace(\\n        hash_deeper_fair_augmented,\\n        tag=f\\"{base.tag}_hash_pointwise_only_fair\\",\\n        model=replace(\\n            hash_deeper_fair_augmented.model,\\n            hash_only_pointwise=True,\\n            pointwise_codebook_size=384,\\n            linear_codebook_size=384,\\n        ),\\n    )\\n\\n    dense_teacher_fair = replace(\\n        hash_deeper_fair_augmented,\\n        tag=f\\"{base.tag}_dense_teacher_fair\\",\\n        model=replace(\\n            hash_deeper_fair_augmented.model,\\n            teacher_name=\\"\\",\\n            student_name=\\"dense_dscnn_teacher\\",\\n            teacher_channels=96,\\n            teacher_num_blocks=5,\\n            student_dropout=0.10,\\n        ),\\n        train=replace(\\n            hash_deeper_fair_augmented.train,\\n            student_epochs=18,\\n            label_smoothing=0.05,\\n            kd_alpha=0.0,\\n        ),\\n        export=replace(\\n            hash_deeper_fair_augmented.export,\\n            model_stem=\\"dense_hash_teacher\\",\\n        ),\\n    )\\n\\n    hash_deeper_fair_kd = replace(\\n        hash_deeper_fair_augmented,\\n        tag=f\\"{base.tag}_hash_deeper_fair_kd\\",\\n        teacher_reuse_tag=f\\"{base.tag}_dense_teacher_fair\\",\\n        model=replace(\\n            hash_deeper_fair_augmented.model,\\n            teacher_name=\\"dense_dscnn_teacher\\",\\n            student_name=\\"hash_dscnn_deeper\\",\\n            teacher_channels=96,\\n            teacher_num_blocks=5,\\n        ),\\n        train=replace(\\n            hash_deeper_fair_augmented.train,\\n            teacher_epochs=18,\\n            student_pretrain_epochs=6,\\n            student_epochs=14,\\n            student_polish_epochs=4,\\n            teacher_lr=8e-4,\\n            student_lr=9e-4,\\n            polish_lr=2e-4,\\n            teacher_scheduler_name=\\"cosine\\",\\n            student_scheduler_name=\\"cosine\\",\\n            teacher_label_smoothing=0.05,\\n            label_smoothing=0.02,\\n            kd_alpha=0.60,\\n            kd_temperature=4.0,\\n            teacher_early_stopping_patience=5,\\n            student_early_stopping_patience=6,\\n        ),\\n    )\\n\\n    return {\\n        \\"hash_deeper_usermix_ce\\": hash_deeper_usermix_ce,\\n        \\"hash_deeper_fair_ce\\": hash_deeper_fair_ce,\\n        \\"hash_deeper_fair_augmented\\": hash_deeper_fair_augmented,\\n        \\"hash_deeper_fair_signed\\": hash_deeper_fair_signed,\\n        \\"hash_deeper_fair_ce_exact_microfrontend\\": hash_deeper_fair_ce_exact_microfrontend,\\n        \\"hash_deeper_fair_ce_exact_microfrontend_tuned\\": hash_deeper_fair_ce_exact_microfrontend_tuned,\\n        \\"hash_deeper_fair_specaug_exact_microfrontend\\": hash_deeper_fair_specaug_exact_microfrontend,\\n        \\"hash_deeper_fair_pointwise_budget_exact_microfrontend\\": hash_deeper_fair_pointwise_budget_exact_microfrontend,\\n        \\"hash_deeper_fair_balanced_budget_exact_microfrontend\\": hash_deeper_fair_balanced_budget_exact_microfrontend,\\n        \\"hash_deeper_fair_3block_big_pointwise_exact_microfrontend\\": hash_deeper_fair_3block_big_pointwise_exact_microfrontend,\\n        \\"hash_deeper_fair_augmented_exact_microfrontend\\": hash_deeper_fair_augmented_exact_microfrontend,\\n        \\"hash_deeper_fair_signed_exact_microfrontend\\": hash_deeper_fair_signed_exact_microfrontend,\\n        \\"hash_deeper_fair_signed_cached_exact_microfrontend\\": hash_deeper_fair_signed_cached_exact_microfrontend,\\n        \\"hash_deeper_fair_residual_exact_microfrontend\\": hash_deeper_fair_residual_exact_microfrontend,\\n        \\"hash_deeper_fair_residual_specaug_exact_microfrontend\\": hash_deeper_fair_residual_specaug_exact_microfrontend,\\n        \\"hash_pointwise_only_fair\\": hash_pointwise_only_fair,\\n        \\"dense_teacher_fair\\": dense_teacher_fair,\\n        \\"hash_deeper_fair_kd\\": hash_deeper_fair_kd,\\n    }\\n", "code/training/hash_kws_lab/data.py": "from __future__ import annotations\\n\\nimport hashlib\\nimport json\\nimport os\\nimport random\\nfrom collections import Counter\\nfrom dataclasses import asdict\\nfrom functools import lru_cache\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nimport torch\\nimport torch.nn.functional as F\\nfrom torch.utils.data import DataLoader, Dataset, Subset\\nfrom tqdm.auto import tqdm\\n\\nfrom .config import ExperimentConfig\\n\\ntry:\\n    import torchaudio\\n    from torchaudio.datasets import SPEECHCOMMANDS\\nexcept ImportError:  # pragma: no cover\\n    torchaudio = None\\n    SPEECHCOMMANDS = None\\n\\ntry:\\n    import tensorflow as tf\\nexcept ImportError:  # pragma: no cover\\n    tf = None\\n\\n\\ndef ensure_torchaudio_available() -> None:\\n    if torchaudio is None or SPEECHCOMMANDS is None:\\n        raise ImportError(\\"torchaudio is required for hash KWS experiments\\")\\n\\n\\ndef ensure_exact_microfrontend_available() -> None:\\n    if tf is None:\\n        raise ImportError(\\n            \\"tensorflow is required for exact microfrontend hash-KWS experiments\\"\\n        )\\n    if _microfrontend_module() is None:\\n        raise ImportError(\\n            \\"tensorflow microfrontend op is unavailable. \\"\\n            \\"Install a TensorFlow build that provides \\"\\n            \\"tensorflow.lite.experimental.microfrontend.python.ops.audio_microfrontend_op.\\"\\n        )\\n\\n\\n@lru_cache(maxsize=1)\\ndef _microfrontend_module():\\n    if tf is None:\\n        return None\\n    try:\\n        from tensorflow.lite.experimental.microfrontend.python.ops import (  # type: ignore\\n            audio_microfrontend_op,\\n        )\\n    except Exception:\\n        return None\\n    return audio_microfrontend_op\\n\\n\\ndef _waveform_to_int16(waveform: torch.Tensor) -> Any:\\n    if tf is None:\\n        raise ImportError(\\"tensorflow is required for exact microfrontend extraction\\")\\n    flattened = waveform.detach().cpu().to(torch.float32).view(-1).clamp_(-1.0, 1.0)\\n    tensor = tf.convert_to_tensor(flattened.numpy(), dtype=tf.float32)\\n    return tf.cast(tf.round(tensor * 32767.0), tf.int16)\\n\\n\\ndef _fit_exact_feature_frames(features: Any, experiment: ExperimentConfig) -> Any:\\n    if tf is None:\\n        raise ImportError(\\"tensorflow is required for exact microfrontend extraction\\")\\n    config = experiment.feature\\n    features = tf.cast(features, tf.float32)\\n    features = features[: config.frame_count]\\n    pad_frames = tf.maximum(0, config.frame_count - tf.shape(features)[0])\\n    features = tf.pad(features, [[0, pad_frames], [0, 0]])\\n    features.set_shape([config.frame_count, config.n_mels])\\n    return features\\n\\n\\ndef _extract_exact_microfrontend_feature_map(\\n    waveform: torch.Tensor,\\n    experiment: ExperimentConfig,\\n) -> torch.Tensor:\\n    ensure_exact_microfrontend_available()\\n    if tf is None:\\n        raise ImportError(\\"tensorflow is required for exact microfrontend extraction\\")\\n\\n    config = experiment.feature\\n    module = _microfrontend_module()\\n    if module is None:\\n        raise ImportError(\\"Exact microfrontend op is unavailable\\")\\n\\n    audio = _waveform_to_int16(waveform)\\n    common_kwargs = {\\n        \\"sample_rate\\": config.sample_rate,\\n        \\"num_channels\\": config.n_mels,\\n        \\"lower_band_limit\\": config.lower_band_limit,\\n        \\"upper_band_limit\\": config.upper_band_limit,\\n        \\"smoothing_bits\\": config.smoothing_bits,\\n        \\"even_smoothing\\": config.even_smoothing,\\n        \\"odd_smoothing\\": config.odd_smoothing,\\n        \\"min_signal_remaining\\": config.min_signal_remaining,\\n        \\"enable_pcan\\": config.enable_pcan,\\n        \\"pcan_strength\\": config.pcan_strength,\\n        \\"pcan_offset\\": config.pcan_offset,\\n        \\"gain_bits\\": config.gain_bits,\\n        \\"enable_log\\": config.enable_log,\\n        \\"scale_shift\\": config.scale_shift,\\n        \\"left_context\\": 0,\\n        \\"right_context\\": 0,\\n        \\"frame_stride\\": 1,\\n        \\"zero_padding\\": False,\\n    }\\n    signature_variants = (\\n        {\\n            \\"window_size\\": config.window_ms,\\n            \\"window_step\\": config.hop_ms,\\n        },\\n        {\\n            \\"window_size_ms\\": config.window_ms,\\n            \\"window_step_ms\\": config.hop_ms,\\n        },\\n    )\\n\\n    features = None\\n    for variant in signature_variants:\\n        try:\\n            features = module.audio_microfrontend(audio, **common_kwargs, **variant)\\n            break\\n        except TypeError:\\n            continue\\n    if features is None:\\n        raise RuntimeError(\\"Failed to invoke the TensorFlow microfrontend op\\")\\n\\n    divisor = max(float(config.exact_value_divisor), 1e-6)\\n    quantized = tf.round((tf.cast(features, tf.float32) * 256.0) / divisor) - 128.0\\n    quantized = tf.clip_by_value(quantized, -128.0, 127.0)\\n    quantized = _fit_exact_feature_frames(quantized, experiment)\\n    tensor = torch.from_numpy(quantized.numpy()).to(torch.float32)\\n    return tensor.transpose(0, 1).unsqueeze(0)\\n\\n\\ndef _cache_signature(experiment: ExperimentConfig, subset: str, training: bool) -> str:\\n    dataset = experiment.dataset\\n    payload = {\\n        \\"version\\": 2,\\n        \\"subset\\": subset,\\n        \\"training\\": training,\\n        \\"vocabulary_preset\\": experiment.vocabulary_preset,\\n        \\"commands\\": experiment.commands,\\n        \\"all_labels\\": experiment.all_labels,\\n        \\"feature\\": asdict(experiment.feature),\\n        \\"dataset_mix_and_waveform\\": {\\n            \\"seed\\": dataset.seed,\\n            \\"unknown_fraction\\": dataset.unknown_fraction,\\n            \\"silence_fraction\\": dataset.silence_fraction,\\n            \\"silence_reference\\": dataset.silence_reference,\\n            \\"time_shift_ms\\": dataset.time_shift_ms,\\n            \\"gain_min\\": dataset.gain_min,\\n            \\"gain_max\\": dataset.gain_max,\\n            \\"noise_stddev\\": dataset.noise_stddev,\\n            \\"cache_dtype\\": dataset.cache_dtype,\\n        },\\n    }\\n    encoded = json.dumps(payload, sort_keys=True, ensure_ascii=False).encode(\\"utf-8\\")\\n    return hashlib.sha1(encoded).hexdigest()[:16]\\n\\n\\nclass AudioFeatureExtractor(torch.nn.Module):\\n    def __init__(self, experiment: ExperimentConfig) -> None:\\n        super().__init__()\\n        ensure_torchaudio_available()\\n        self.config = experiment.feature\\n        self.experiment = experiment\\n        self.mel = None\\n        if self.config.frontend_name == \\"log_mel\\":\\n            self.mel = torchaudio.transforms.MelSpectrogram(\\n                sample_rate=self.config.sample_rate,\\n                n_fft=self.config.window_samples,\\n                win_length=self.config.window_samples,\\n                hop_length=self.config.hop_samples,\\n                n_mels=self.config.n_mels,\\n                center=self.config.center,\\n                power=2.0,\\n            )\\n        elif self.config.frontend_name == \\"exact_microfrontend\\":\\n            if self.config.require_exact_microfrontend:\\n                ensure_exact_microfrontend_available()\\n        else:\\n            raise ValueError(f\\"Unsupported frontend_name: {self.config.frontend_name}\\")\\n\\n    def _normalize(self, spec: torch.Tensor) -> torch.Tensor:\\n        if self.config.normalize_mode == \\"none\\":\\n            return spec\\n        if self.config.normalize_mode == \\"instance\\":\\n            return (spec - spec.mean()) / (spec.std() + 1e-5)\\n        if self.config.normalize_mode == \\"per_frequency\\":\\n            mean = spec.mean(dim=-1, keepdim=True)\\n            std = spec.std(dim=-1, keepdim=True)\\n            return (spec - mean) / (std + 1e-5)\\n        raise ValueError(f\\"Unsupported normalize_mode: {self.config.normalize_mode}\\")\\n\\n    def forward(self, waveform: torch.Tensor) -> torch.Tensor:\\n        waveform = waveform.to(torch.float32)\\n        if self.config.frontend_name == \\"log_mel\\":\\n            if self.mel is None:\\n                raise RuntimeError(\\"log-mel frontend is not initialized\\")\\n            spec = self.mel(waveform)\\n            spec = torch.log(spec + self.config.log_offset)\\n        elif self.config.frontend_name == \\"exact_microfrontend\\":\\n            spec = _extract_exact_microfrontend_feature_map(waveform, self.experiment)\\n        else:\\n            raise ValueError(f\\"Unsupported frontend_name: {self.config.frontend_name}\\")\\n        return self._normalize(spec)\\n\\n\\ndef _apply_spec_augment(feature_map: torch.Tensor, experiment: ExperimentConfig, rng: random.Random) -> torch.Tensor:\\n    config = experiment.feature\\n    if config.specaugment_prob <= 0.0 or rng.random() > config.specaugment_prob:\\n        return feature_map\\n    augmented = feature_map.clone()\\n    if config.time_mask_max > 0:\\n        width = rng.randint(0, config.time_mask_max)\\n        if width > 0 and augmented.shape[-1] > width:\\n            start = rng.randint(0, augmented.shape[-1] - width)\\n            augmented[:, :, start : start + width] = 0.0\\n    if config.freq_mask_max > 0:\\n        height = rng.randint(0, config.freq_mask_max)\\n        if height > 0 and augmented.shape[-2] > height:\\n            start = rng.randint(0, augmented.shape[-2] - height)\\n            augmented[:, start : start + height, :] = 0.0\\n    return augmented\\n\\n\\nclass SpeechCommandsHashDataset(Dataset[tuple[torch.Tensor, int]]):\\n    def __init__(self, root: Path, experiment: ExperimentConfig, subset: str, training: bool) -> None:\\n        ensure_torchaudio_available()\\n        self.experiment = experiment\\n        self.subset = subset\\n        self.training = training\\n        self.sample_rate = experiment.feature.sample_rate\\n        self.clip_samples = experiment.feature.clip_samples\\n        self.feature_extractor = AudioFeatureExtractor(experiment)\\n        self.base = SPEECHCOMMANDS(root=str(root), download=True, subset=subset)\\n        self.base_path = Path(getattr(self.base, \\"_path\\", root))\\n\\n        walker = list(getattr(self.base, \\"_walker\\", []))\\n        if not walker:\\n            walker = list(range(len(self.base)))\\n\\n        known_indices: list[int] = []\\n        unknown_indices: list[int] = []\\n        discovered_labels: Counter[str] = Counter()\\n        wanted_words = set(experiment.commands)\\n\\n        for index, item in enumerate(walker):\\n            label = self._label_from_item(index, item)\\n            discovered_labels[label] += 1\\n            if label in wanted_words:\\n                known_indices.append(index)\\n            elif label != \\"_background_noise_\\":\\n                unknown_indices.append(index)\\n\\n        if not known_indices:\\n            preview = dict(discovered_labels.most_common(20))\\n            raise ValueError(\\n                f\\"No wanted words found for subset={subset!r}. Discovered labels sample={preview}\\"\\n            )\\n\\n        offset = {\\"training\\": 0, \\"validation\\": 1, \\"testing\\": 2}[subset]\\n        self.rng = random.Random(experiment.dataset.seed + offset)\\n        self.rng.shuffle(unknown_indices)\\n\\n        unknown_target = 0\\n        if experiment.dataset.unknown_fraction > 0 and unknown_indices:\\n            unknown_target = max(1, int(round(len(known_indices) * experiment.dataset.unknown_fraction)))\\n            unknown_target = min(len(unknown_indices), unknown_target)\\n        selected_unknown = unknown_indices[:unknown_target]\\n\\n        silence_reference_count = len(known_indices)\\n        if experiment.dataset.silence_reference == \\"mixed\\":\\n            silence_reference_count = len(known_indices) + len(selected_unknown)\\n        silence_target = 0\\n        if experiment.dataset.silence_fraction > 0:\\n            silence_target = max(1, int(round(silence_reference_count * experiment.dataset.silence_fraction)))\\n\\n        self.entries: list[tuple[str, int | None, int]] = []\\n        for index in known_indices:\\n            label = self._label_from_item(index, walker[index] if isinstance(walker[index], str) else index)\\n            self.entries.append((\\"speech\\", index, experiment.label_to_index[label]))\\n        for index in selected_unknown:\\n            self.entries.append((\\"speech\\", index, experiment.label_to_index[\\"unknown\\"]))\\n        for silence_index in range(silence_target):\\n            self.entries.append((\\"silence\\", silence_index, experiment.label_to_index[\\"silence\\"]))\\n\\n        self.rng.shuffle(self.entries)\\n        self.background_noises = self._load_background_noise()\\n        self.cache_enabled = bool(experiment.dataset.cache_features)\\n        self.cache_dtype = experiment.dataset.cache_dtype\\n        self.cached_features: torch.Tensor | None = None\\n        self.cached_labels: torch.Tensor | None = None\\n        self.cache_status: dict[str, Any] | None = None\\n        signature = _cache_signature(experiment, subset=subset, training=training)\\n        feature_cache_root = os.environ.get(\\"HASH_KWS_FEATURE_CACHE_ROOT\\", \\"\\").strip()\\n        cache_root = Path(feature_cache_root) if feature_cache_root else root / \\"hash_feature_cache\\"\\n        self.cache_path = cache_root / f\\"{signature}.pt\\"\\n\\n    def _label_from_item(self, index: int, item: Any) -> str:\\n        if isinstance(item, str):\\n            path = Path(item)\\n            if path.parent.name:\\n                return path.parent.name\\n        _, _, label, *_ = self.base[index]\\n        return str(label)\\n\\n    def _load_background_noise(self) -> list[torch.Tensor]:\\n        noises: list[torch.Tensor] = []\\n        candidate_dirs = [\\n            self.base_path / \\"_background_noise_\\",\\n            self.base_path.parent / \\"_background_noise_\\",\\n        ]\\n        data_root = os.environ.get(\\"SPEECHCOMMANDS_DATA_ROOT\\", \\"\\").strip()\\n        if data_root:\\n            candidate_dirs.append(Path(data_root) / \\"_background_noise_\\")\\n\\n        noise_dir = None\\n        for candidate in candidate_dirs:\\n            if candidate.exists():\\n                noise_dir = candidate\\n                break\\n        if noise_dir is None:\\n            return noises\\n\\n        for noise_path in sorted(noise_dir.glob(\\"*.wav\\")):\\n            waveform, sample_rate = torchaudio.load(str(noise_path))\\n            waveform = waveform.mean(dim=0, keepdim=True)\\n            if sample_rate != self.sample_rate:\\n                waveform = torchaudio.functional.resample(waveform, sample_rate, self.sample_rate)\\n            noises.append(waveform)\\n        return noises\\n\\n    def _prepare_waveform(self, waveform: torch.Tensor, sample_rate: int) -> torch.Tensor:\\n        waveform = waveform.mean(dim=0, keepdim=True)\\n        if sample_rate != self.sample_rate:\\n            waveform = torchaudio.functional.resample(waveform, sample_rate, self.sample_rate)\\n        if waveform.shape[1] < self.clip_samples:\\n            waveform = F.pad(waveform, (0, self.clip_samples - waveform.shape[1]))\\n        elif waveform.shape[1] > self.clip_samples:\\n            waveform = waveform[:, : self.clip_samples]\\n        return waveform\\n\\n    def _augment_waveform(self, waveform: torch.Tensor, index: int) -> torch.Tensor:\\n        dataset_cfg = self.experiment.dataset\\n        if dataset_cfg.time_shift_ms > 0:\\n            shift_samples = (self.sample_rate * dataset_cfg.time_shift_ms) // 1000\\n            local_rng = random.Random(dataset_cfg.seed * 1_000_003 + index)\\n            shift = local_rng.randint(-shift_samples, shift_samples)\\n            if shift > 0:\\n                waveform = F.pad(waveform, (shift, 0))[:, : self.clip_samples]\\n            elif shift < 0:\\n                waveform = F.pad(waveform, (0, -shift))[:, -shift : -shift + self.clip_samples]\\n        if dataset_cfg.gain_min != 1.0 or dataset_cfg.gain_max != 1.0:\\n            local_rng = random.Random(dataset_cfg.seed * 1_000_033 + index)\\n            gain = local_rng.uniform(dataset_cfg.gain_min, dataset_cfg.gain_max)\\n            waveform = waveform * gain\\n        if dataset_cfg.noise_stddev > 0.0:\\n            waveform = waveform + torch.randn_like(waveform) * dataset_cfg.noise_stddev\\n        return waveform.clamp_(-1.0, 1.0)\\n\\n    def _make_silence(self, index: int) -> torch.Tensor:\\n        if not self.background_noises:\\n            return torch.zeros(1, self.clip_samples)\\n        noise = self.background_noises[index % len(self.background_noises)]\\n        if noise.shape[1] <= self.clip_samples:\\n            waveform = F.pad(noise, (0, max(self.clip_samples - noise.shape[1], 0)))\\n            return waveform[:, : self.clip_samples] * 0.1\\n        start = (index * 9973) % (noise.shape[1] - self.clip_samples + 1)\\n        return noise[:, start : start + self.clip_samples] * 0.1\\n\\n    def _can_store_int8_cache(self) -> bool:\\n        return (\\n            self.experiment.feature.frontend_name == \\"exact_microfrontend\\"\\n            and self.experiment.feature.normalize_mode == \\"none\\"\\n            and self.cache_dtype == \\"int8\\"\\n        )\\n\\n    def _compute_item(self, index: int) -> tuple[torch.Tensor, int]:\\n        kind, base_index, label_id = self.entries[index]\\n        if kind == \\"speech\\":\\n            waveform, sample_rate, _, *_ = self.base[int(base_index)]\\n            waveform = self._prepare_waveform(waveform, int(sample_rate))\\n        else:\\n            waveform = self._make_silence(index)\\n\\n        if self.training:\\n            waveform = self._augment_waveform(waveform, index=index)\\n\\n        features = self.feature_extractor(waveform)\\n        if self.training:\\n            local_rng = random.Random(self.experiment.dataset.seed * 65_537 + index)\\n            features = _apply_spec_augment(features, self.experiment, local_rng)\\n        return features.squeeze(0), label_id\\n\\n    def _cache_tensor(self, features: torch.Tensor) -> torch.Tensor:\\n        if self._can_store_int8_cache():\\n            return features.to(torch.int8)\\n        return features.to(torch.float32)\\n\\n    def _restore_cached_tensor(self, features: torch.Tensor) -> torch.Tensor:\\n        return features.to(torch.float32)\\n\\n    def materialize_feature_cache(self) -> dict[str, Any]:\\n        if not self.cache_enabled:\\n            self.cache_status = {\\n                \\"enabled\\": False,\\n                \\"status\\": \\"disabled\\",\\n                \\"path\\": str(self.cache_path),\\n            }\\n            return self.cache_status\\n\\n        if self.cached_features is not None and self.cached_labels is not None:\\n            self.cache_status = {\\n                \\"enabled\\": True,\\n                \\"status\\": \\"memory\\",\\n                \\"path\\": str(self.cache_path),\\n                \\"items\\": int(self.cached_labels.numel()),\\n            }\\n            return self.cache_status\\n\\n        if self.cache_path.exists():\\n            payload = torch.load(self.cache_path, map_location=\\"cpu\\")\\n            self.cached_features = payload[\\"features\\"].cpu()\\n            self.cached_labels = payload[\\"labels\\"].cpu()\\n            self.cache_status = {\\n                \\"enabled\\": True,\\n                \\"status\\": \\"loaded\\",\\n                \\"path\\": str(self.cache_path),\\n                \\"items\\": int(self.cached_labels.numel()),\\n                \\"dtype\\": str(self.cached_features.dtype),\\n            }\\n            return self.cache_status\\n\\n        self.cache_path.parent.mkdir(parents=True, exist_ok=True)\\n        cached_features: list[torch.Tensor] = []\\n        cached_labels: list[int] = []\\n        progress = tqdm(\\n            range(len(self.entries)),\\n            desc=f\\"cache {self.subset} {self.experiment.feature.frontend_name}\\",\\n            leave=False,\\n        )\\n        for index in progress:\\n            features, label_id = self._compute_item(index)\\n            cached_features.append(self._cache_tensor(features))\\n            cached_labels.append(int(label_id))\\n\\n        self.cached_features = torch.stack(cached_features, dim=0).cpu()\\n        self.cached_labels = torch.tensor(cached_labels, dtype=torch.int64)\\n        torch.save(\\n            {\\n                \\"features\\": self.cached_features,\\n                \\"labels\\": self.cached_labels,\\n                \\"feature_shape\\": list(self.cached_features.shape),\\n                \\"frontend_name\\": self.experiment.feature.frontend_name,\\n                \\"subset\\": self.subset,\\n                \\"training\\": self.training,\\n            },\\n            self.cache_path,\\n        )\\n        self.cache_status = {\\n            \\"enabled\\": True,\\n            \\"status\\": \\"built\\",\\n            \\"path\\": str(self.cache_path),\\n            \\"items\\": int(self.cached_labels.numel()),\\n            \\"dtype\\": str(self.cached_features.dtype),\\n        }\\n        return self.cache_status\\n\\n    def __len__(self) -> int:\\n        return len(self.entries)\\n\\n    def __getitem__(self, index: int) -> tuple[torch.Tensor, int]:\\n        if self.cached_features is not None and self.cached_labels is not None:\\n            features = self._restore_cached_tensor(self.cached_features[index])\\n            label_id = int(self.cached_labels[index].item())\\n            return features, label_id\\n        return self._compute_item(index)\\n\\n\\ndef describe_dataset(dataset: Dataset[tuple[torch.Tensor, int]], labels: list[str]) -> dict[str, int]:\\n    counts = {label: 0 for label in labels}\\n    source = dataset.dataset if isinstance(dataset, Subset) else dataset\\n    indices = dataset.indices if isinstance(dataset, Subset) else range(len(source))\\n    for index in indices:\\n        _, _, label_id = source.entries[index]\\n        counts[labels[int(label_id)]] += 1\\n    return counts\\n\\n\\ndef maybe_limit(dataset: Dataset[Any], limit: int | None) -> Dataset[Any]:\\n    if limit is None or limit >= len(dataset):\\n        return dataset\\n    if limit <= 0:\\n        raise ValueError(f\\"Dataset limit must be positive, got {limit}\\")\\n    return Subset(dataset, list(range(limit)))\\n\\n\\ndef ensure_non_empty(name: str, dataset: Dataset[Any]) -> None:\\n    if len(dataset) == 0:\\n        raise ValueError(f\\"{name} dataset is empty\\")\\n\\n\\ndef _project_root(start: Path) -> Path:\\n    current = start.resolve()\\n    for candidate in [current, *current.parents]:\\n        if (candidate / \\"code\\").exists() and (candidate / \\"notes\\").exists():\\n            return candidate\\n    return start.resolve()\\n\\n\\ndef prepare_dataloaders(\\n    project_root: Path | None,\\n    experiment: ExperimentConfig,\\n    device: torch.device,\\n) -> dict[str, Any]:\\n    ensure_torchaudio_available()\\n    if project_root is None:\\n        project_root = _project_root(Path.cwd())\\n\\n    data_root_override = os.environ.get(\\"SPEECHCOMMANDS_DATA_ROOT\\", \\"\\").strip()\\n    data_root = Path(data_root_override) if data_root_override else project_root / experiment.dataset.data_dir\\n    data_root.mkdir(parents=True, exist_ok=True)\\n\\n    train_dataset = maybe_limit(\\n        SpeechCommandsHashDataset(data_root, experiment=experiment, subset=\\"training\\", training=True),\\n        experiment.dataset.train_limit,\\n    )\\n    val_dataset = maybe_limit(\\n        SpeechCommandsHashDataset(data_root, experiment=experiment, subset=\\"validation\\", training=False),\\n        experiment.dataset.val_limit,\\n    )\\n    test_dataset = maybe_limit(\\n        SpeechCommandsHashDataset(data_root, experiment=experiment, subset=\\"testing\\", training=False),\\n        experiment.dataset.test_limit,\\n    )\\n\\n    ensure_non_empty(\\"train\\", train_dataset)\\n    ensure_non_empty(\\"validation\\", val_dataset)\\n    ensure_non_empty(\\"test\\", test_dataset)\\n\\n    cache_summary: dict[str, Any] = {}\\n    for split_name, dataset in (\\n        (\\"train\\", train_dataset),\\n        (\\"validation\\", val_dataset),\\n        (\\"test\\", test_dataset),\\n    ):\\n        if isinstance(dataset, Subset):\\n            source = dataset.dataset\\n            cache_summary[split_name] = {\\n                \\"enabled\\": False,\\n                \\"status\\": \\"skipped_for_limited_subset\\",\\n                \\"items\\": len(dataset),\\n                \\"source_items\\": len(source),\\n            }\\n            print(\\n                f\\"Skipping full feature-cache materialization for limited {split_name}: \\"\\n                f\\"{len(dataset)} of {len(source)} items.\\",\\n                flush=True,\\n            )\\n            continue\\n        source = dataset.dataset if isinstance(dataset, Subset) else dataset\\n        if hasattr(source, \\"materialize_feature_cache\\"):\\n            print(\\n                f\\"Preparing feature cache for {split_name}: \\"\\n                f\\"{len(source)} source items -> {source.cache_path}\\",\\n                flush=True,\\n            )\\n            cache_summary[split_name] = source.materialize_feature_cache()\\n\\n    loader_kwargs = {\\n        \\"batch_size\\": experiment.dataset.batch_size,\\n        \\"num_workers\\": experiment.dataset.num_workers,\\n        \\"pin_memory\\": device.type == \\"cuda\\",\\n    }\\n    if experiment.dataset.num_workers > 0:\\n        loader_kwargs[\\"persistent_workers\\"] = experiment.dataset.persistent_workers\\n        loader_kwargs[\\"prefetch_factor\\"] = experiment.dataset.prefetch_factor\\n    train_loader = DataLoader(train_dataset, shuffle=True, **loader_kwargs)\\n    val_loader = DataLoader(val_dataset, shuffle=False, **loader_kwargs)\\n    test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)\\n\\n    feature_preview, _ = next(iter(train_loader))\\n    return {\\n        \\"project_root\\": project_root,\\n        \\"data_root\\": data_root,\\n        \\"loaders\\": {\\n            \\"train\\": train_loader,\\n            \\"validation\\": val_loader,\\n            \\"test\\": test_loader,\\n        },\\n        \\"summary\\": {\\n            \\"train\\": describe_dataset(train_dataset, experiment.all_labels),\\n            \\"validation\\": describe_dataset(val_dataset, experiment.all_labels),\\n            \\"test\\": describe_dataset(test_dataset, experiment.all_labels),\\n        },\\n        \\"cache_summary\\": cache_summary,\\n        \\"feature_preview_shape\\": list(feature_preview.shape),\\n    }\\n", "code/training/hash_kws_lab/models.py": "from __future__ import annotations\\n\\nfrom typing import Any\\n\\nimport torch\\nimport torch.nn as nn\\nimport torch.nn.functional as F\\n\\nfrom .config import ExperimentConfig\\n\\n\\nCONV_HASH_OC = 1337\\nCONV_HASH_IC = 7919\\nCONV_HASH_KH = 2971\\nCONV_HASH_KW = 6151\\nCONV_HASH_LAYER = 104729\\n\\nDW_HASH_CH = 1337\\nDW_HASH_KH = 7919\\nDW_HASH_KW = 2971\\nDW_HASH_LAYER = 104729\\n\\nLINEAR_HASH_A = 1337\\nLINEAR_HASH_B = 7919\\nLINEAR_HASH_C = 2971\\n\\nSIGN_HASH_A = 4099\\nSIGN_HASH_B = 6151\\nSIGN_HASH_C = 14887\\n\\n\\ndef _pair(value: int | tuple[int, int]) -> tuple[int, int]:\\n    if isinstance(value, tuple):\\n        return value\\n    return (value, value)\\n\\n\\ndef _layer_codebook_size(\\n    sizes: tuple[int, ...] | list[int],\\n    index: int,\\n    fallback: int,\\n) -> int:\\n    if index < len(sizes):\\n        selected = int(sizes[index])\\n        if selected <= 0:\\n            raise ValueError(f\\"Codebook size must be positive, got {selected}\\")\\n        return selected\\n    return int(fallback)\\n\\n\\nclass AnalyticHashLinear(nn.Module):\\n    def __init__(\\n        self,\\n        in_dim: int,\\n        out_dim: int,\\n        codebook_size: int,\\n        layer_id: int,\\n        hash_a: int,\\n        hash_b: int,\\n        hash_c: int,\\n        signed_hash: bool = False,\\n    ) -> None:\\n        super().__init__()\\n        self.in_dim = in_dim\\n        self.out_dim = out_dim\\n        self.codebook_size = codebook_size\\n        self.layer_id = layer_id\\n        self.hash_a = hash_a\\n        self.hash_b = hash_b\\n        self.hash_c = hash_c\\n        self.signed_hash = signed_hash\\n\\n        self.codebook = nn.Parameter(torch.randn(codebook_size) * 0.01)\\n        self.bias = nn.Parameter(torch.zeros(out_dim))\\n\\n        self.register_buffer(\\"i_idx\\", torch.arange(out_dim, dtype=torch.long).view(out_dim, 1), persistent=False)\\n        self.register_buffer(\\"j_idx\\", torch.arange(in_dim, dtype=torch.long).view(1, in_dim), persistent=False)\\n\\n    def hash_indices(self) -> torch.Tensor:\\n        return (\\n            (self.i_idx * self.hash_a + self.j_idx * self.hash_b + self.layer_id * self.hash_c)\\n            % self.codebook_size\\n        )\\n\\n    def hash_signs(self) -> torch.Tensor:\\n        bits = (self.i_idx * SIGN_HASH_A + self.j_idx * SIGN_HASH_B + self.layer_id * SIGN_HASH_C) % 2\\n        return bits.to(torch.float32).mul_(2.0).sub_(1.0)\\n\\n    def materialize_weight(self) -> torch.Tensor:\\n        weight = self.codebook[self.hash_indices()]\\n        if self.signed_hash:\\n            weight = weight * self.hash_signs().to(weight.device, dtype=weight.dtype)\\n        return weight\\n\\n    def compact_parameter_count(self) -> int:\\n        return self.codebook.numel() + self.bias.numel()\\n\\n    def virtual_parameter_count(self) -> int:\\n        return self.out_dim * self.in_dim\\n\\n    def export_spec(self) -> dict[str, Any]:\\n        return {\\n            \\"type\\": \\"analytic_hash_linear\\",\\n            \\"in_dim\\": self.in_dim,\\n            \\"out_dim\\": self.out_dim,\\n            \\"codebook_size\\": self.codebook_size,\\n            \\"layer_id\\": self.layer_id,\\n            \\"signed_hash\\": self.signed_hash,\\n        }\\n\\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\\n        return F.linear(x, self.materialize_weight(), self.bias)\\n\\n\\nclass AnalyticHashConv2d(nn.Module):\\n    def __init__(\\n        self,\\n        in_channels: int,\\n        out_channels: int,\\n        kernel_size: int | tuple[int, int],\\n        codebook_size: int,\\n        layer_id: int,\\n        stride: int | tuple[int, int] = 1,\\n        padding: int | tuple[int, int] = 0,\\n        signed_hash: bool = False,\\n    ) -> None:\\n        super().__init__()\\n        self.in_channels = in_channels\\n        self.out_channels = out_channels\\n        self.kernel_size = _pair(kernel_size)\\n        self.stride = _pair(stride)\\n        self.padding = _pair(padding)\\n        self.codebook_size = codebook_size\\n        self.layer_id = layer_id\\n        self.signed_hash = signed_hash\\n\\n        self.codebook = nn.Parameter(torch.randn(codebook_size) * 0.01)\\n        self.bias = nn.Parameter(torch.zeros(out_channels))\\n\\n        self.register_buffer(\\"oc_idx\\", torch.arange(out_channels, dtype=torch.long).view(out_channels, 1, 1, 1), persistent=False)\\n        self.register_buffer(\\"ic_idx\\", torch.arange(in_channels, dtype=torch.long).view(1, in_channels, 1, 1), persistent=False)\\n        self.register_buffer(\\"kh_idx\\", torch.arange(self.kernel_size[0], dtype=torch.long).view(1, 1, self.kernel_size[0], 1), persistent=False)\\n        self.register_buffer(\\"kw_idx\\", torch.arange(self.kernel_size[1], dtype=torch.long).view(1, 1, 1, self.kernel_size[1]), persistent=False)\\n\\n    def hash_indices(self) -> torch.Tensor:\\n        return (\\n            (\\n                self.oc_idx * CONV_HASH_OC\\n                + self.ic_idx * CONV_HASH_IC\\n                + self.kh_idx * CONV_HASH_KH\\n                + self.kw_idx * CONV_HASH_KW\\n                + self.layer_id * CONV_HASH_LAYER\\n            )\\n            % self.codebook_size\\n        )\\n\\n    def hash_signs(self) -> torch.Tensor:\\n        bits = (\\n            self.oc_idx * SIGN_HASH_A\\n            + self.ic_idx * SIGN_HASH_B\\n            + self.kh_idx * SIGN_HASH_C\\n            + self.kw_idx * (SIGN_HASH_A + SIGN_HASH_B)\\n            + self.layer_id * (SIGN_HASH_C + 11)\\n        ) % 2\\n        return bits.to(torch.float32).mul_(2.0).sub_(1.0)\\n\\n    def materialize_weight(self) -> torch.Tensor:\\n        weight = self.codebook[self.hash_indices()]\\n        if self.signed_hash:\\n            weight = weight * self.hash_signs().to(weight.device, dtype=weight.dtype)\\n        return weight\\n\\n    def compact_parameter_count(self) -> int:\\n        return self.codebook.numel() + self.bias.numel()\\n\\n    def virtual_parameter_count(self) -> int:\\n        return self.out_channels * self.in_channels * self.kernel_size[0] * self.kernel_size[1]\\n\\n    def export_spec(self) -> dict[str, Any]:\\n        return {\\n            \\"type\\": \\"analytic_hash_conv2d\\",\\n            \\"in_channels\\": self.in_channels,\\n            \\"out_channels\\": self.out_channels,\\n            \\"kernel_size\\": list(self.kernel_size),\\n            \\"stride\\": list(self.stride),\\n            \\"padding\\": list(self.padding),\\n            \\"codebook_size\\": self.codebook_size,\\n            \\"layer_id\\": self.layer_id,\\n            \\"signed_hash\\": self.signed_hash,\\n        }\\n\\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\\n        return F.conv2d(\\n            x,\\n            self.materialize_weight(),\\n            self.bias,\\n            stride=self.stride,\\n            padding=self.padding,\\n        )\\n\\n\\nclass AnalyticHashDepthwiseConv2d(nn.Module):\\n    def __init__(\\n        self,\\n        channels: int,\\n        kernel_size: int | tuple[int, int],\\n        codebook_size: int,\\n        layer_id: int,\\n        stride: int | tuple[int, int] = 1,\\n        padding: int | tuple[int, int] = 0,\\n        signed_hash: bool = False,\\n    ) -> None:\\n        super().__init__()\\n        self.channels = channels\\n        self.kernel_size = _pair(kernel_size)\\n        self.stride = _pair(stride)\\n        self.padding = _pair(padding)\\n        self.codebook_size = codebook_size\\n        self.layer_id = layer_id\\n        self.signed_hash = signed_hash\\n\\n        self.codebook = nn.Parameter(torch.randn(codebook_size) * 0.01)\\n        self.bias = nn.Parameter(torch.zeros(channels))\\n\\n        self.register_buffer(\\"ch_idx\\", torch.arange(channels, dtype=torch.long).view(channels, 1, 1), persistent=False)\\n        self.register_buffer(\\"kh_idx\\", torch.arange(self.kernel_size[0], dtype=torch.long).view(1, self.kernel_size[0], 1), persistent=False)\\n        self.register_buffer(\\"kw_idx\\", torch.arange(self.kernel_size[1], dtype=torch.long).view(1, 1, self.kernel_size[1]), persistent=False)\\n\\n    def hash_indices(self) -> torch.Tensor:\\n        return (\\n            (\\n                self.ch_idx * DW_HASH_CH\\n                + self.kh_idx * DW_HASH_KH\\n                + self.kw_idx * DW_HASH_KW\\n                + self.layer_id * DW_HASH_LAYER\\n            )\\n            % self.codebook_size\\n        )\\n\\n    def hash_signs(self) -> torch.Tensor:\\n        bits = (\\n            self.ch_idx * SIGN_HASH_A\\n            + self.kh_idx * SIGN_HASH_B\\n            + self.kw_idx * SIGN_HASH_C\\n            + self.layer_id * (SIGN_HASH_A + 29)\\n        ) % 2\\n        return bits.to(torch.float32).mul_(2.0).sub_(1.0)\\n\\n    def materialize_weight(self) -> torch.Tensor:\\n        weight = self.codebook[self.hash_indices()]\\n        if self.signed_hash:\\n            weight = weight * self.hash_signs().to(weight.device, dtype=weight.dtype)\\n        return weight.unsqueeze(1)\\n\\n    def compact_parameter_count(self) -> int:\\n        return self.codebook.numel() + self.bias.numel()\\n\\n    def virtual_parameter_count(self) -> int:\\n        return self.channels * self.kernel_size[0] * self.kernel_size[1]\\n\\n    def export_spec(self) -> dict[str, Any]:\\n        return {\\n            \\"type\\": \\"analytic_hash_depthwise_conv2d\\",\\n            \\"channels\\": self.channels,\\n            \\"kernel_size\\": list(self.kernel_size),\\n            \\"stride\\": list(self.stride),\\n            \\"padding\\": list(self.padding),\\n            \\"codebook_size\\": self.codebook_size,\\n            \\"layer_id\\": self.layer_id,\\n            \\"signed_hash\\": self.signed_hash,\\n        }\\n\\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\\n        return F.conv2d(\\n            x,\\n            self.materialize_weight(),\\n            self.bias,\\n            stride=self.stride,\\n            padding=self.padding,\\n            groups=self.channels,\\n        )\\n\\n\\nclass DSCNNBlock(nn.Module):\\n    def __init__(\\n        self,\\n        channels: int,\\n        depthwise: nn.Module,\\n        pointwise: nn.Module,\\n        residual: bool = False,\\n    ) -> None:\\n        super().__init__()\\n        self.channels = channels\\n        self.depthwise = depthwise\\n        self.bn_dw = nn.BatchNorm2d(channels)\\n        self.pointwise = pointwise\\n        self.bn_pw = nn.BatchNorm2d(channels)\\n        self.residual = residual\\n\\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\\n        shortcut = x\\n        x = torch.relu(self.bn_dw(self.depthwise(x)))\\n        x = self.bn_pw(self.pointwise(x))\\n        if self.residual:\\n            x = x + shortcut\\n        x = torch.relu(x)\\n        return x\\n\\n\\nclass HashDSCNN(nn.Module):\\n    def __init__(\\n        self,\\n        channels: int,\\n        num_blocks: int,\\n        num_classes: int,\\n        stem_codebook_size: int,\\n        depthwise_codebook_size: int,\\n        pointwise_codebook_size: int,\\n        linear_codebook_size: int,\\n        depthwise_codebook_sizes: tuple[int, ...] | list[int],\\n        pointwise_codebook_sizes: tuple[int, ...] | list[int],\\n        signed_hash: bool,\\n        hash_only_pointwise: bool,\\n        use_residual: bool,\\n        dropout: float,\\n    ) -> None:\\n        super().__init__()\\n        layer_id = 0\\n        self.use_residual = use_residual\\n\\n        if hash_only_pointwise:\\n            self.conv0 = nn.Conv2d(1, channels, kernel_size=3, stride=2, padding=1, bias=True)\\n        else:\\n            self.conv0 = AnalyticHashConv2d(\\n                1,\\n                channels,\\n                kernel_size=3,\\n                codebook_size=stem_codebook_size,\\n                layer_id=layer_id,\\n                stride=2,\\n                padding=1,\\n                signed_hash=signed_hash,\\n            )\\n            layer_id += 1\\n        self.bn0 = nn.BatchNorm2d(channels)\\n\\n        blocks: list[DSCNNBlock] = []\\n        for block_index in range(num_blocks):\\n            if hash_only_pointwise:\\n                depthwise = nn.Conv2d(\\n                    channels,\\n                    channels,\\n                    kernel_size=3,\\n                    padding=1,\\n                    groups=channels,\\n                    bias=True,\\n                )\\n            else:\\n                depthwise = AnalyticHashDepthwiseConv2d(\\n                    channels,\\n                    kernel_size=3,\\n                    codebook_size=_layer_codebook_size(\\n                        depthwise_codebook_sizes,\\n                        block_index,\\n                        depthwise_codebook_size,\\n                    ),\\n                    layer_id=layer_id,\\n                    padding=1,\\n                    signed_hash=signed_hash,\\n                )\\n                layer_id += 1\\n\\n            pointwise = AnalyticHashConv2d(\\n                channels,\\n                channels,\\n                kernel_size=1,\\n                codebook_size=_layer_codebook_size(\\n                    pointwise_codebook_sizes,\\n                    block_index,\\n                    pointwise_codebook_size,\\n                ),\\n                layer_id=layer_id,\\n                signed_hash=signed_hash,\\n            )\\n            layer_id += 1\\n            blocks.append(\\n                DSCNNBlock(\\n                    channels=channels,\\n                    depthwise=depthwise,\\n                    pointwise=pointwise,\\n                    residual=use_residual,\\n                )\\n            )\\n\\n        self.blocks = nn.ModuleList(blocks)\\n        self.pool = nn.AdaptiveAvgPool2d(1)\\n        self.dropout = nn.Dropout(dropout)\\n        self.fc = AnalyticHashLinear(\\n            channels,\\n            num_classes,\\n            codebook_size=linear_codebook_size,\\n            layer_id=layer_id,\\n            hash_a=LINEAR_HASH_A,\\n            hash_b=LINEAR_HASH_B,\\n            hash_c=LINEAR_HASH_C,\\n            signed_hash=signed_hash,\\n        )\\n\\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\\n        if x.dim() == 3:\\n            x = x.unsqueeze(1)\\n        x = torch.relu(self.bn0(self.conv0(x)))\\n        for block in self.blocks:\\n            x = block(x)\\n        x = self.pool(x).view(x.shape[0], -1)\\n        x = self.dropout(x)\\n        return self.fc(x)\\n\\n\\nclass DenseDSCNN(nn.Module):\\n    def __init__(self, channels: int, num_blocks: int, num_classes: int, dropout: float) -> None:\\n        super().__init__()\\n        self.conv0 = nn.Conv2d(1, channels, kernel_size=3, stride=2, padding=1, bias=False)\\n        self.bn0 = nn.BatchNorm2d(channels)\\n\\n        blocks: list[DSCNNBlock] = []\\n        for _ in range(num_blocks):\\n            depthwise = nn.Conv2d(\\n                channels,\\n                channels,\\n                kernel_size=3,\\n                padding=1,\\n                groups=channels,\\n                bias=False,\\n            )\\n            pointwise = nn.Conv2d(channels, channels, kernel_size=1, bias=False)\\n            blocks.append(DSCNNBlock(channels=channels, depthwise=depthwise, pointwise=pointwise))\\n\\n        self.blocks = nn.ModuleList(blocks)\\n        self.pool = nn.AdaptiveAvgPool2d(1)\\n        self.dropout = nn.Dropout(dropout)\\n        self.fc = nn.Linear(channels, num_classes)\\n\\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\\n        if x.dim() == 3:\\n            x = x.unsqueeze(1)\\n        x = torch.relu(self.bn0(self.conv0(x)))\\n        for block in self.blocks:\\n            x = block(x)\\n        x = self.pool(x).view(x.shape[0], -1)\\n        x = self.dropout(x)\\n        return self.fc(x)\\n\\n\\ndef build_model_by_name(model_name: str, experiment: ExperimentConfig, role: str) -> nn.Module:\\n    if model_name == \\"hash_dscnn_deeper\\":\\n        return HashDSCNN(\\n            channels=experiment.model.channels,\\n            num_blocks=experiment.model.num_blocks,\\n            num_classes=experiment.num_labels,\\n            stem_codebook_size=experiment.model.stem_codebook_size or experiment.model.codebook_size,\\n            depthwise_codebook_size=experiment.model.depthwise_codebook_size or experiment.model.codebook_size,\\n            pointwise_codebook_size=experiment.model.pointwise_codebook_size or experiment.model.codebook_size,\\n            linear_codebook_size=experiment.model.linear_codebook_size or experiment.model.codebook_size,\\n            depthwise_codebook_sizes=experiment.model.depthwise_codebook_sizes,\\n            pointwise_codebook_sizes=experiment.model.pointwise_codebook_sizes,\\n            signed_hash=experiment.model.signed_hash,\\n            hash_only_pointwise=experiment.model.hash_only_pointwise,\\n            use_residual=experiment.model.use_residual,\\n            dropout=experiment.model.student_dropout,\\n        )\\n    if model_name == \\"dense_dscnn_teacher\\":\\n        channels = experiment.model.teacher_channels if role == \\"teacher\\" else experiment.model.channels\\n        num_blocks = experiment.model.teacher_num_blocks if role == \\"teacher\\" else experiment.model.num_blocks\\n        dropout = experiment.model.teacher_dropout if role == \\"teacher\\" else experiment.model.student_dropout\\n        return DenseDSCNN(channels=channels, num_blocks=num_blocks, num_classes=experiment.num_labels, dropout=dropout)\\n    raise KeyError(f\\"Unknown model name: {model_name}\\")\\n\\n\\ndef build_teacher_model(experiment: ExperimentConfig) -> nn.Module | None:\\n    if not experiment.model.teacher_name:\\n        return None\\n    return build_model_by_name(experiment.model.teacher_name, experiment=experiment, role=\\"teacher\\")\\n\\n\\ndef build_student_model(experiment: ExperimentConfig) -> nn.Module:\\n    return build_model_by_name(experiment.model.student_name, experiment=experiment, role=\\"student\\")\\n\\n\\ndef count_parameters(model: nn.Module) -> int:\\n    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)\\n\\n\\ndef count_hash_compact_parameters(model: nn.Module) -> int:\\n    total = 0\\n    for module in model.modules():\\n        if hasattr(module, \\"compact_parameter_count\\"):\\n            total += int(module.compact_parameter_count())\\n    return total\\n\\n\\ndef count_virtual_dense_parameters(model: nn.Module) -> int:\\n    total = 0\\n    for module in model.modules():\\n        if hasattr(module, \\"virtual_parameter_count\\"):\\n            total += int(module.virtual_parameter_count())\\n    return total\\n\\n\\ndef collect_layer_inventory(model: nn.Module) -> list[dict[str, Any]]:\\n    inventory: list[dict[str, Any]] = []\\n    for name, module in model.named_modules():\\n        if name == \\"\\":\\n            continue\\n        if hasattr(module, \\"export_spec\\"):\\n            payload = dict(module.export_spec())\\n            payload[\\"name\\"] = name\\n            payload[\\"compact_parameters\\"] = int(module.compact_parameter_count())\\n            payload[\\"virtual_parameters\\"] = int(module.virtual_parameter_count())\\n            inventory.append(payload)\\n        elif isinstance(module, nn.Conv2d):\\n            inventory.append(\\n                {\\n                    \\"name\\": name,\\n                    \\"type\\": \\"conv2d\\",\\n                    \\"in_channels\\": module.in_channels,\\n                    \\"out_channels\\": module.out_channels,\\n                    \\"kernel_size\\": list(module.kernel_size),\\n                    \\"stride\\": list(module.stride),\\n                    \\"padding\\": list(module.padding),\\n                    \\"groups\\": module.groups,\\n                }\\n            )\\n        elif isinstance(module, nn.Linear):\\n            inventory.append(\\n                {\\n                    \\"name\\": name,\\n                    \\"type\\": \\"linear\\",\\n                    \\"in_features\\": module.in_features,\\n                    \\"out_features\\": module.out_features,\\n                }\\n            )\\n    return inventory\\n\\n\\ndef estimate_maccs(model: nn.Module, input_shape: tuple[int, int, int]) -> int:\\n    total = 0\\n    handles = []\\n\\n    def hook(module: nn.Module, inputs: tuple[torch.Tensor, ...], output: torch.Tensor) -> None:\\n        nonlocal total\\n        if not isinstance(output, torch.Tensor):\\n            return\\n        if isinstance(module, (nn.Conv2d, AnalyticHashConv2d)):\\n            in_tensor = inputs[0]\\n            out_h, out_w = int(output.shape[-2]), int(output.shape[-1])\\n            kh, kw = _pair(module.kernel_size)  # type: ignore[arg-type]\\n            groups = int(getattr(module, \\"groups\\", 1))\\n            out_channels = int(output.shape[1])\\n            in_channels = int(in_tensor.shape[1])\\n            total += out_h * out_w * out_channels * (in_channels // groups) * kh * kw\\n        elif isinstance(module, AnalyticHashDepthwiseConv2d):\\n            out_h, out_w = int(output.shape[-2]), int(output.shape[-1])\\n            kh, kw = module.kernel_size\\n            total += out_h * out_w * module.channels * kh * kw\\n        elif isinstance(module, nn.Linear):\\n            total += int(module.in_features) * int(module.out_features)\\n        elif isinstance(module, AnalyticHashLinear):\\n            total += int(module.in_dim) * int(module.out_dim)\\n\\n    for module in model.modules():\\n        if isinstance(module, (nn.Conv2d, nn.Linear, AnalyticHashConv2d, AnalyticHashDepthwiseConv2d, AnalyticHashLinear)):\\n            handles.append(module.register_forward_hook(hook))\\n\\n    device = next(model.parameters()).device\\n    dummy = torch.zeros((1, *input_shape), dtype=torch.float32, device=device)\\n    was_training = model.training\\n    model.eval()\\n    with torch.no_grad():\\n        model(dummy)\\n    if was_training:\\n        model.train()\\n    for handle in handles:\\n        handle.remove()\\n    return int(total)\\n\\n\\ndef summarize_model(model: nn.Module, experiment: ExperimentConfig) -> dict[str, Any]:\\n    return {\\n        \\"trainable_parameters\\": count_parameters(model),\\n        \\"hash_compact_parameters\\": count_hash_compact_parameters(model),\\n        \\"virtual_dense_parameters\\": count_virtual_dense_parameters(model),\\n        \\"maccs_rough\\": estimate_maccs(model, experiment.model_input_shape),\\n        \\"layer_inventory\\": collect_layer_inventory(model),\\n    }\\n", "code/training/hash_kws_lab/trainer.py": "from __future__ import annotations\\n\\nimport copy\\nimport hashlib\\nimport json\\nimport math\\nimport os\\nimport time\\nfrom contextlib import nullcontext\\nfrom dataclasses import asdict\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nimport torch\\nimport torch.nn as nn\\nimport torch.nn.functional as F\\nfrom torch.utils.data import DataLoader, Dataset\\nfrom tqdm.auto import tqdm\\n\\nfrom .config import ExperimentConfig\\n\\n\\ndef _autocast_context(device: torch.device, enabled: bool):\\n    if not enabled or device.type != \\"cuda\\":\\n        return nullcontext()\\n    return torch.autocast(device_type=\\"cuda\\", dtype=torch.float16)\\n\\n\\ndef _build_grad_scaler(device: torch.device, enabled: bool):\\n    scaler_enabled = bool(enabled and device.type == \\"cuda\\")\\n    if hasattr(torch, \\"amp\\") and hasattr(torch.amp, \\"GradScaler\\"):\\n        try:\\n            return torch.amp.GradScaler(device.type, enabled=scaler_enabled)\\n        except TypeError:\\n            return torch.amp.GradScaler(enabled=scaler_enabled)\\n    return torch.cuda.amp.GradScaler(enabled=scaler_enabled)\\n\\n\\nclass ModelEMA:\\n    def __init__(self, model: nn.Module, decay: float) -> None:\\n        self.decay = decay\\n        self.parameter_keys = {key for key, _ in model.named_parameters()}\\n        self.shadow = {key: value.detach().clone() for key, value in model.state_dict().items()}\\n        self.backup: dict[str, torch.Tensor] | None = None\\n\\n    @torch.no_grad()\\n    def update(self, model: nn.Module) -> None:\\n        for key, value in model.state_dict().items():\\n            if key not in self.parameter_keys:\\n                self.shadow[key] = value.detach().clone()\\n                continue\\n            if not torch.is_floating_point(value):\\n                self.shadow[key] = value.detach().clone()\\n                continue\\n            self.shadow[key].mul_(self.decay).add_(value.detach(), alpha=1.0 - self.decay)\\n\\n    def apply_to(self, model: nn.Module) -> None:\\n        self.backup = {key: value.detach().clone() for key, value in model.state_dict().items()}\\n        model.load_state_dict(self.shadow, strict=True)\\n\\n    def restore(self, model: nn.Module) -> None:\\n        if self.backup is not None:\\n            model.load_state_dict(self.backup, strict=True)\\n            self.backup = None\\n\\n\\ndef build_optimizer(model: nn.Module, lr: float, weight_decay: float, optimizer_name: str) -> torch.optim.Optimizer:\\n    params = [parameter for parameter in model.parameters() if parameter.requires_grad]\\n    if optimizer_name == \\"adam\\":\\n        return torch.optim.Adam(params, lr=lr, weight_decay=weight_decay)\\n    if optimizer_name == \\"adamw\\":\\n        return torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)\\n    raise KeyError(f\\"Unknown optimizer_name: {optimizer_name}\\")\\n\\n\\ndef build_scheduler(\\n    optimizer: torch.optim.Optimizer,\\n    scheduler_name: str,\\n    epochs: int,\\n) -> torch.optim.lr_scheduler._LRScheduler | None:\\n    if scheduler_name == \\"none\\":\\n        return None\\n    if scheduler_name == \\"cosine\\":\\n        return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, epochs))\\n    raise KeyError(f\\"Unknown scheduler_name: {scheduler_name}\\")\\n\\n\\ndef evaluate(\\n    model: nn.Module,\\n    loader: DataLoader[Any],\\n    device: torch.device,\\n    label_smoothing: float = 0.0,\\n    top_k: int = 3,\\n    use_amp: bool = True,\\n    desc: str = \\"eval\\",\\n) -> dict[str, float]:\\n    model.eval()\\n    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)\\n    total_loss = 0.0\\n    total_correct = 0\\n    total_topk = 0\\n    total_items = 0\\n\\n    progress = tqdm(loader, desc=desc, leave=False)\\n    for batch in progress:\\n        features, targets, _ = _unpack_batch(batch)\\n        features = features.to(device, non_blocking=True)\\n        targets = targets.to(device, non_blocking=True)\\n        with _autocast_context(device, enabled=use_amp):\\n            logits = model(features)\\n            loss = criterion(logits, targets)\\n\\n        batch_size = int(features.shape[0])\\n        total_loss += float(loss.item()) * batch_size\\n        total_correct += int((logits.argmax(dim=1) == targets).sum().item())\\n        topk = logits.topk(min(top_k, logits.shape[1]), dim=1).indices\\n        total_topk += int((topk == targets.unsqueeze(1)).any(dim=1).sum().item())\\n        total_items += batch_size\\n        progress.set_postfix(\\n            loss=f\\"{total_loss / max(total_items, 1):.4f}\\",\\n            acc=f\\"{total_correct / max(total_items, 1):.4f}\\",\\n        )\\n\\n    return {\\n        \\"loss\\": total_loss / max(total_items, 1),\\n        \\"accuracy\\": total_correct / max(total_items, 1),\\n        f\\"top{top_k}_accuracy\\": total_topk / max(total_items, 1),\\n    }\\n\\n\\ndef _cross_entropy(logits: torch.Tensor, targets: torch.Tensor, label_smoothing: float) -> torch.Tensor:\\n    return F.cross_entropy(logits, targets, label_smoothing=label_smoothing)\\n\\n\\ndef _kd_loss(student_logits: torch.Tensor, teacher_logits: torch.Tensor, temperature: float) -> torch.Tensor:\\n    student_log_probs = F.log_softmax(student_logits / temperature, dim=1)\\n    teacher_probs = F.softmax(teacher_logits / temperature, dim=1)\\n    return F.kl_div(student_log_probs, teacher_probs, reduction=\\"batchmean\\") * (temperature ** 2)\\n\\n\\ndef _scheduled_value(\\n    start: float,\\n    final: float | None,\\n    schedule_name: str,\\n    epoch: int,\\n    epochs: int,\\n) -> float:\\n    name = schedule_name.strip().lower()\\n    if final is None or name in (\\"\\", \\"none\\", \\"constant\\"):\\n        return float(start)\\n\\n    progress = 0.0 if epochs <= 1 else float(epoch - 1) / float(epochs - 1)\\n    if name in (\\"linear\\", \\"linear_decay\\", \\"linear_ramp\\"):\\n        factor = progress\\n    elif name in (\\"cosine\\", \\"cosine_decay\\"):\\n        factor = 0.5 - (0.5 * math.cos(math.pi * progress))\\n    else:\\n        raise KeyError(f\\"Unknown schedule_name: {schedule_name}\\")\\n    return float(start + ((final - start) * factor))\\n\\n\\ndef _unpack_batch(batch: Any) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor | None]:\\n    if not isinstance(batch, (tuple, list)):\\n        raise TypeError(f\\"Expected dataloader batch tuple/list, got {type(batch)!r}\\")\\n    if len(batch) == 2:\\n        features, targets = batch\\n        return features, targets, None\\n    if len(batch) == 3:\\n        features, targets, teacher_logits = batch\\n        return features, targets, teacher_logits\\n    raise ValueError(f\\"Expected 2 or 3 batch items, got {len(batch)}\\")\\n\\n\\ndef _state_dict_fingerprint(model: nn.Module) -> str:\\n    digest = hashlib.sha1()\\n    for key, value in sorted(model.state_dict().items()):\\n        tensor = value.detach().cpu().contiguous()\\n        digest.update(key.encode(\\"utf-8\\"))\\n        digest.update(str(tuple(tensor.shape)).encode(\\"utf-8\\"))\\n        digest.update(str(tensor.dtype).encode(\\"utf-8\\"))\\n        digest.update(tensor.numpy().tobytes())\\n    return digest.hexdigest()[:16]\\n\\n\\ndef _teacher_logits_cache_path(\\n    teacher: nn.Module,\\n    loader: DataLoader[Any],\\n    experiment: ExperimentConfig,\\n    split_name: str,\\n) -> tuple[Path, dict[str, Any]]:\\n    configured = experiment.train.teacher_logits_cache_dir.strip()\\n    env_root = os.environ.get(\\"HASH_KWS_TEACHER_LOGITS_CACHE_ROOT\\", \\"\\").strip()\\n    root = Path(configured or env_root or (Path(experiment.export.artifacts_dir) / \\"teacher_logits\\"))\\n    metadata = {\\n        \\"version\\": 1,\\n        \\"split_name\\": split_name,\\n        \\"items\\": len(loader.dataset),\\n        \\"labels\\": experiment.all_labels,\\n        \\"feature\\": asdict(experiment.feature),\\n        \\"dataset\\": {\\n            \\"seed\\": experiment.dataset.seed,\\n            \\"unknown_fraction\\": experiment.dataset.unknown_fraction,\\n            \\"silence_fraction\\": experiment.dataset.silence_fraction,\\n            \\"silence_reference\\": experiment.dataset.silence_reference,\\n            \\"time_shift_ms\\": experiment.dataset.time_shift_ms,\\n            \\"gain_min\\": experiment.dataset.gain_min,\\n            \\"gain_max\\": experiment.dataset.gain_max,\\n            \\"noise_stddev\\": experiment.dataset.noise_stddev,\\n            \\"train_limit\\": experiment.dataset.train_limit,\\n            \\"val_limit\\": experiment.dataset.val_limit,\\n            \\"test_limit\\": experiment.dataset.test_limit,\\n        },\\n        \\"teacher\\": {\\n            \\"name\\": experiment.model.teacher_name,\\n            \\"channels\\": experiment.model.teacher_channels,\\n            \\"num_blocks\\": experiment.model.teacher_num_blocks,\\n            \\"dropout\\": experiment.model.teacher_dropout,\\n            \\"state_fingerprint\\": _state_dict_fingerprint(teacher),\\n        },\\n    }\\n    signature = hashlib.sha1(\\n        json.dumps(metadata, sort_keys=True, ensure_ascii=False).encode(\\"utf-8\\")\\n    ).hexdigest()[:16]\\n    return root / f\\"{signature}_{split_name}_teacher_logits.pt\\", metadata\\n\\n\\nclass TeacherLogitsDataset(Dataset[tuple[torch.Tensor, int, torch.Tensor]]):\\n    def __init__(self, source: Dataset[Any], logits: torch.Tensor) -> None:\\n        if len(source) != int(logits.shape[0]):\\n            raise ValueError(f\\"Dataset/logit length mismatch: {len(source)} vs {int(logits.shape[0])}\\")\\n        self.source = source\\n        self.logits = logits.cpu()\\n\\n    def __len__(self) -> int:\\n        return len(self.source)\\n\\n    def __getitem__(self, index: int) -> tuple[torch.Tensor, int, torch.Tensor]:\\n        features, target = self.source[index]\\n        return features, int(target), self.logits[index]\\n\\n\\ndef _loader_like(loader: DataLoader[Any], dataset: Dataset[Any], shuffle: bool) -> DataLoader[Any]:\\n    kwargs: dict[str, Any] = {\\n        \\"batch_size\\": loader.batch_size or 1,\\n        \\"shuffle\\": shuffle,\\n        \\"num_workers\\": loader.num_workers,\\n        \\"pin_memory\\": loader.pin_memory,\\n        \\"drop_last\\": loader.drop_last,\\n        \\"collate_fn\\": loader.collate_fn,\\n    }\\n    if loader.num_workers > 0:\\n        kwargs[\\"persistent_workers\\"] = False\\n        if loader.prefetch_factor is not None:\\n            kwargs[\\"prefetch_factor\\"] = loader.prefetch_factor\\n    return DataLoader(dataset, **kwargs)\\n\\n\\ndef materialize_teacher_logits(\\n    teacher: nn.Module,\\n    loader: DataLoader[Any],\\n    experiment: ExperimentConfig,\\n    device: torch.device,\\n    split_name: str = \\"train\\",\\n) -> dict[str, Any]:\\n    \\"\\"\\"Precompute teacher logits and return a train loader that serves them.\\n\\n    The cache key includes the exact feature/dataset setup and a fingerprint of\\n    the teacher state, so multiple student recipes can reuse logits only when\\n    the supervising model is actually identical.\\n    \\"\\"\\"\\n\\n    cache_path, metadata = _teacher_logits_cache_path(teacher, loader, experiment, split_name)\\n    cache_path.parent.mkdir(parents=True, exist_ok=True)\\n    dtype_name = experiment.train.teacher_logits_cache_dtype.strip().lower()\\n    stored_dtype = torch.float16 if dtype_name == \\"float16\\" else torch.float32\\n\\n    cache_status = \\"built\\"\\n    logits: torch.Tensor | None = None\\n    if cache_path.exists() and not experiment.train.teacher_logits_cache_rebuild:\\n        payload = torch.load(cache_path, map_location=\\"cpu\\")\\n        cached_metadata = payload.get(\\"metadata\\", {})\\n        cached_logits = payload.get(\\"logits\\")\\n        if (\\n            isinstance(cached_logits, torch.Tensor)\\n            and int(cached_logits.shape[0]) == len(loader.dataset)\\n            and cached_metadata == metadata\\n        ):\\n            logits = cached_logits.cpu()\\n            cache_status = \\"loaded\\"\\n\\n    if logits is None:\\n        was_training = teacher.training\\n        teacher.eval()\\n        teacher.to(device)\\n        sequential_loader = _loader_like(loader, loader.dataset, shuffle=False)\\n        chunks: list[torch.Tensor] = []\\n        progress = tqdm(sequential_loader, desc=f\\"cache {split_name} teacher logits\\", leave=False)\\n        with torch.no_grad():\\n            for batch in progress:\\n                features, _, _ = _unpack_batch(batch)\\n                features = features.to(device, non_blocking=True)\\n                with _autocast_context(device, enabled=experiment.train.use_amp):\\n                    batch_logits = teacher(features)\\n                chunks.append(batch_logits.detach().to(\\"cpu\\", dtype=stored_dtype))\\n        if was_training:\\n            teacher.train()\\n        logits = torch.cat(chunks, dim=0)\\n        torch.save(\\n            {\\n                \\"logits\\": logits,\\n                \\"metadata\\": metadata,\\n                \\"dtype\\": str(logits.dtype),\\n            },\\n            cache_path,\\n        )\\n\\n    cached_dataset = TeacherLogitsDataset(loader.dataset, logits)\\n    cached_loader = _loader_like(loader, cached_dataset, shuffle=True)\\n    return {\\n        \\"loader\\": cached_loader,\\n        \\"path\\": str(cache_path),\\n        \\"status\\": cache_status,\\n        \\"items\\": int(logits.shape[0]),\\n        \\"dtype\\": str(logits.dtype),\\n        \\"metadata\\": metadata,\\n    }\\n\\n\\ndef load_model_checkpoint(\\n    model: nn.Module,\\n    checkpoint_path: str | Path,\\n    device: torch.device,\\n) -> dict[str, Any]:\\n    path = Path(checkpoint_path)\\n    payload = torch.load(path, map_location=device)\\n    state_dict: dict[str, torch.Tensor]\\n    if isinstance(payload, dict) and isinstance(payload.get(\\"state_dict\\"), dict):\\n        state_dict = payload[\\"state_dict\\"]\\n    elif isinstance(payload, dict) and isinstance(payload.get(\\"best_state\\"), dict):\\n        state_dict = payload[\\"best_state\\"]\\n    elif isinstance(payload, dict):\\n        state_dict = payload\\n    else:\\n        raise TypeError(f\\"Unsupported checkpoint payload type: {type(payload)!r}\\")\\n    model.load_state_dict(state_dict, strict=True)\\n    return {\\n        \\"path\\": str(path),\\n        \\"state_keys\\": len(state_dict),\\n        \\"payload_keys\\": sorted(payload.keys()) if isinstance(payload, dict) else [],\\n    }\\n\\n\\ndef _train_stage(\\n    model: nn.Module,\\n    train_loader: DataLoader[Any],\\n    val_loader: DataLoader[Any],\\n    device: torch.device,\\n    epochs: int,\\n    stage_name: str,\\n    lr: float,\\n    optimizer_name: str,\\n    scheduler_name: str,\\n    weight_decay: float,\\n    label_smoothing: float,\\n    grad_clip_norm: float,\\n    use_amp: bool,\\n    use_ema: bool,\\n    ema_decay: float,\\n    eval_with_ema: bool,\\n    top_k: int,\\n    early_stopping_patience: int,\\n    teacher: nn.Module | None = None,\\n    kd_alpha: float = 0.0,\\n    kd_alpha_schedule: str = \\"constant\\",\\n    kd_alpha_final: float | None = None,\\n    kd_temperature: float = 4.0,\\n    kd_temperature_schedule: str = \\"constant\\",\\n    kd_temperature_final: float | None = None,\\n) -> dict[str, Any]:\\n    optimizer = build_optimizer(model, lr=lr, weight_decay=weight_decay, optimizer_name=optimizer_name)\\n    scheduler = build_scheduler(optimizer, scheduler_name=scheduler_name, epochs=epochs)\\n    scaler = _build_grad_scaler(device, enabled=use_amp)\\n    ema = ModelEMA(model, decay=ema_decay) if use_ema else None\\n    history: list[dict[str, float]] = []\\n    best_state = copy.deepcopy(model.state_dict())\\n    best_val_accuracy = -1.0\\n    best_epoch = 0\\n    stale_epochs = 0\\n    started = time.perf_counter()\\n\\n    if teacher is not None:\\n        teacher.eval()\\n        teacher.to(device)\\n\\n    for epoch in range(1, epochs + 1):\\n        model.train()\\n        total_loss = 0.0\\n        total_correct = 0\\n        total_items = 0\\n        epoch_kd_alpha = _scheduled_value(\\n            kd_alpha,\\n            kd_alpha_final,\\n            kd_alpha_schedule,\\n            epoch,\\n            epochs,\\n        )\\n        epoch_kd_temperature = _scheduled_value(\\n            kd_temperature,\\n            kd_temperature_final,\\n            kd_temperature_schedule,\\n            epoch,\\n            epochs,\\n        )\\n\\n        progress = tqdm(train_loader, desc=f\\"{stage_name} | epoch {epoch}/{epochs}\\", leave=True)\\n        for batch in progress:\\n            features, targets, cached_teacher_logits = _unpack_batch(batch)\\n            features = features.to(device, non_blocking=True)\\n            targets = targets.to(device, non_blocking=True)\\n\\n            optimizer.zero_grad(set_to_none=True)\\n            with _autocast_context(device, enabled=use_amp):\\n                logits = model(features)\\n                loss = _cross_entropy(logits, targets, label_smoothing=label_smoothing)\\n                if teacher is not None and epoch_kd_alpha > 0.0:\\n                    if cached_teacher_logits is None:\\n                        with torch.no_grad():\\n                            teacher_logits = teacher(features)\\n                    else:\\n                        teacher_logits = cached_teacher_logits.to(\\n                            device,\\n                            non_blocking=True,\\n                            dtype=logits.dtype,\\n                        )\\n                    loss = (1.0 - epoch_kd_alpha) * loss + epoch_kd_alpha * _kd_loss(\\n                        logits,\\n                        teacher_logits,\\n                        temperature=epoch_kd_temperature,\\n                    )\\n\\n            scaler.scale(loss).backward()\\n            if grad_clip_norm > 0.0:\\n                scaler.unscale_(optimizer)\\n                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)\\n            scaler.step(optimizer)\\n            scaler.update()\\n            if ema is not None:\\n                ema.update(model)\\n\\n            batch_size = int(features.shape[0])\\n            total_loss += float(loss.item()) * batch_size\\n            total_correct += int((logits.argmax(dim=1) == targets).sum().item())\\n            total_items += batch_size\\n            progress.set_postfix(\\n                loss=f\\"{total_loss / max(total_items, 1):.4f}\\",\\n                acc=f\\"{total_correct / max(total_items, 1):.4f}\\",\\n            )\\n\\n        if scheduler is not None:\\n            scheduler.step()\\n\\n        train_metrics = {\\n            \\"loss\\": total_loss / max(total_items, 1),\\n            \\"accuracy\\": total_correct / max(total_items, 1),\\n        }\\n        if ema is not None and eval_with_ema:\\n            ema.apply_to(model)\\n        val_metrics = evaluate(\\n            model,\\n            val_loader,\\n            device=device,\\n            label_smoothing=0.0,\\n            top_k=top_k,\\n            use_amp=use_amp,\\n            desc=f\\"{stage_name} | val {epoch}/{epochs}\\",\\n        )\\n        if ema is not None and eval_with_ema:\\n            ema.restore(model)\\n\\n        history.append(\\n            {\\n                \\"epoch\\": float(epoch),\\n                \\"train_loss\\": train_metrics[\\"loss\\"],\\n                \\"train_accuracy\\": train_metrics[\\"accuracy\\"],\\n                \\"val_loss\\": val_metrics[\\"loss\\"],\\n                \\"val_accuracy\\": val_metrics[\\"accuracy\\"],\\n                \\"lr\\": float(optimizer.param_groups[0][\\"lr\\"]),\\n                \\"kd_alpha\\": float(epoch_kd_alpha if teacher is not None else 0.0),\\n                \\"kd_temperature\\": float(epoch_kd_temperature if teacher is not None else 0.0),\\n            }\\n        )\\n\\n        if val_metrics[\\"accuracy\\"] > best_val_accuracy:\\n            best_val_accuracy = val_metrics[\\"accuracy\\"]\\n            best_epoch = epoch\\n            stale_epochs = 0\\n            best_state = copy.deepcopy(ema.shadow if ema is not None and eval_with_ema else model.state_dict())\\n        else:\\n            stale_epochs += 1\\n\\n        if early_stopping_patience > 0 and stale_epochs >= early_stopping_patience:\\n            break\\n\\n    model.load_state_dict(best_state, strict=True)\\n    return {\\n        \\"history\\": history,\\n        \\"best_state\\": best_state,\\n        \\"best_val_accuracy\\": best_val_accuracy,\\n        \\"best_epoch\\": best_epoch,\\n        \\"elapsed_sec\\": time.perf_counter() - started,\\n    }\\n\\n\\ndef train_teacher(\\n    teacher: nn.Module,\\n    loaders: dict[str, DataLoader[Any]],\\n    experiment: ExperimentConfig,\\n    device: torch.device,\\n) -> dict[str, Any]:\\n    return _train_stage(\\n        model=teacher.to(device),\\n        train_loader=loaders[\\"train\\"],\\n        val_loader=loaders[\\"validation\\"],\\n        device=device,\\n        epochs=experiment.train.teacher_epochs,\\n        stage_name=\\"hash_teacher\\",\\n        lr=experiment.train.teacher_lr,\\n        optimizer_name=experiment.train.optimizer_name,\\n        scheduler_name=experiment.train.teacher_scheduler_name,\\n        weight_decay=experiment.train.weight_decay,\\n        label_smoothing=experiment.train.teacher_label_smoothing,\\n        grad_clip_norm=experiment.train.grad_clip_norm,\\n        use_amp=experiment.train.use_amp,\\n        use_ema=experiment.train.use_ema,\\n        ema_decay=experiment.train.ema_decay,\\n        eval_with_ema=experiment.train.eval_with_ema,\\n        top_k=experiment.train.top_k,\\n        early_stopping_patience=experiment.train.teacher_early_stopping_patience,\\n    )\\n\\n\\ndef train_student(\\n    student: nn.Module,\\n    loaders: dict[str, DataLoader[Any]],\\n    experiment: ExperimentConfig,\\n    device: torch.device,\\n    teacher: nn.Module | None = None,\\n) -> dict[str, Any]:\\n    student = student.to(device)\\n    full_history: list[dict[str, float]] = []\\n    stage_summaries: list[dict[str, Any]] = []\\n    best_state = copy.deepcopy(student.state_dict())\\n    teacher_logits_cache: dict[str, Any] | None = None\\n\\n    if experiment.train.student_pretrain_epochs > 0:\\n        pretrain = _train_stage(\\n            model=student,\\n            train_loader=loaders[\\"train\\"],\\n            val_loader=loaders[\\"validation\\"],\\n            device=device,\\n            epochs=experiment.train.student_pretrain_epochs,\\n            stage_name=\\"hash_student_pretrain\\",\\n            lr=experiment.train.student_lr,\\n            optimizer_name=experiment.train.optimizer_name,\\n            scheduler_name=experiment.train.student_scheduler_name,\\n            weight_decay=experiment.train.weight_decay,\\n            label_smoothing=experiment.train.label_smoothing,\\n            grad_clip_norm=experiment.train.grad_clip_norm,\\n            use_amp=experiment.train.use_amp,\\n            use_ema=experiment.train.use_ema,\\n            ema_decay=experiment.train.ema_decay,\\n            eval_with_ema=experiment.train.eval_with_ema,\\n            top_k=experiment.train.top_k,\\n            early_stopping_patience=experiment.train.student_early_stopping_patience,\\n        )\\n        best_state = copy.deepcopy(student.state_dict())\\n        full_history.extend(pretrain[\\"history\\"])\\n        stage_summaries.append(\\n            {\\n                \\"stage\\": \\"pretrain\\",\\n                \\"best_val_accuracy\\": pretrain[\\"best_val_accuracy\\"],\\n                \\"best_epoch\\": pretrain[\\"best_epoch\\"],\\n                \\"elapsed_sec\\": pretrain[\\"elapsed_sec\\"],\\n            }\\n        )\\n\\n    main_train_loader = loaders[\\"train\\"]\\n    if (\\n        teacher is not None\\n        and experiment.train.uses_distillation\\n        and experiment.train.cache_teacher_logits\\n    ):\\n        teacher_logits_cache = materialize_teacher_logits(\\n            teacher=teacher,\\n            loader=loaders[\\"train\\"],\\n            experiment=experiment,\\n            device=device,\\n            split_name=\\"train\\",\\n        )\\n        main_train_loader = teacher_logits_cache[\\"loader\\"]\\n\\n    main_train = _train_stage(\\n        model=student,\\n        train_loader=main_train_loader,\\n        val_loader=loaders[\\"validation\\"],\\n        device=device,\\n        epochs=experiment.train.student_epochs,\\n        stage_name=\\"hash_student\\",\\n        lr=experiment.train.student_lr,\\n        optimizer_name=experiment.train.optimizer_name,\\n        scheduler_name=experiment.train.student_scheduler_name,\\n        weight_decay=experiment.train.weight_decay,\\n        label_smoothing=experiment.train.label_smoothing,\\n        grad_clip_norm=experiment.train.grad_clip_norm,\\n        use_amp=experiment.train.use_amp,\\n        use_ema=experiment.train.use_ema,\\n        ema_decay=experiment.train.ema_decay,\\n        eval_with_ema=experiment.train.eval_with_ema,\\n        top_k=experiment.train.top_k,\\n        early_stopping_patience=experiment.train.student_early_stopping_patience,\\n        teacher=teacher,\\n        kd_alpha=experiment.train.kd_alpha,\\n        kd_alpha_schedule=experiment.train.kd_alpha_schedule,\\n        kd_alpha_final=experiment.train.kd_alpha_final,\\n        kd_temperature=experiment.train.kd_temperature,\\n        kd_temperature_schedule=experiment.train.kd_temperature_schedule,\\n        kd_temperature_final=experiment.train.kd_temperature_final,\\n    )\\n    best_state = copy.deepcopy(student.state_dict())\\n    full_history.extend(main_train[\\"history\\"])\\n    stage_summaries.append(\\n        {\\n            \\"stage\\": \\"student\\",\\n            \\"best_val_accuracy\\": main_train[\\"best_val_accuracy\\"],\\n            \\"best_epoch\\": main_train[\\"best_epoch\\"],\\n            \\"elapsed_sec\\": main_train[\\"elapsed_sec\\"],\\n            \\"distillation_enabled\\": bool(teacher is not None and experiment.train.uses_distillation),\\n            \\"teacher_logits_cache\\": {\\n                key: value\\n                for key, value in (teacher_logits_cache or {}).items()\\n                if key not in (\\"loader\\", \\"metadata\\")\\n            },\\n            \\"kd_alpha_schedule\\": experiment.train.kd_alpha_schedule,\\n            \\"kd_alpha_final\\": experiment.train.kd_alpha_final,\\n            \\"kd_temperature_schedule\\": experiment.train.kd_temperature_schedule,\\n            \\"kd_temperature_final\\": experiment.train.kd_temperature_final,\\n        }\\n    )\\n\\n    if experiment.train.student_polish_epochs > 0:\\n        polish_label_smoothing = (\\n            experiment.train.polish_label_smoothing\\n            if experiment.train.polish_label_smoothing is not None\\n            else max(0.0, experiment.train.label_smoothing * 0.5)\\n        )\\n        polish = _train_stage(\\n            model=student,\\n            train_loader=loaders[\\"train\\"],\\n            val_loader=loaders[\\"validation\\"],\\n            device=device,\\n            epochs=experiment.train.student_polish_epochs,\\n            stage_name=\\"hash_student_polish\\",\\n            lr=experiment.train.polish_lr,\\n            optimizer_name=experiment.train.optimizer_name,\\n            scheduler_name=\\"none\\",\\n            weight_decay=experiment.train.weight_decay,\\n            label_smoothing=polish_label_smoothing,\\n            grad_clip_norm=experiment.train.grad_clip_norm,\\n            use_amp=experiment.train.use_amp,\\n            use_ema=experiment.train.use_ema,\\n            ema_decay=experiment.train.ema_decay,\\n            eval_with_ema=experiment.train.eval_with_ema,\\n            top_k=experiment.train.top_k,\\n            early_stopping_patience=max(0, experiment.train.student_early_stopping_patience // 2),\\n        )\\n        best_state = copy.deepcopy(student.state_dict())\\n        full_history.extend(polish[\\"history\\"])\\n        stage_summaries.append(\\n            {\\n                \\"stage\\": \\"polish\\",\\n                \\"best_val_accuracy\\": polish[\\"best_val_accuracy\\"],\\n                \\"best_epoch\\": polish[\\"best_epoch\\"],\\n                \\"elapsed_sec\\": polish[\\"elapsed_sec\\"],\\n                \\"label_smoothing\\": polish_label_smoothing,\\n            }\\n        )\\n\\n    student.load_state_dict(best_state, strict=True)\\n    test_metrics = evaluate(\\n        student,\\n        loaders[\\"test\\"],\\n        device=device,\\n        label_smoothing=0.0,\\n        top_k=experiment.train.top_k,\\n        use_amp=experiment.train.use_amp,\\n        desc=\\"hash_student | test\\",\\n    )\\n    return {\\n        \\"history\\": full_history,\\n        \\"test_metrics\\": test_metrics,\\n        \\"best_state\\": best_state,\\n        \\"stage_summaries\\": stage_summaries,\\n    }\\n", "code/training/hash_kws_lab/reporting.py": "from __future__ import annotations\\n\\nimport json\\nfrom copy import deepcopy\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nimport matplotlib.pyplot as plt\\n\\nfrom .config import ExperimentConfig\\n\\n\\ndef get_run_dir(project_root: Path, experiment: ExperimentConfig) -> Path:\\n    run_dir = project_root / \\"code\\" / \\"training\\" / \\"hash_runs\\" / experiment.tag\\n    run_dir.mkdir(parents=True, exist_ok=True)\\n    return run_dir\\n\\n\\ndef _state_path(run_dir: Path) -> Path:\\n    return run_dir / \\"run_state.json\\"\\n\\n\\ndef _summary_path(run_dir: Path) -> Path:\\n    return run_dir / \\"run_summary.md\\"\\n\\n\\ndef _read_state(run_dir: Path) -> dict[str, Any]:\\n    path = _state_path(run_dir)\\n    if not path.exists():\\n        return {}\\n    return json.loads(path.read_text(encoding=\\"utf-8\\"))\\n\\n\\ndef _write_state(run_dir: Path, state: dict[str, Any]) -> None:\\n    _state_path(run_dir).write_text(\\n        json.dumps(state, ensure_ascii=False, indent=2),\\n        encoding=\\"utf-8\\",\\n    )\\n\\n\\ndef _merge_dict(base: dict[str, Any], update: dict[str, Any]) -> dict[str, Any]:\\n    merged = deepcopy(base)\\n    for key, value in update.items():\\n        if isinstance(value, dict) and isinstance(merged.get(key), dict):\\n            merged[key] = _merge_dict(merged[key], value)\\n        else:\\n            merged[key] = value\\n    return merged\\n\\n\\ndef initialize_run_state(\\n    project_root: Path,\\n    experiment: ExperimentConfig,\\n    recipe_name: str,\\n    dataset_summary: dict[str, Any] | None = None,\\n) -> Path:\\n    run_dir = get_run_dir(project_root, experiment)\\n    state = _merge_dict(\\n        _read_state(run_dir),\\n        {\\n            \\"experiment\\": experiment.to_dict(),\\n            \\"recipe_name\\": recipe_name,\\n            \\"dataset_summary\\": dataset_summary or {},\\n            \\"stages\\": {},\\n            \\"artifacts\\": {},\\n            \\"notes\\": [],\\n        },\\n    )\\n    _write_state(run_dir, state)\\n    write_run_summary(run_dir)\\n    return run_dir\\n\\n\\ndef save_json_artifact(run_dir: Path, name: str, payload: Any) -> Path:\\n    path = run_dir / name\\n    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding=\\"utf-8\\")\\n    return path\\n\\n\\ndef save_text_artifact(run_dir: Path, name: str, text: str) -> Path:\\n    path = run_dir / name\\n    path.write_text(text, encoding=\\"utf-8\\")\\n    return path\\n\\n\\ndef save_history(run_dir: Path, stage_name: str, history: list[dict[str, float]]) -> Path:\\n    return save_json_artifact(run_dir, f\\"{stage_name}_history.json\\", history)\\n\\n\\ndef save_metrics(run_dir: Path, stage_name: str, metrics: dict[str, Any]) -> Path:\\n    return save_json_artifact(run_dir, f\\"{stage_name}_metrics.json\\", metrics)\\n\\n\\ndef save_model_summary(run_dir: Path, stage_name: str, summary_text: str) -> Path:\\n    return save_text_artifact(run_dir, f\\"{stage_name}_model_summary.txt\\", summary_text)\\n\\n\\ndef save_history_plots(run_dir: Path, stage_name: str, history: list[dict[str, float]]) -> list[str]:\\n    if not history:\\n        return []\\n    keys = history[0].keys()\\n    plot_paths: list[str] = []\\n    groups = {\\n        \\"loss\\": [key for key in keys if \\"loss\\" in key],\\n        \\"metrics\\": [key for key in keys if \\"accuracy\\" in key and key != \\"epoch\\"],\\n        \\"lr\\": [key for key in keys if key == \\"lr\\"],\\n    }\\n    for suffix, selected_keys in groups.items():\\n        if not selected_keys:\\n            continue\\n        epochs = [entry[\\"epoch\\"] for entry in history]\\n        plt.figure(figsize=(10, 4))\\n        for key in selected_keys:\\n            plt.plot(epochs, [entry[key] for entry in history], label=key)\\n        plt.title(f\\"{stage_name} {suffix}\\")\\n        plt.xlabel(\\"Epoch\\")\\n        plt.grid(True, alpha=0.3)\\n        plt.legend()\\n        path = run_dir / f\\"{stage_name}_{suffix}.png\\"\\n        plt.savefig(path, bbox_inches=\\"tight\\")\\n        plt.close()\\n        plot_paths.append(str(path))\\n    return plot_paths\\n\\n\\ndef update_stage_state(\\n    run_dir: Path,\\n    stage_name: str,\\n    metrics: dict[str, Any] | None = None,\\n    history_path: str | None = None,\\n    plot_paths: list[str] | None = None,\\n    summary_path: str | None = None,\\n    extra: dict[str, Any] | None = None,\\n) -> None:\\n    state = _read_state(run_dir)\\n    stage_block = state.setdefault(\\"stages\\", {}).get(stage_name, {})\\n    update = {\\n        \\"metrics\\": metrics or stage_block.get(\\"metrics\\", {}),\\n        \\"history_path\\": history_path or stage_block.get(\\"history_path\\", \\"\\"),\\n        \\"plot_paths\\": plot_paths or stage_block.get(\\"plot_paths\\", []),\\n        \\"summary_path\\": summary_path or stage_block.get(\\"summary_path\\", \\"\\"),\\n    }\\n    if extra:\\n        update[\\"extra\\"] = _merge_dict(stage_block.get(\\"extra\\", {}), extra)\\n    state.setdefault(\\"stages\\", {})[stage_name] = _merge_dict(stage_block, update)\\n    _write_state(run_dir, state)\\n    write_run_summary(run_dir)\\n\\n\\ndef add_note(run_dir: Path, title: str, body: str) -> None:\\n    state = _read_state(run_dir)\\n    state.setdefault(\\"notes\\", []).append({\\"title\\": title, \\"body\\": body})\\n    _write_state(run_dir, state)\\n    write_run_summary(run_dir)\\n\\n\\ndef record_export_artifacts(run_dir: Path, metadata: dict[str, Any]) -> None:\\n    state = _read_state(run_dir)\\n    state[\\"artifacts\\"] = _merge_dict(state.get(\\"artifacts\\", {}), metadata)\\n    _write_state(run_dir, state)\\n    write_run_summary(run_dir)\\n\\n\\ndef write_run_summary(run_dir: Path) -> Path:\\n    state = _read_state(run_dir)\\n    lines = [\\n        f\\"# Run Summary: {state.get(\'experiment\', {}).get(\'tag\', run_dir.name)}\\",\\n        \\"\\",\\n        f\\"Recipe: `{state.get(\'recipe_name\', \'\')}`\\",\\n        \\"\\",\\n        \\"## Experiment\\",\\n        \\"\\",\\n        \\"```json\\",\\n        json.dumps(state.get(\\"experiment\\", {}), ensure_ascii=False, indent=2),\\n        \\"```\\",\\n        \\"\\",\\n        \\"## Dataset\\",\\n        \\"\\",\\n        \\"```json\\",\\n        json.dumps(state.get(\\"dataset_summary\\", {}), ensure_ascii=False, indent=2),\\n        \\"```\\",\\n        \\"\\",\\n        \\"## Stages\\",\\n        \\"\\",\\n    ]\\n\\n    for stage_name, payload in state.get(\\"stages\\", {}).items():\\n        lines.extend(\\n            [\\n                f\\"### {stage_name}\\",\\n                \\"\\",\\n                \\"```json\\",\\n                json.dumps(payload.get(\\"metrics\\", {}), ensure_ascii=False, indent=2),\\n                \\"```\\",\\n                \\"\\",\\n            ]\\n        )\\n        if payload.get(\\"history_path\\"):\\n            lines.append(f\\"History: `{payload[\'history_path\']}`\\")\\n        if payload.get(\\"summary_path\\"):\\n            lines.append(f\\"Model summary: `{payload[\'summary_path\']}`\\")\\n        for plot_path in payload.get(\\"plot_paths\\", []):\\n            lines.append(f\\"Plot: `{plot_path}`\\")\\n        if payload.get(\\"extra\\"):\\n            lines.extend(\\n                [\\n                    \\"Extra:\\",\\n                    \\"```json\\",\\n                    json.dumps(payload[\\"extra\\"], ensure_ascii=False, indent=2),\\n                    \\"```\\",\\n                ]\\n            )\\n        lines.append(\\"\\")\\n\\n    if state.get(\\"artifacts\\"):\\n        lines.extend(\\n            [\\n                \\"## Artifacts\\",\\n                \\"\\",\\n                \\"```json\\",\\n                json.dumps(state[\\"artifacts\\"], ensure_ascii=False, indent=2),\\n                \\"```\\",\\n                \\"\\",\\n            ]\\n        )\\n\\n    if state.get(\\"notes\\"):\\n        lines.extend([\\"## Notes\\", \\"\\"])\\n        for note in state[\\"notes\\"]:\\n            lines.append(f\\"### {note.get(\'title\', \'note\')}\\")\\n            lines.append(\\"\\")\\n            lines.append(note.get(\\"body\\", \\"\\"))\\n            lines.append(\\"\\")\\n\\n    summary_path = _summary_path(run_dir)\\n    summary_path.write_text(\\"\\\\n\\".join(lines), encoding=\\"utf-8\\")\\n    return summary_path\\n", "code/training/hash_kws_lab/export.py": "from __future__ import annotations\\n\\nimport json\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nimport torch\\nimport torch.nn as nn\\n\\nfrom .config import ExperimentConfig\\nfrom .models import collect_layer_inventory, summarize_model\\n\\n\\ndef _cpu_state_dict(model: nn.Module) -> dict[str, torch.Tensor]:\\n    return {\\n        key: value.detach().cpu().clone()\\n        for key, value in model.state_dict().items()\\n    }\\n\\n\\ndef export_model_bundle(\\n    model: nn.Module,\\n    experiment: ExperimentConfig,\\n    stage_name: str = \\"student\\",\\n) -> dict[str, Any]:\\n    artifact_dir = Path(experiment.export.artifacts_dir) / experiment.tag\\n    artifact_dir.mkdir(parents=True, exist_ok=True)\\n\\n    model_stem = f\\"{experiment.export.model_stem}_{stage_name}\\"\\n    bundle_path = artifact_dir / f\\"{model_stem}.pt\\"\\n    metadata_path = artifact_dir / f\\"{model_stem}_metadata.json\\"\\n\\n    model_summary = summarize_model(model, experiment)\\n    bundle = {\\n        \\"experiment\\": experiment.to_dict(),\\n        \\"stage_name\\": stage_name,\\n        \\"model_summary\\": model_summary,\\n        \\"layer_inventory\\": collect_layer_inventory(model),\\n        \\"state_dict\\": _cpu_state_dict(model),\\n    }\\n    torch.save(bundle, bundle_path)\\n\\n    metadata = {\\n        \\"experiment\\": experiment.to_dict(),\\n        \\"stage_name\\": stage_name,\\n        \\"bundle_path\\": str(bundle_path),\\n        \\"model_summary\\": model_summary,\\n        \\"layer_inventory\\": collect_layer_inventory(model),\\n    }\\n    metadata_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding=\\"utf-8\\")\\n    return {\\n        \\"bundle\\": {\\n            \\"path\\": str(bundle_path),\\n            \\"metadata_path\\": str(metadata_path),\\n        },\\n        \\"model_summary\\": model_summary,\\n    }\\n", "code/training/hashednet95/__init__.py": "from .hashednet95_recipes import (\\n    build_hashednet95_recipe_book,\\n    describe_hashednet95_recipe,\\n    with_drive_cache_paths,\\n)\\n\\n__all__ = [\\n    \\"build_hashednet95_recipe_book\\",\\n    \\"describe_hashednet95_recipe\\",\\n    \\"with_drive_cache_paths\\",\\n]\\n", "code/training/hashednet95/hashednet95_recipes.py": "from __future__ import annotations\\n\\nimport os\\nfrom dataclasses import replace\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nfrom hash_kws_lab.config import ExperimentConfig, make_experiment\\nfrom hash_kws_lab.models import build_student_model, summarize_model\\nfrom hash_kws_lab.recipes import build_recipe_book as build_hash_recipe_book\\n\\n\\ndef _tuned_exact_reference(base: ExperimentConfig) -> ExperimentConfig:\\n    return build_hash_recipe_book(base)[\\"hash_deeper_fair_ce_exact_microfrontend_tuned\\"]\\n\\n\\ndef _exact_cached(\\n    experiment: ExperimentConfig,\\n    *,\\n    batch_size: int,\\n    epochs: int,\\n    patience: int,\\n    label_smoothing: float = 0.02,\\n) -> ExperimentConfig:\\n    return replace(\\n        experiment,\\n        dataset=replace(\\n            experiment.dataset,\\n            batch_size=batch_size,\\n            num_workers=2,\\n            cache_features=True,\\n            cache_dtype=\\"int8\\",\\n            time_shift_ms=0,\\n            gain_min=1.0,\\n            gain_max=1.0,\\n            noise_stddev=0.0,\\n        ),\\n        feature=replace(\\n            experiment.feature,\\n            frontend_name=\\"exact_microfrontend\\",\\n            normalize_mode=\\"none\\",\\n            require_exact_microfrontend=True,\\n        ),\\n        train=replace(\\n            experiment.train,\\n            student_epochs=epochs,\\n            student_scheduler_name=\\"cosine\\",\\n            label_smoothing=label_smoothing,\\n            grad_clip_norm=1.0,\\n            student_early_stopping_patience=patience,\\n            use_ema=False,\\n            eval_with_ema=False,\\n        ),\\n    )\\n\\n\\ndef build_hashednet95_recipe_book(base: ExperimentConfig) -> dict[str, ExperimentConfig]:\\n    \\"\\"\\"High-upside HashedNet-style recipes for the deploy-aligned KWS branch.\\n\\n    These recipes keep the exact firmware microfrontend and current custom\\n    ESP32 hash-runtime format, while adding the missing HashedNets levers:\\n    signed hashing, per-layer codebook budgets, and virtual-width inflation.\\n    \\"\\"\\"\\n\\n    reference = _tuned_exact_reference(base)\\n\\n    hn95_paper_inflate96_fixedk = _exact_cached(\\n        replace(\\n            reference,\\n            tag=f\\"{base.tag}_hn95_paper_inflate96_fixedk\\",\\n            model=replace(\\n                reference.model,\\n                channels=96,\\n                num_blocks=4,\\n                stem_codebook_size=500,\\n                depthwise_codebook_size=500,\\n                pointwise_codebook_size=500,\\n                linear_codebook_size=500,\\n                depthwise_codebook_sizes=(),\\n                pointwise_codebook_sizes=(),\\n                signed_hash=True,\\n                use_residual=False,\\n            ),\\n        ),\\n        batch_size=128,\\n        epochs=48,\\n        patience=8,\\n    )\\n\\n    hn95_layerwise96_signed_residual = _exact_cached(\\n        replace(\\n            reference,\\n            tag=f\\"{base.tag}_hn95_layerwise96_signed_residual\\",\\n            model=replace(\\n                reference.model,\\n                channels=96,\\n                num_blocks=4,\\n                stem_codebook_size=384,\\n                depthwise_codebook_size=192,\\n                pointwise_codebook_size=1024,\\n                linear_codebook_size=384,\\n                depthwise_codebook_sizes=(224, 192, 160, 128),\\n                pointwise_codebook_sizes=(896, 1024, 1152, 1280),\\n                signed_hash=True,\\n                use_residual=True,\\n            ),\\n        ),\\n        batch_size=128,\\n        epochs=64,\\n        patience=10,\\n    )\\n\\n    hn95_layerwise96_signed_cached_specaug = replace(\\n        hn95_layerwise96_signed_residual,\\n        tag=f\\"{base.tag}_hn95_layerwise96_signed_cached_specaug\\",\\n        feature=replace(\\n            hn95_layerwise96_signed_residual.feature,\\n            specaugment_prob=0.25,\\n            time_mask_max=4,\\n            freq_mask_max=3,\\n        ),\\n        train=replace(\\n            hn95_layerwise96_signed_residual.train,\\n            label_smoothing=0.03,\\n        ),\\n    )\\n\\n    hn95_inflate128_fixed_storage_paper = _exact_cached(\\n        replace(\\n            reference,\\n            tag=f\\"{base.tag}_hn95_inflate128_fixed_storage_paper\\",\\n            model=replace(\\n                reference.model,\\n                channels=128,\\n                num_blocks=4,\\n                stem_codebook_size=500,\\n                depthwise_codebook_size=500,\\n                pointwise_codebook_size=500,\\n                linear_codebook_size=500,\\n                depthwise_codebook_sizes=(),\\n                pointwise_codebook_sizes=(),\\n                signed_hash=True,\\n                use_residual=False,\\n            ),\\n        ),\\n        batch_size=96,\\n        epochs=64,\\n        patience=10,\\n    )\\n\\n    hn95_inflate128_layerwise_signed_residual = _exact_cached(\\n        replace(\\n            reference,\\n            tag=f\\"{base.tag}_hn95_inflate128_layerwise_signed_residual\\",\\n            model=replace(\\n                reference.model,\\n                channels=128,\\n                num_blocks=4,\\n                stem_codebook_size=512,\\n                depthwise_codebook_size=224,\\n                pointwise_codebook_size=1536,\\n                linear_codebook_size=512,\\n                depthwise_codebook_sizes=(288, 256, 224, 192),\\n                pointwise_codebook_sizes=(1280, 1536, 1792, 2048),\\n                signed_hash=True,\\n                use_residual=True,\\n            ),\\n        ),\\n        batch_size=96,\\n        epochs=80,\\n        patience=12,\\n    )\\n\\n    hn95_3block128_big_pointwise_signed = _exact_cached(\\n        replace(\\n            reference,\\n            tag=f\\"{base.tag}_hn95_3block128_big_pointwise_signed\\",\\n            model=replace(\\n                reference.model,\\n                channels=128,\\n                num_blocks=3,\\n                stem_codebook_size=512,\\n                depthwise_codebook_size=256,\\n                pointwise_codebook_size=2048,\\n                linear_codebook_size=512,\\n                depthwise_codebook_sizes=(288, 256, 224),\\n                pointwise_codebook_sizes=(1536, 2048, 2304),\\n                signed_hash=True,\\n                use_residual=True,\\n            ),\\n        ),\\n        batch_size=96,\\n        epochs=80,\\n        patience=12,\\n    )\\n\\n    hn95_kd128_layerwise_signed_residual = replace(\\n        hn95_inflate128_layerwise_signed_residual,\\n        tag=f\\"{base.tag}_hn95_kd128_layerwise_signed_residual\\",\\n        teacher_reuse_tag=f\\"{base.tag}_hn95_kd128_layerwise_signed_residual\\",\\n        model=replace(\\n            hn95_inflate128_layerwise_signed_residual.model,\\n            teacher_name=\\"dense_dscnn_teacher\\",\\n            teacher_channels=128,\\n            teacher_num_blocks=6,\\n            teacher_dropout=0.05,\\n        ),\\n        train=replace(\\n            hn95_inflate128_layerwise_signed_residual.train,\\n            teacher_epochs=48,\\n            student_pretrain_epochs=6,\\n            student_epochs=40,\\n            student_polish_epochs=4,\\n            teacher_lr=8e-4,\\n            student_lr=8e-4,\\n            polish_lr=2e-4,\\n            teacher_scheduler_name=\\"cosine\\",\\n            student_scheduler_name=\\"cosine\\",\\n            teacher_label_smoothing=0.02,\\n            label_smoothing=0.02,\\n            kd_alpha=0.55,\\n            kd_temperature=4.5,\\n            teacher_early_stopping_patience=8,\\n            student_early_stopping_patience=10,\\n        ),\\n    )\\n\\n    hn95_kd128_cached_schedule = replace(\\n        hn95_kd128_layerwise_signed_residual,\\n        tag=f\\"{base.tag}_hn95_kd128_cached_schedule\\",\\n        teacher_reuse_tag=f\\"{base.tag}_hn95_kd128_layerwise_signed_residual\\",\\n        train=replace(\\n            hn95_kd128_layerwise_signed_residual.train,\\n            student_epochs=52,\\n            student_polish_epochs=8,\\n            label_smoothing=0.025,\\n            polish_label_smoothing=0.0,\\n            kd_alpha=0.65,\\n            kd_alpha_schedule=\\"cosine\\",\\n            kd_alpha_final=0.35,\\n            kd_temperature=5.5,\\n            kd_temperature_schedule=\\"cosine\\",\\n            kd_temperature_final=3.0,\\n            cache_teacher_logits=True,\\n            teacher_logits_cache_dtype=\\"float16\\",\\n            student_early_stopping_patience=12,\\n        ),\\n    )\\n\\n    hn95_kd128_rich_teacher160_cached_schedule = replace(\\n        hn95_kd128_cached_schedule,\\n        tag=f\\"{base.tag}_hn95_kd128_rich_teacher160_cached_schedule\\",\\n        teacher_reuse_tag=f\\"{base.tag}_hn95_teacher160_dense_reusable\\",\\n        model=replace(\\n            hn95_kd128_cached_schedule.model,\\n            teacher_channels=160,\\n            teacher_num_blocks=7,\\n            teacher_dropout=0.06,\\n        ),\\n        train=replace(\\n            hn95_kd128_cached_schedule.train,\\n            teacher_epochs=72,\\n            teacher_lr=6e-4,\\n            teacher_label_smoothing=0.035,\\n            teacher_early_stopping_patience=10,\\n        ),\\n    )\\n\\n    hn95_kd112_latency_balanced_cached_schedule = replace(\\n        hn95_kd128_cached_schedule,\\n        tag=f\\"{base.tag}_hn95_kd112_latency_balanced_cached_schedule\\",\\n        model=replace(\\n            hn95_kd128_cached_schedule.model,\\n            channels=112,\\n            stem_codebook_size=448,\\n            depthwise_codebook_size=224,\\n            pointwise_codebook_size=1280,\\n            linear_codebook_size=448,\\n            depthwise_codebook_sizes=(256, 224, 192, 160),\\n            pointwise_codebook_sizes=(1024, 1280, 1536, 1792),\\n        ),\\n        dataset=replace(\\n            hn95_kd128_cached_schedule.dataset,\\n            batch_size=112,\\n        ),\\n        train=replace(\\n            hn95_kd128_cached_schedule.train,\\n            student_epochs=50,\\n            student_polish_epochs=6,\\n        ),\\n    )\\n\\n    hn95_kd96_latency_guard_cached_schedule = replace(\\n        hn95_kd128_cached_schedule,\\n        tag=f\\"{base.tag}_hn95_kd96_latency_guard_cached_schedule\\",\\n        model=replace(\\n            hn95_kd128_cached_schedule.model,\\n            channels=96,\\n            stem_codebook_size=384,\\n            depthwise_codebook_size=192,\\n            pointwise_codebook_size=1024,\\n            linear_codebook_size=384,\\n            depthwise_codebook_sizes=(224, 192, 160, 128),\\n            pointwise_codebook_sizes=(896, 1024, 1152, 1280),\\n        ),\\n        dataset=replace(\\n            hn95_kd128_cached_schedule.dataset,\\n            batch_size=128,\\n        ),\\n        train=replace(\\n            hn95_kd128_cached_schedule.train,\\n            student_epochs=48,\\n            student_polish_epochs=6,\\n            student_early_stopping_patience=10,\\n        ),\\n    )\\n\\n    hn95_kd128_pw2048_budget_cached_schedule = replace(\\n        hn95_kd128_cached_schedule,\\n        tag=f\\"{base.tag}_hn95_kd128_pw2048_budget_cached_schedule\\",\\n        model=replace(\\n            hn95_kd128_cached_schedule.model,\\n            stem_codebook_size=448,\\n            depthwise_codebook_size=192,\\n            pointwise_codebook_size=2048,\\n            linear_codebook_size=640,\\n            depthwise_codebook_sizes=(256, 224, 192, 160),\\n            pointwise_codebook_sizes=(1536, 1792, 2048, 2304),\\n        ),\\n        train=replace(\\n            hn95_kd128_cached_schedule.train,\\n            student_epochs=56,\\n            student_polish_epochs=8,\\n        ),\\n    )\\n\\n    hn95_kd128_hard_polish_cached = replace(\\n        hn95_kd128_cached_schedule,\\n        tag=f\\"{base.tag}_hn95_kd128_hard_polish_cached\\",\\n        train=replace(\\n            hn95_kd128_cached_schedule.train,\\n            label_smoothing=0.015,\\n            student_epochs=50,\\n            student_polish_epochs=10,\\n            polish_lr=1.2e-4,\\n            polish_label_smoothing=0.0,\\n        ),\\n    )\\n\\n    hn95_kd128_cached_schedule_s29 = replace(\\n        hn95_kd128_cached_schedule,\\n        tag=f\\"{base.tag}_hn95_kd128_cached_schedule_s29\\",\\n        train=replace(\\n            hn95_kd128_cached_schedule.train,\\n            seed=29,\\n        ),\\n    )\\n\\n    hn95_kd128_cached_schedule_s47 = replace(\\n        hn95_kd128_cached_schedule,\\n        tag=f\\"{base.tag}_hn95_kd128_cached_schedule_s47\\",\\n        train=replace(\\n            hn95_kd128_cached_schedule.train,\\n            seed=47,\\n        ),\\n    )\\n\\n    return {\\n        \\"hn95_paper_inflate96_fixedk\\": hn95_paper_inflate96_fixedk,\\n        \\"hn95_layerwise96_signed_residual\\": hn95_layerwise96_signed_residual,\\n        \\"hn95_layerwise96_signed_cached_specaug\\": hn95_layerwise96_signed_cached_specaug,\\n        \\"hn95_inflate128_fixed_storage_paper\\": hn95_inflate128_fixed_storage_paper,\\n        \\"hn95_inflate128_layerwise_signed_residual\\": hn95_inflate128_layerwise_signed_residual,\\n        \\"hn95_3block128_big_pointwise_signed\\": hn95_3block128_big_pointwise_signed,\\n        \\"hn95_kd128_layerwise_signed_residual\\": hn95_kd128_layerwise_signed_residual,\\n        \\"hn95_kd128_cached_schedule\\": hn95_kd128_cached_schedule,\\n        \\"hn95_kd128_rich_teacher160_cached_schedule\\": hn95_kd128_rich_teacher160_cached_schedule,\\n        \\"hn95_kd112_latency_balanced_cached_schedule\\": hn95_kd112_latency_balanced_cached_schedule,\\n        \\"hn95_kd96_latency_guard_cached_schedule\\": hn95_kd96_latency_guard_cached_schedule,\\n        \\"hn95_kd128_pw2048_budget_cached_schedule\\": hn95_kd128_pw2048_budget_cached_schedule,\\n        \\"hn95_kd128_hard_polish_cached\\": hn95_kd128_hard_polish_cached,\\n        \\"hn95_kd128_cached_schedule_s29\\": hn95_kd128_cached_schedule_s29,\\n        \\"hn95_kd128_cached_schedule_s47\\": hn95_kd128_cached_schedule_s47,\\n    }\\n\\n\\ndef with_drive_cache_paths(\\n    experiment: ExperimentConfig,\\n    drive_cache_root: str | Path,\\n) -> ExperimentConfig:\\n    \\"\\"\\"Persist Speech Commands, exact-feature cache, and exports on Drive.\\"\\"\\"\\n\\n    root = Path(drive_cache_root)\\n    data_root = Path(os.environ.get(\\"SPEECHCOMMANDS_DATA_ROOT\\", root / \\"speechcommands_v2\\"))\\n    feature_cache_root = root / \\"hash_feature_cache\\"\\n    teacher_logits_cache_root = root / \\"teacher_logits_cache\\"\\n    artifact_root = root / \\"hash_artifacts\\"\\n    data_root.mkdir(parents=True, exist_ok=True)\\n    feature_cache_root.mkdir(parents=True, exist_ok=True)\\n    teacher_logits_cache_root.mkdir(parents=True, exist_ok=True)\\n    artifact_root.mkdir(parents=True, exist_ok=True)\\n    os.environ.setdefault(\\"SPEECHCOMMANDS_DATA_ROOT\\", str(data_root))\\n    os.environ.setdefault(\\"HASH_KWS_FEATURE_CACHE_ROOT\\", str(feature_cache_root))\\n    os.environ.setdefault(\\"HASH_KWS_TEACHER_LOGITS_CACHE_ROOT\\", str(teacher_logits_cache_root))\\n    return replace(\\n        experiment,\\n        train=replace(\\n            experiment.train,\\n            teacher_logits_cache_dir=str(teacher_logits_cache_root),\\n        ),\\n        export=replace(\\n            experiment.export,\\n            artifacts_dir=str(artifact_root),\\n        ),\\n    )\\n\\n\\ndef describe_hashednet95_recipe(experiment: ExperimentConfig) -> dict[str, Any]:\\n    student = build_student_model(experiment)\\n    summary = summarize_model(student, experiment)\\n    reference = _tuned_exact_reference(\\n        make_experiment(tag=\\"hash_kws12_iterlab_v1\\", vocabulary_preset=experiment.vocabulary_preset)\\n    )\\n    reference_summary = summarize_model(build_student_model(reference), reference)\\n    virtual = int(summary[\\"virtual_dense_parameters\\"])\\n    compact = int(summary[\\"hash_compact_parameters\\"])\\n    ref_virtual = int(reference_summary[\\"virtual_dense_parameters\\"])\\n    ref_compact = int(reference_summary[\\"hash_compact_parameters\\"])\\n    return {\\n        \\"tag\\": experiment.tag,\\n        \\"labels\\": experiment.all_labels,\\n        \\"model_input_shape\\": experiment.model_input_shape,\\n        \\"channels\\": experiment.model.channels,\\n        \\"num_blocks\\": experiment.model.num_blocks,\\n        \\"signed_hash\\": experiment.model.signed_hash,\\n        \\"use_residual\\": experiment.model.use_residual,\\n        \\"train_seed\\": experiment.train.seed,\\n        \\"teacher_reuse_tag\\": experiment.teacher_reuse_tag,\\n        \\"teacher_name\\": experiment.model.teacher_name,\\n        \\"teacher_channels\\": experiment.model.teacher_channels,\\n        \\"teacher_num_blocks\\": experiment.model.teacher_num_blocks,\\n        \\"teacher_dropout\\": experiment.model.teacher_dropout,\\n        \\"kd_alpha\\": experiment.train.kd_alpha,\\n        \\"kd_alpha_schedule\\": experiment.train.kd_alpha_schedule,\\n        \\"kd_alpha_final\\": experiment.train.kd_alpha_final,\\n        \\"kd_temperature\\": experiment.train.kd_temperature,\\n        \\"kd_temperature_schedule\\": experiment.train.kd_temperature_schedule,\\n        \\"kd_temperature_final\\": experiment.train.kd_temperature_final,\\n        \\"cache_teacher_logits\\": experiment.train.cache_teacher_logits,\\n        \\"student_polish_epochs\\": experiment.train.student_polish_epochs,\\n        \\"polish_label_smoothing\\": experiment.train.polish_label_smoothing,\\n        \\"depthwise_codebook_sizes\\": list(experiment.model.depthwise_codebook_sizes),\\n        \\"pointwise_codebook_sizes\\": list(experiment.model.pointwise_codebook_sizes),\\n        \\"student_summary\\": summary,\\n        \\"reference_summary\\": reference_summary,\\n        \\"virtual_inflation_vs_reference\\": virtual / max(ref_virtual, 1),\\n        \\"compact_growth_vs_reference\\": compact / max(ref_compact, 1),\\n        \\"virtual_per_compact_parameter\\": virtual / max(compact, 1),\\n        \\"reference_virtual_per_compact_parameter\\": ref_virtual / max(ref_compact, 1),\\n    }\\n", "code/scripts/export_hash_kws_firmware.py": "from __future__ import annotations\\n\\nimport argparse\\nimport json\\nimport sys\\nfrom dataclasses import asdict\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nimport torch\\nimport torch.nn as nn\\n\\n\\nREPO_ROOT = Path(__file__).resolve().parents[2]\\nTRAINING_ROOT = REPO_ROOT / \\"code\\" / \\"training\\"\\nif str(TRAINING_ROOT) not in sys.path:\\n    sys.path.insert(0, str(TRAINING_ROOT))\\n\\nfrom hash_kws_lab.config import ExperimentConfig, experiment_from_dict\\nfrom hash_kws_lab.data import prepare_dataloaders\\nfrom hash_kws_lab.models import (  # noqa: E402\\n    AnalyticHashConv2d,\\n    AnalyticHashDepthwiseConv2d,\\n    AnalyticHashLinear,\\n    HashDSCNN,\\n    build_student_model,\\n    summarize_model,\\n)\\n\\n\\ndef _parse_args() -> argparse.Namespace:\\n    parser = argparse.ArgumentParser(\\n        description=\\"Export a trained hash-KWS PyTorch bundle into ESP32 firmware arrays.\\",\\n    )\\n    parser.add_argument(\\n        \\"--bundle\\",\\n        type=Path,\\n        required=True,\\n        help=\\"Path to the .pt bundle produced by the hash KWS lab.\\",\\n    )\\n    parser.add_argument(\\n        \\"--output-dir\\",\\n        type=Path,\\n        default=REPO_ROOT / \\"code\\" / \\"firmware\\" / \\"hash_kws_runtime\\",\\n        help=\\"Directory where hash_model_data.cpp and metadata will be written.\\",\\n    )\\n    parser.add_argument(\\n        \\"--project-root\\",\\n        type=Path,\\n        default=REPO_ROOT,\\n        help=\\"Repository root for resolving data paths.\\",\\n    )\\n    parser.add_argument(\\n        \\"--device\\",\\n        type=str,\\n        default=\\"auto\\",\\n        choices=[\\"auto\\", \\"cpu\\", \\"cuda\\"],\\n        help=\\"Device used for calibration batches.\\",\\n    )\\n    parser.add_argument(\\n        \\"--calibration-split\\",\\n        type=str,\\n        default=\\"validation\\",\\n        choices=[\\"train\\", \\"validation\\", \\"test\\"],\\n        help=\\"Dataset split used for activation calibration.\\",\\n    )\\n    parser.add_argument(\\n        \\"--calibration-batches\\",\\n        type=int,\\n        default=8,\\n        help=\\"How many batches to use for activation scale calibration.\\",\\n    )\\n    return parser.parse_args()\\n\\n\\ndef _resolve_device(name: str) -> torch.device:\\n    if name == \\"cpu\\":\\n        return torch.device(\\"cpu\\")\\n    if name == \\"cuda\\":\\n        if not torch.cuda.is_available():\\n            raise RuntimeError(\\"CUDA was requested for calibration, but it is not available.\\")\\n        return torch.device(\\"cuda\\")\\n    return torch.device(\\"cuda\\" if torch.cuda.is_available() else \\"cpu\\")\\n\\n\\ndef _load_bundle(bundle_path: Path) -> tuple[ExperimentConfig, dict[str, Any], nn.Module]:\\n    bundle = torch.load(bundle_path, map_location=\\"cpu\\")\\n    experiment_payload = bundle.get(\\"experiment\\")\\n    if not isinstance(experiment_payload, dict):\\n        raise ValueError(\\"Bundle does not contain a serialized experiment payload.\\")\\n\\n    experiment = experiment_from_dict(experiment_payload)\\n    model = build_student_model(experiment)\\n    state_dict = bundle.get(\\"state_dict\\")\\n    if not isinstance(state_dict, dict):\\n        raise ValueError(\\"Bundle does not contain a state_dict.\\")\\n    model.load_state_dict(state_dict, strict=True)\\n    model.eval()\\n    return experiment, bundle, model\\n\\n\\ndef _ensure_exportable_hash_model(model: nn.Module, experiment: ExperimentConfig) -> HashDSCNN:\\n    if not isinstance(model, HashDSCNN):\\n        raise TypeError(f\\"Expected HashDSCNN student model, got {type(model).__name__}\\")\\n    if experiment.model.hash_only_pointwise:\\n        raise NotImplementedError(\\n            \\"Firmware exporter currently supports fully hashed models only. \\"\\n            \\"The pointwise-only recipe needs dense-kernel support in the runtime.\\"\\n        )\\n    if not isinstance(model.conv0, AnalyticHashConv2d):\\n        raise TypeError(\\"Runtime exporter expects a hashed stem convolution.\\")\\n    for block in model.blocks:\\n        if not isinstance(block.depthwise, AnalyticHashDepthwiseConv2d):\\n            raise TypeError(\\"Runtime exporter expects hashed depthwise blocks.\\")\\n        if not isinstance(block.pointwise, AnalyticHashConv2d):\\n            raise TypeError(\\"Runtime exporter expects hashed pointwise blocks.\\")\\n    if not isinstance(model.fc, AnalyticHashLinear):\\n        raise TypeError(\\"Runtime exporter expects a hashed linear classifier.\\")\\n    return model\\n\\n\\ndef _bn_fold(conv_bias: torch.Tensor, bn: nn.BatchNorm2d) -> tuple[torch.Tensor, torch.Tensor]:\\n    gamma = bn.weight.detach().cpu().to(torch.float32)\\n    beta = bn.bias.detach().cpu().to(torch.float32)\\n    running_mean = bn.running_mean.detach().cpu().to(torch.float32)\\n    running_var = bn.running_var.detach().cpu().to(torch.float32)\\n    denom = torch.sqrt(running_var + float(bn.eps))\\n    post_scale = gamma / denom\\n    post_bias = beta + post_scale * (conv_bias.detach().cpu().to(torch.float32) - running_mean)\\n    return post_scale, post_bias\\n\\n\\ndef _quantize_codebook(codebook: torch.Tensor) -> tuple[list[int], float]:\\n    values = codebook.detach().cpu().to(torch.float32).view(-1)\\n    max_abs = float(values.abs().max().item())\\n    if max_abs <= 1e-12:\\n        return [0 for _ in range(values.numel())], 1.0\\n    scale = max_abs / 127.0\\n    quantized = torch.clamp(torch.round(values / scale), min=-127, max=127).to(torch.int8)\\n    return [int(item) for item in quantized.tolist()], float(scale)\\n\\n\\ndef _activation_forward(model: HashDSCNN, features: torch.Tensor) -> tuple[torch.Tensor, list[torch.Tensor]]:\\n    if features.dim() == 3:\\n        features = features.unsqueeze(1)\\n    stages: list[torch.Tensor] = []\\n    x = features\\n    x = torch.relu(model.bn0(model.conv0(x)))\\n    stages.append(x)\\n    for block in model.blocks:\\n        shortcut = x\\n        x = torch.relu(block.bn_dw(block.depthwise(x)))\\n        stages.append(x)\\n        x = block.bn_pw(block.pointwise(x))\\n        if block.residual:\\n            x = x + shortcut\\n        x = torch.relu(x)\\n        stages.append(x)\\n    return features, stages\\n\\n\\n@torch.no_grad()\\ndef _collect_activation_quant_params(\\n    model: HashDSCNN,\\n    loader: torch.utils.data.DataLoader[Any],\\n    device: torch.device,\\n    max_batches: int,\\n) -> list[dict[str, float]]:\\n    if max_batches <= 0:\\n        raise ValueError(\\"calibration-batches must be positive\\")\\n\\n    stage_names = [\\"stem\\"]\\n    for block_index in range(len(model.blocks)):\\n        stage_names.append(f\\"block{block_index}_depthwise\\")\\n        stage_names.append(f\\"block{block_index}_pointwise\\")\\n\\n    model = model.to(device)\\n    model.eval()\\n    input_abs_max = 0.0\\n    stage_abs_max = [0.0 for _ in stage_names]\\n\\n    for batch_index, (features, _) in enumerate(loader):\\n        if batch_index >= max_batches:\\n            break\\n        features = features.to(device=device, dtype=torch.float32, non_blocking=(device.type == \\"cuda\\"))\\n        input_tensor, stages = _activation_forward(model, features)\\n        input_abs_max = max(input_abs_max, float(input_tensor.abs().amax().item()))\\n        for index, stage in enumerate(stages):\\n            stage_abs_max[index] = max(stage_abs_max[index], float(stage.abs().amax().item()))\\n\\n    if input_abs_max <= 1e-12:\\n        input_abs_max = 1.0\\n    stage_output_scales = [max(value / 127.0, 1e-8) for value in stage_abs_max]\\n    input_scale = max(input_abs_max / 127.0, 1e-8)\\n\\n    quant_params: list[dict[str, float]] = []\\n    previous_scale = input_scale\\n    for stage_name, output_scale in zip(stage_names, stage_output_scales):\\n        quant_params.append(\\n            {\\n                \\"stage\\": stage_name,\\n                \\"input_scale\\": float(previous_scale),\\n                \\"output_scale\\": float(output_scale),\\n            }\\n        )\\n        previous_scale = output_scale\\n    return quant_params\\n\\n\\ndef _float_list(tensor: torch.Tensor) -> list[float]:\\n    return [float(value) for value in tensor.detach().cpu().to(torch.float32).view(-1).tolist()]\\n\\n\\ndef _format_cpp_float(value: float) -> str:\\n    text = f\\"{value:.9g}\\"\\n    if text in {\\"0\\", \\"-0\\"}:\\n        text = \\"0.0\\"\\n    elif all(marker not in text for marker in (\\".\\", \\"e\\", \\"E\\")):\\n        text = f\\"{text}.0\\"\\n    return f\\"{text}f\\"\\n\\n\\ndef _format_cpp_array(name: str, ctype: str, values: list[str], values_per_line: int = 8) -> str:\\n    lines: list[str] = [f\\"constexpr {ctype} {name}[] = {{\\"]\\n    for start in range(0, len(values), values_per_line):\\n        chunk = values[start : start + values_per_line]\\n        lines.append(\\"    \\" + \\", \\".join(chunk) + \\",\\")\\n    lines.append(\\"};\\")\\n    return \\"\\\\n\\".join(lines)\\n\\n\\ndef _emit_conv_layer(layer_name: str, layer: AnalyticHashConv2d, post_scale: list[float], post_bias: list[float]) -> tuple[str, str]:\\n    quantized_codebook, codebook_scale = _quantize_codebook(layer.codebook)\\n    codebook_name = f\\"k{layer_name}Codebook\\"\\n    scale_name = f\\"k{layer_name}PostScale\\"\\n    bias_name = f\\"k{layer_name}PostBias\\"\\n    arrays = [\\n        _format_cpp_array(codebook_name, \\"int8_t\\", [str(value) for value in quantized_codebook], values_per_line=16),\\n        _format_cpp_array(scale_name, \\"float\\", [_format_cpp_float(value) for value in post_scale]),\\n        _format_cpp_array(bias_name, \\"float\\", [_format_cpp_float(value) for value in post_bias]),\\n    ]\\n    initializer = \\"\\\\n\\".join(\\n        [\\n            \\"{\\",\\n            f\\"    {codebook_name},\\",\\n            f\\"    {_format_cpp_float(codebook_scale)},\\",\\n            f\\"    {scale_name},\\",\\n            f\\"    {bias_name},\\",\\n            f\\"    {layer.codebook_size},\\",\\n            f\\"    {layer.in_channels},\\",\\n            f\\"    {layer.out_channels},\\",\\n            f\\"    {layer.kernel_size[0]},\\",\\n            f\\"    {layer.kernel_size[1]},\\",\\n            f\\"    {layer.stride[0]},\\",\\n            f\\"    {layer.stride[1]},\\",\\n            f\\"    {layer.padding[0]},\\",\\n            f\\"    {layer.padding[1]},\\",\\n            f\\"    {layer.layer_id},\\",\\n            f\\"    {\'true\' if layer.signed_hash else \'false\'},\\",\\n            \\"}\\",\\n        ]\\n    )\\n    return \\"\\\\n\\\\n\\".join(arrays), initializer\\n\\n\\ndef _emit_depthwise_layer(\\n    layer_name: str,\\n    layer: AnalyticHashDepthwiseConv2d,\\n    post_scale: list[float],\\n    post_bias: list[float],\\n) -> tuple[str, str]:\\n    quantized_codebook, codebook_scale = _quantize_codebook(layer.codebook)\\n    codebook_name = f\\"k{layer_name}Codebook\\"\\n    scale_name = f\\"k{layer_name}PostScale\\"\\n    bias_name = f\\"k{layer_name}PostBias\\"\\n    arrays = [\\n        _format_cpp_array(codebook_name, \\"int8_t\\", [str(value) for value in quantized_codebook], values_per_line=16),\\n        _format_cpp_array(scale_name, \\"float\\", [_format_cpp_float(value) for value in post_scale]),\\n        _format_cpp_array(bias_name, \\"float\\", [_format_cpp_float(value) for value in post_bias]),\\n    ]\\n    initializer = \\"\\\\n\\".join(\\n        [\\n            \\"{\\",\\n            f\\"    {codebook_name},\\",\\n            f\\"    {_format_cpp_float(codebook_scale)},\\",\\n            f\\"    {scale_name},\\",\\n            f\\"    {bias_name},\\",\\n            f\\"    {layer.codebook_size},\\",\\n            f\\"    {layer.channels},\\",\\n            f\\"    {layer.kernel_size[0]},\\",\\n            f\\"    {layer.kernel_size[1]},\\",\\n            f\\"    {layer.stride[0]},\\",\\n            f\\"    {layer.stride[1]},\\",\\n            f\\"    {layer.padding[0]},\\",\\n            f\\"    {layer.padding[1]},\\",\\n            f\\"    {layer.layer_id},\\",\\n            f\\"    {\'true\' if layer.signed_hash else \'false\'},\\",\\n            \\"}\\",\\n        ]\\n    )\\n    return \\"\\\\n\\\\n\\".join(arrays), initializer\\n\\n\\ndef _emit_linear_layer(layer_name: str, layer: AnalyticHashLinear) -> tuple[str, str]:\\n    quantized_codebook, codebook_scale = _quantize_codebook(layer.codebook)\\n    codebook_name = f\\"k{layer_name}Codebook\\"\\n    bias_name = f\\"k{layer_name}Bias\\"\\n    arrays = [\\n        _format_cpp_array(codebook_name, \\"int8_t\\", [str(value) for value in quantized_codebook], values_per_line=16),\\n        _format_cpp_array(bias_name, \\"float\\", [_format_cpp_float(value) for value in _float_list(layer.bias)]),\\n    ]\\n    initializer = \\"\\\\n\\".join(\\n        [\\n            \\"{\\",\\n            f\\"    {codebook_name},\\",\\n            f\\"    {_format_cpp_float(codebook_scale)},\\",\\n            f\\"    {bias_name},\\",\\n            f\\"    {layer.codebook_size},\\",\\n            f\\"    {layer.in_dim},\\",\\n            f\\"    {layer.out_dim},\\",\\n            f\\"    {layer.layer_id},\\",\\n            f\\"    {\'true\' if layer.signed_hash else \'false\'},\\",\\n            \\"}\\",\\n        ]\\n    )\\n    return \\"\\\\n\\\\n\\".join(arrays), initializer\\n\\n\\ndef _build_cpp_source(\\n    experiment: ExperimentConfig,\\n    model: HashDSCNN,\\n    quant_params: list[dict[str, float]],\\n    source_bundle: Path,\\n) -> str:\\n    sections: list[str] = []\\n\\n    stem_post_scale, stem_post_bias = _bn_fold(model.conv0.bias, model.bn0)\\n    stem_arrays, stem_initializer = _emit_conv_layer(\\n        \\"Stem\\",\\n        model.conv0,\\n        _float_list(stem_post_scale),\\n        _float_list(stem_post_bias),\\n    )\\n    sections.append(stem_arrays)\\n\\n    depthwise_initializers: list[str] = []\\n    pointwise_initializers: list[str] = []\\n    residual_values: list[str] = []\\n    for block_index, block in enumerate(model.blocks):\\n        dw_post_scale, dw_post_bias = _bn_fold(block.depthwise.bias, block.bn_dw)\\n        dw_arrays, dw_initializer = _emit_depthwise_layer(\\n            f\\"Block{block_index}Depthwise\\",\\n            block.depthwise,\\n            _float_list(dw_post_scale),\\n            _float_list(dw_post_bias),\\n        )\\n        pw_post_scale, pw_post_bias = _bn_fold(block.pointwise.bias, block.bn_pw)\\n        pw_arrays, pw_initializer = _emit_conv_layer(\\n            f\\"Block{block_index}Pointwise\\",\\n            block.pointwise,\\n            _float_list(pw_post_scale),\\n            _float_list(pw_post_bias),\\n        )\\n        sections.append(dw_arrays)\\n        sections.append(pw_arrays)\\n        depthwise_initializers.append(dw_initializer)\\n        pointwise_initializers.append(pw_initializer)\\n        residual_values.append(\\"true\\" if block.residual else \\"false\\")\\n\\n    classifier_arrays, classifier_initializer = _emit_linear_layer(\\"Classifier\\", model.fc)\\n    sections.append(classifier_arrays)\\n\\n    activation_values: list[str] = []\\n    for quant in quant_params:\\n        activation_values.append(\\n            \\"{\\"\\n            f\\"{_format_cpp_float(quant[\'input_scale\'])}, \\"\\n            f\\"{_format_cpp_float(quant[\'output_scale\'])}\\"\\n            \\"}\\"\\n        )\\n    activations_initializer = \\",\\\\n        \\".join(activation_values)\\n\\n    body = [\\n        \'#include \\"hash_model_data.h\\"\',\\n        \\"\\",\\n        \\"namespace hash_kws {\\",\\n        \\"\\",\\n        \\"namespace {\\",\\n        \\"\\",\\n        f\\"// Generated from bundle: {source_bundle.as_posix()}\\",\\n        f\\"// Experiment tag: {experiment.tag}\\",\\n        f\\"// Frontend in training bundle: {experiment.feature.frontend_name}\\",\\n        \\"// Note: runtime input semantics must match the training frontend for accuracy to hold.\\",\\n        \\"\\",\\n        \\"\\\\n\\\\n\\".join(sections),\\n        \\"\\",\\n        \\"}  // namespace\\",\\n        \\"\\",\\n        \\"const HashDscnnModelData g_hash_model = {\\",\\n        \\"    true,\\",\\n        f\\"    {experiment.feature.n_mels},\\",\\n        f\\"    {experiment.feature.frame_count},\\",\\n        \\"    1,\\",\\n        f\\"    {model.conv0.out_channels},\\",\\n        f\\"    {len(model.blocks)},\\",\\n        f\\"    {experiment.num_labels},\\",\\n        f\\"    {stem_initializer},\\",\\n        \\"    {\\",\\n        \\"        \\" + \\",\\\\n        \\".join(depthwise_initializers),\\n        \\"    },\\",\\n        \\"    {\\",\\n        \\"        \\" + \\",\\\\n        \\".join(pointwise_initializers),\\n        \\"    },\\",\\n        \\"    {\\",\\n        \\"        \\" + \\",\\\\n        \\".join(residual_values),\\n        \\"    },\\",\\n        f\\"    {classifier_initializer},\\",\\n        \\"    {\\",\\n        f\\"        {activations_initializer}\\",\\n        \\"    },\\",\\n        \\"};\\",\\n        \\"\\",\\n        \\"}  // namespace hash_kws\\",\\n        \\"\\",\\n    ]\\n    return \\"\\\\n\\".join(body)\\n\\n\\ndef _bundle_flash_summary(model: HashDSCNN, experiment: ExperimentConfig, quant_params: list[dict[str, float]]) -> dict[str, Any]:\\n    model_summary = summarize_model(model, experiment)\\n    codebook_bytes = 0\\n    affine_bytes = 0\\n\\n    codebook_bytes += int(model.conv0.codebook.numel())\\n    affine_bytes += 4 * (int(model.conv0.out_channels) * 2)\\n    for block in model.blocks:\\n        codebook_bytes += int(block.depthwise.codebook.numel())\\n        affine_bytes += 4 * (int(block.depthwise.channels) * 2)\\n        codebook_bytes += int(block.pointwise.codebook.numel())\\n        affine_bytes += 4 * (int(block.pointwise.out_channels) * 2)\\n    codebook_bytes += int(model.fc.codebook.numel())\\n    affine_bytes += 4 * int(model.fc.out_dim)\\n\\n    activation_bytes = len(quant_params) * 8\\n    return {\\n        \\"hash_codebook_bytes_int8\\": codebook_bytes,\\n        \\"post_affine_and_classifier_bias_bytes_float32\\": affine_bytes,\\n        \\"activation_quant_bytes_float32\\": activation_bytes,\\n        \\"approx_runtime_model_bytes\\": codebook_bytes + affine_bytes + activation_bytes,\\n        \\"virtual_dense_int8_weight_bytes\\": model_summary[\\"virtual_dense_parameters\\"],\\n    }\\n\\n\\ndef _metadata_payload(\\n    experiment: ExperimentConfig,\\n    bundle: dict[str, Any],\\n    model: HashDSCNN,\\n    quant_params: list[dict[str, float]],\\n    source_bundle: Path,\\n    output_dir: Path,\\n) -> dict[str, Any]:\\n    return {\\n        \\"source_bundle\\": str(source_bundle),\\n        \\"experiment\\": experiment.to_dict(),\\n        \\"bundle_stage_name\\": bundle.get(\\"stage_name\\", \\"student\\"),\\n        \\"frontend_warning\\": (\\n            \\"The exported runtime preserves model weights and activation scales, \\"\\n            \\"but on-device accuracy still depends on matching the training frontend.\\"\\n        ),\\n        \\"activation_quant_params\\": quant_params,\\n        \\"model_summary\\": summarize_model(model, experiment),\\n        \\"firmware_flash_summary\\": _bundle_flash_summary(model, experiment, quant_params),\\n        \\"runtime_output_dir\\": str(output_dir),\\n    }\\n\\n\\ndef export_bundle_to_firmware(\\n    bundle_path: Path,\\n    output_dir: Path,\\n    project_root: Path,\\n    device: torch.device,\\n    calibration_split: str = \\"validation\\",\\n    calibration_batches: int = 8,\\n) -> dict[str, Any]:\\n    bundle_path = bundle_path.resolve()\\n    output_dir = output_dir.resolve()\\n    project_root = project_root.resolve()\\n    output_dir.mkdir(parents=True, exist_ok=True)\\n\\n    experiment, bundle, model = _load_bundle(bundle_path)\\n    model = _ensure_exportable_hash_model(model, experiment)\\n\\n    calibration = prepare_dataloaders(\\n        project_root=project_root,\\n        experiment=experiment,\\n        device=device,\\n    )\\n    loader = calibration[\\"loaders\\"][calibration_split]\\n    quant_params = _collect_activation_quant_params(\\n        model=model,\\n        loader=loader,\\n        device=device,\\n        max_batches=calibration_batches,\\n    )\\n\\n    cpp_source = _build_cpp_source(\\n        experiment=experiment,\\n        model=model,\\n        quant_params=quant_params,\\n        source_bundle=bundle_path,\\n    )\\n    cpp_path = output_dir / \\"hash_model_data.cpp\\"\\n    cpp_path.write_text(cpp_source, encoding=\\"utf-8\\")\\n\\n    metadata = _metadata_payload(\\n        experiment=experiment,\\n        bundle=bundle,\\n        model=model,\\n        quant_params=quant_params,\\n        source_bundle=bundle_path,\\n        output_dir=output_dir,\\n    )\\n    metadata_path = output_dir / \\"hash_model_export_metadata.json\\"\\n    metadata_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding=\\"utf-8\\")\\n    return {\\n        \\"cpp_path\\": str(cpp_path),\\n        \\"metadata_path\\": str(metadata_path),\\n        \\"output_dir\\": str(output_dir),\\n        \\"activation_quant_params\\": quant_params,\\n        \\"firmware_flash_summary\\": metadata[\\"firmware_flash_summary\\"],\\n    }\\n\\n\\ndef main() -> None:\\n    args = _parse_args()\\n    device = _resolve_device(args.device)\\n    result = export_bundle_to_firmware(\\n        bundle_path=args.bundle,\\n        output_dir=args.output_dir,\\n        project_root=args.project_root,\\n        device=device,\\n        calibration_split=args.calibration_split,\\n        calibration_batches=args.calibration_batches,\\n    )\\n    print(json.dumps(result, ensure_ascii=False, indent=2))\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "code/firmware/hash_kws_runtime/README.md": "# Hash KWS Runtime\\n\\nThis folder contains a custom low-level inference path for hash-compressed KWS models on ESP32.\\n\\nThe design goal is different from the existing TFLite Micro path:\\n\\n- preserve the memory win of analytic-hash codebooks;\\n- avoid dense weight materialization;\\n- reuse the existing `49 x 40` audio feature pipeline from the firmware side;\\n- keep the runtime simple enough to optimize incrementally on `ESP32-S3`.\\n\\n## Current Runtime Strategy\\n\\n- Input features are expected as a `49 x 40` `int8` spectrogram buffer.\\n- Internal activations use `int8` double-buffer scratch memory.\\n- Hash codebooks are stored as quantized `int8` arrays plus one scale per layer.\\n- BatchNorm is folded into per-output-channel post-affine parameters instead of into dense weights.\\n- The final layer produces logits.\\n- The current recommended firmware policy is not pure always-on smoothing:\\n  - sparse idle probes during quiet periods;\\n  - speech episodes when recent slices show activity;\\n  - peak-hold command selection inside the episode;\\n  - concise serial output by default for live board checks.\\n\\n## Current Board Status\\n\\n- Current live-board feedback says this branch works tolerably as a deployment baseline.\\n- The main remaining problem is heavy invoke latency on the deeper hash model, not microphone capture.\\n- Because of that, increasing temporal input size above `49` frames is not the right next step for the latency branch.\\n- If training-side changes are needed, prefer:\\n  - exact frontend alignment;\\n  - equal or smaller time footprint;\\n  - lower-cost temporal resolution before trying larger windows.\\n\\n## Export Path\\n\\nGenerate the firmware arrays from a trained hash bundle with:\\n\\n```powershell\\npython code/scripts/export_hash_kws_firmware.py `\\n  --bundle code/training/hash_artifacts/<experiment-tag>/hash_kws_student_student.pt\\n```\\n\\nThe exporter will:\\n\\n- restore the PyTorch hash model from the compact bundle;\\n- fold `BatchNorm` into post-convolution affine parameters;\\n- calibrate per-stage activation scales on a few dataset batches;\\n- quantize each codebook to `int8` without materializing dense weights;\\n- overwrite `hash_model_data.cpp` and emit `hash_model_export_metadata.json`.\\n\\n## Important Limitations\\n\\n- The placeholder `hash_model_data.cpp` intentionally marks the model as unavailable until a real export is generated.\\n- Current runtime math preserves the hash-compressed weights, but accuracy still depends on the on-device frontend matching the training frontend.\\n- Current runtime uses two full `int8` activation buffers. For the `64 x 20 x 25` deeper hash model this is about `64 KB` of scratch, so the next memory optimization target is fused `depthwise -> pointwise` streaming.\\n", "code/firmware/hash_kws_runtime/hash_model_types.h": "#ifndef DIPLOMA_ESP32_HASH_KWS_RUNTIME_HASH_MODEL_TYPES_H_\\n#define DIPLOMA_ESP32_HASH_KWS_RUNTIME_HASH_MODEL_TYPES_H_\\n\\n#include <cstddef>\\n#include <cstdint>\\n\\nnamespace hash_kws {\\n\\nconstexpr int kHashInputRows = 40;\\nconstexpr int kHashInputCols = 49;\\nconstexpr int kHashInputChannels = 1;\\nconstexpr int kHashMaxBlocks = 4;\\nconstexpr int kHashMaxActivationStages = 1 + (2 * kHashMaxBlocks);\\nconstexpr int kHashMaxChannels = 128;\\nconstexpr int kHashMaxClasses = 16;\\n\\nstruct HashActivationQuantParams {\\n  float input_scale;\\n  float output_scale;\\n};\\n\\nstruct HashConvLayerData {\\n  const int8_t* codebook;\\n  float codebook_scale;\\n  const float* post_scale;\\n  const float* post_bias;\\n  int codebook_size;\\n  int in_channels;\\n  int out_channels;\\n  int kernel_h;\\n  int kernel_w;\\n  int stride_h;\\n  int stride_w;\\n  int padding_h;\\n  int padding_w;\\n  int layer_id;\\n  bool signed_hash;\\n};\\n\\nstruct HashDepthwiseLayerData {\\n  const int8_t* codebook;\\n  float codebook_scale;\\n  const float* post_scale;\\n  const float* post_bias;\\n  int codebook_size;\\n  int channels;\\n  int kernel_h;\\n  int kernel_w;\\n  int stride_h;\\n  int stride_w;\\n  int padding_h;\\n  int padding_w;\\n  int layer_id;\\n  bool signed_hash;\\n};\\n\\nstruct HashLinearLayerData {\\n  const int8_t* codebook;\\n  float codebook_scale;\\n  const float* bias;\\n  int codebook_size;\\n  int in_dim;\\n  int out_dim;\\n  int layer_id;\\n  bool signed_hash;\\n};\\n\\nstruct HashDscnnModelData {\\n  bool available;\\n  int input_rows;\\n  int input_cols;\\n  int input_channels;\\n  int stem_out_channels;\\n  int num_blocks;\\n  int num_classes;\\n  HashConvLayerData stem;\\n  HashDepthwiseLayerData depthwise[kHashMaxBlocks];\\n  HashConvLayerData pointwise[kHashMaxBlocks];\\n  bool block_residual[kHashMaxBlocks];\\n  HashLinearLayerData classifier;\\n  HashActivationQuantParams activations[kHashMaxActivationStages];\\n};\\n\\n}  // namespace hash_kws\\n\\n#endif  // DIPLOMA_ESP32_HASH_KWS_RUNTIME_HASH_MODEL_TYPES_H_\\n", "code/firmware/hash_kws_runtime/hash_model_settings.h": "#ifndef DIPLOMA_ESP32_HASH_KWS_RUNTIME_HASH_MODEL_SETTINGS_H_\\n#define DIPLOMA_ESP32_HASH_KWS_RUNTIME_HASH_MODEL_SETTINGS_H_\\n\\n#include <cstdint>\\n\\nnamespace hash_kws {\\n\\nconstexpr int kCategoryCount = 12;\\nconstexpr int kUnknownIndex = 10;\\nconstexpr int kSilenceIndex = 11;\\n\\nextern const char* kCategoryLabels[kCategoryCount];\\n\\n}  // namespace hash_kws\\n\\n#endif  // DIPLOMA_ESP32_HASH_KWS_RUNTIME_HASH_MODEL_SETTINGS_H_\\n", "code/firmware/hash_kws_runtime/hash_model_settings.cpp": "#include \\"hash_model_settings.h\\"\\n\\nnamespace hash_kws {\\n\\nconst char* kCategoryLabels[kCategoryCount] = {\\n    \\"yes\\",\\n    \\"no\\",\\n    \\"up\\",\\n    \\"down\\",\\n    \\"left\\",\\n    \\"right\\",\\n    \\"on\\",\\n    \\"off\\",\\n    \\"stop\\",\\n    \\"go\\",\\n    \\"unknown\\",\\n    \\"silence\\",\\n};\\n\\n}  // namespace hash_kws\\n", "code/firmware/hash_kws_runtime/hash_model_data.h": "#ifndef DIPLOMA_ESP32_HASH_KWS_RUNTIME_HASH_MODEL_DATA_H_\\n#define DIPLOMA_ESP32_HASH_KWS_RUNTIME_HASH_MODEL_DATA_H_\\n\\n#include \\"hash_model_types.h\\"\\n\\nnamespace hash_kws {\\n\\nextern const HashDscnnModelData g_hash_model;\\n\\n}  // namespace hash_kws\\n\\n#endif  // DIPLOMA_ESP32_HASH_KWS_RUNTIME_HASH_MODEL_DATA_H_\\n", "code/firmware/hash_kws_runtime/hash_model_data.cpp": "#include \\"hash_model_data.h\\"\\n\\nnamespace hash_kws {\\n\\nnamespace {\\n\\nconstexpr int8_t kPlaceholderCodebook[] = {0};\\nconstexpr float kPlaceholderAffine[] = {1.0f};\\nconstexpr float kPlaceholderBias[] = {0.0f};\\n\\nconstexpr HashConvLayerData kPlaceholderConv = {\\n    kPlaceholderCodebook,\\n    1.0f,\\n    kPlaceholderAffine,\\n    kPlaceholderBias,\\n    1,\\n    1,\\n    1,\\n    1,\\n    1,\\n    1,\\n    1,\\n    0,\\n    0,\\n    0,\\n    false,\\n};\\n\\nconstexpr HashDepthwiseLayerData kPlaceholderDepthwise = {\\n    kPlaceholderCodebook,\\n    1.0f,\\n    kPlaceholderAffine,\\n    kPlaceholderBias,\\n    1,\\n    1,\\n    1,\\n    1,\\n    1,\\n    1,\\n    0,\\n    0,\\n    0,\\n    false,\\n};\\n\\nconstexpr HashLinearLayerData kPlaceholderLinear = {\\n    kPlaceholderCodebook,\\n    1.0f,\\n    kPlaceholderBias,\\n    1,\\n    1,\\n    1,\\n    0,\\n    false,\\n};\\n\\nconstexpr HashActivationQuantParams kPlaceholderActivation = {1.0f, 1.0f};\\n\\n}  // namespace\\n\\nconst HashDscnnModelData g_hash_model = {\\n    false,\\n    kHashInputRows,\\n    kHashInputCols,\\n    kHashInputChannels,\\n    1,\\n    kHashMaxBlocks,\\n    1,\\n    kPlaceholderConv,\\n    {\\n        kPlaceholderDepthwise,\\n        kPlaceholderDepthwise,\\n        kPlaceholderDepthwise,\\n        kPlaceholderDepthwise,\\n    },\\n    {\\n        kPlaceholderConv,\\n        kPlaceholderConv,\\n        kPlaceholderConv,\\n        kPlaceholderConv,\\n    },\\n    {\\n        false,\\n        false,\\n        false,\\n        false,\\n    },\\n    kPlaceholderLinear,\\n    {\\n        kPlaceholderActivation,\\n        kPlaceholderActivation,\\n        kPlaceholderActivation,\\n        kPlaceholderActivation,\\n        kPlaceholderActivation,\\n        kPlaceholderActivation,\\n        kPlaceholderActivation,\\n        kPlaceholderActivation,\\n        kPlaceholderActivation,\\n    },\\n};\\n\\n}  // namespace hash_kws\\n", "code/firmware/hash_kws_runtime/hash_recognize_commands.h": "#ifndef DIPLOMA_ESP32_HASH_KWS_RUNTIME_HASH_RECOGNIZE_COMMANDS_H_\\n#define DIPLOMA_ESP32_HASH_KWS_RUNTIME_HASH_RECOGNIZE_COMMANDS_H_\\n\\n#include <cstdint>\\n\\n#include \\"hash_model_settings.h\\"\\n#include \\"tensorflow/lite/c/common.h\\"\\n#include \\"tensorflow/lite/micro/micro_error_reporter.h\\"\\n\\nnamespace hash_kws {\\n\\nclass PreviousResultsQueue {\\n public:\\n  explicit PreviousResultsQueue(tflite::ErrorReporter* error_reporter)\\n      : error_reporter_(error_reporter), front_index_(0), size_(0) {}\\n\\n  struct Result {\\n    Result() : time_(0), scores() {}\\n    Result(int32_t time, const int8_t* input_scores) : time_(time) {\\n      for (int i = 0; i < kCategoryCount; ++i) {\\n        scores[i] = input_scores[i];\\n      }\\n    }\\n    int32_t time_;\\n    int8_t scores[kCategoryCount];\\n  };\\n\\n  int size() const { return size_; }\\n  int capacity() const { return kMaxResults; }\\n  bool empty() const { return size_ == 0; }\\n  Result& front() { return results_[front_index_]; }\\n  Result& back() {\\n    int back_index = front_index_ + (size_ - 1);\\n    if (back_index >= kMaxResults) {\\n      back_index -= kMaxResults;\\n    }\\n    return results_[back_index];\\n  }\\n\\n  void push_back(const Result& entry);\\n  Result pop_front();\\n  Result& from_front(int offset);\\n\\n private:\\n  tflite::ErrorReporter* error_reporter_;\\n  static constexpr int kMaxResults = 50;\\n  Result results_[kMaxResults];\\n  int front_index_;\\n  int size_;\\n};\\n\\nclass HashRecognizeCommands {\\n public:\\n  explicit HashRecognizeCommands(tflite::ErrorReporter* error_reporter,\\n                                 int32_t average_window_duration_ms = 1000,\\n                                 uint8_t detection_threshold = 180,\\n                                 int32_t suppression_ms = 1500,\\n                                 int32_t minimum_count = 3);\\n\\n  int category_count() const { return kCategoryCount; }\\n\\n  TfLiteStatus ProcessLatestResults(const int8_t* latest_results,\\n                                    int latest_results_size,\\n                                    int32_t current_time_ms,\\n                                    const char** found_command,\\n                                    uint8_t* score,\\n                                    bool* is_new_command);\\n\\n private:\\n  tflite::ErrorReporter* error_reporter_;\\n  int32_t average_window_duration_ms_;\\n  uint8_t detection_threshold_;\\n  int32_t suppression_ms_;\\n  int32_t minimum_count_;\\n\\n  PreviousResultsQueue previous_results_;\\n  const char* previous_top_label_;\\n  int32_t previous_top_label_time_;\\n};\\n\\n}  // namespace hash_kws\\n\\n#endif  // DIPLOMA_ESP32_HASH_KWS_RUNTIME_HASH_RECOGNIZE_COMMANDS_H_\\n", "code/firmware/hash_kws_runtime/hash_recognize_commands.cpp": "#include \\"hash_recognize_commands.h\\"\\n\\n#include <limits>\\n\\nnamespace hash_kws {\\n\\nvoid PreviousResultsQueue::push_back(const Result& entry) {\\n  if (size() >= kMaxResults) {\\n    TF_LITE_REPORT_ERROR(error_reporter_,\\n                         \\"Couldn\'t push_back latest result, too many already!\\");\\n    return;\\n  }\\n  size_ += 1;\\n  back() = entry;\\n}\\n\\nPreviousResultsQueue::Result PreviousResultsQueue::pop_front() {\\n  if (size() <= 0) {\\n    TF_LITE_REPORT_ERROR(error_reporter_, \\"Couldn\'t pop_front result, none present!\\");\\n    return Result();\\n  }\\n  Result result = front();\\n  front_index_ += 1;\\n  if (front_index_ >= kMaxResults) {\\n    front_index_ = 0;\\n  }\\n  size_ -= 1;\\n  return result;\\n}\\n\\nPreviousResultsQueue::Result& PreviousResultsQueue::from_front(int offset) {\\n  if ((offset < 0) || (offset >= size_)) {\\n    TF_LITE_REPORT_ERROR(error_reporter_, \\"Attempt to read beyond the end of the queue!\\");\\n    offset = size_ - 1;\\n  }\\n  int index = front_index_ + offset;\\n  if (index >= kMaxResults) {\\n    index -= kMaxResults;\\n  }\\n  return results_[index];\\n}\\n\\nHashRecognizeCommands::HashRecognizeCommands(tflite::ErrorReporter* error_reporter,\\n                                             int32_t average_window_duration_ms,\\n                                             uint8_t detection_threshold,\\n                                             int32_t suppression_ms,\\n                                             int32_t minimum_count)\\n    : error_reporter_(error_reporter),\\n      average_window_duration_ms_(average_window_duration_ms),\\n      detection_threshold_(detection_threshold),\\n      suppression_ms_(suppression_ms),\\n      minimum_count_(minimum_count),\\n      previous_results_(error_reporter),\\n      previous_top_label_(kCategoryLabels[kSilenceIndex]),\\n      previous_top_label_time_(std::numeric_limits<int32_t>::min()) {}\\n\\nTfLiteStatus HashRecognizeCommands::ProcessLatestResults(const int8_t* latest_results,\\n                                                         int latest_results_size,\\n                                                         int32_t current_time_ms,\\n                                                         const char** found_command,\\n                                                         uint8_t* score,\\n                                                         bool* is_new_command) {\\n  if (latest_results_size != kCategoryCount) {\\n    TF_LITE_REPORT_ERROR(error_reporter_,\\n                         \\"Expected %d hash-KWS scores, got %d\\",\\n                         kCategoryCount, latest_results_size);\\n    return kTfLiteError;\\n  }\\n\\n  if ((!previous_results_.empty()) && (current_time_ms < previous_results_.front().time_)) {\\n    TF_LITE_REPORT_ERROR(error_reporter_,\\n                         \\"Results must be fed in increasing time order.\\");\\n    return kTfLiteError;\\n  }\\n\\n  const int64_t time_limit = current_time_ms - average_window_duration_ms_;\\n  while ((!previous_results_.empty()) && (previous_results_.front().time_ <= time_limit)) {\\n    previous_results_.pop_front();\\n  }\\n\\n  // In fake-mic mode timestamps can hit the window boundary exactly, so prune\\n  // first and keep the queue bounded before appending the newest result.\\n  while (previous_results_.size() >= previous_results_.capacity()) {\\n    previous_results_.pop_front();\\n  }\\n  previous_results_.push_back({current_time_ms, latest_results});\\n\\n  const int64_t how_many_results = previous_results_.size();\\n  const int64_t earliest_time = previous_results_.front().time_;\\n  const int64_t samples_duration = current_time_ms - earliest_time;\\n  if ((how_many_results < minimum_count_) || (samples_duration < (average_window_duration_ms_ / 4))) {\\n    *found_command = previous_top_label_;\\n    *score = 0;\\n    *is_new_command = false;\\n    return kTfLiteOk;\\n  }\\n\\n  int32_t average_scores[kCategoryCount];\\n  for (int offset = 0; offset < previous_results_.size(); ++offset) {\\n    PreviousResultsQueue::Result previous_result = previous_results_.from_front(offset);\\n    const int8_t* scores = previous_result.scores;\\n    for (int i = 0; i < kCategoryCount; ++i) {\\n      if (offset == 0) {\\n        average_scores[i] = scores[i] + 128;\\n      } else {\\n        average_scores[i] += scores[i] + 128;\\n      }\\n    }\\n  }\\n  for (int i = 0; i < kCategoryCount; ++i) {\\n    average_scores[i] /= how_many_results;\\n  }\\n\\n  int current_top_index = 0;\\n  int32_t current_top_score = 0;\\n  for (int i = 0; i < kCategoryCount; ++i) {\\n    if (average_scores[i] > current_top_score) {\\n      current_top_score = average_scores[i];\\n      current_top_index = i;\\n    }\\n  }\\n  const char* current_top_label = kCategoryLabels[current_top_index];\\n\\n  int64_t time_since_last_top;\\n  if ((previous_top_label_ == kCategoryLabels[kSilenceIndex]) ||\\n      (previous_top_label_time_ == std::numeric_limits<int32_t>::min())) {\\n    time_since_last_top = std::numeric_limits<int32_t>::max();\\n  } else {\\n    time_since_last_top = current_time_ms - previous_top_label_time_;\\n  }\\n\\n  if ((current_top_score > detection_threshold_) &&\\n      ((current_top_label != previous_top_label_) || (time_since_last_top > suppression_ms_))) {\\n    previous_top_label_ = current_top_label;\\n    previous_top_label_time_ = current_time_ms;\\n    *is_new_command = true;\\n  } else {\\n    *is_new_command = false;\\n  }\\n\\n  *found_command = current_top_label;\\n  *score = static_cast<uint8_t>(current_top_score);\\n  return kTfLiteOk;\\n}\\n\\n}  // namespace hash_kws\\n", "code/firmware/hash_kws_runtime/hash_kws_runner.h": "#ifndef DIPLOMA_ESP32_HASH_KWS_RUNTIME_HASH_KWS_RUNNER_H_\\n#define DIPLOMA_ESP32_HASH_KWS_RUNTIME_HASH_KWS_RUNNER_H_\\n\\n#include <cstddef>\\n#include <cstdint>\\n\\n#include \\"hash_model_data.h\\"\\n\\nnamespace hash_kws {\\n\\nclass HashKwsRunner {\\n public:\\n  explicit HashKwsRunner(const HashDscnnModelData* model = &g_hash_model)\\n      : model_(model) {}\\n\\n  bool IsReady() const;\\n  int num_classes() const;\\n  size_t RequiredSingleScratchBytes() const;\\n  size_t RequiredScratchArenaBytes() const;\\n\\n  // Converts the existing firmware feature layout [time][frequency]\\n  // into the model layout [channel=1][frequency][time].\\n  void PrepareInputFromMicroFeatures(const int8_t* feature_slices,\\n                                     int8_t* model_input) const;\\n\\n  bool Invoke(const int8_t* model_input,\\n              int8_t* scratch_a,\\n              int8_t* scratch_b,\\n              int8_t* output_scores) const;\\n\\n private:\\n  const HashDscnnModelData* model_;\\n};\\n\\n}  // namespace hash_kws\\n\\n#endif  // DIPLOMA_ESP32_HASH_KWS_RUNTIME_HASH_KWS_RUNNER_H_\\n", "code/firmware/hash_kws_runtime/hash_kws_runner.cpp": "#include \\"hash_kws_runner.h\\"\\n\\n#include <algorithm>\\n#include <cmath>\\n#include <cstddef>\\n#include <cstdint>\\n\\nnamespace hash_kws {\\n\\nnamespace {\\n\\nconstexpr int kConvHashOc = 1337;\\nconstexpr int kConvHashIc = 7919;\\nconstexpr int kConvHashKh = 2971;\\nconstexpr int kConvHashKw = 6151;\\nconstexpr int kConvHashLayer = 104729;\\n\\nconstexpr int kDwHashCh = 1337;\\nconstexpr int kDwHashKh = 7919;\\nconstexpr int kDwHashKw = 2971;\\nconstexpr int kDwHashLayer = 104729;\\n\\nconstexpr int kLinearHashA = 1337;\\nconstexpr int kLinearHashB = 7919;\\nconstexpr int kLinearHashC = 2971;\\n\\nconstexpr int kSignHashA = 4099;\\nconstexpr int kSignHashB = 6151;\\nconstexpr int kSignHashC = 14887;\\n\\ninline int WrapPositiveMod(int value, int modulus) {\\n  int result = value % modulus;\\n  if (result < 0) {\\n    result += modulus;\\n  }\\n  return result;\\n}\\n\\ninline float HashSign(bool enabled, int value) {\\n  if (!enabled) {\\n    return 1.0f;\\n  }\\n  return ((value & 1) == 0) ? -1.0f : 1.0f;\\n}\\n\\ninline int OutputDim(int input, int kernel, int stride, int padding) {\\n  return ((input + (2 * padding) - kernel) / stride) + 1;\\n}\\n\\ninline int8_t QuantizeToInt8(float value, float scale) {\\n  if (scale <= 0.0f) {\\n    scale = 1.0f;\\n  }\\n  int quantized = static_cast<int>(std::lround(value / scale));\\n  if (quantized < -128) {\\n    quantized = -128;\\n  }\\n  if (quantized > 127) {\\n    quantized = 127;\\n  }\\n  return static_cast<int8_t>(quantized);\\n}\\n\\ninline float DequantizeCodebookValue(int8_t value, float scale) {\\n  if (scale <= 0.0f) {\\n    scale = 1.0f;\\n  }\\n  return static_cast<float>(value) * scale;\\n}\\n\\nfloat HashWeight(const HashConvLayerData& layer,\\n                 int output_channel,\\n                 int input_channel,\\n                 int kernel_row,\\n                 int kernel_col) {\\n  const int raw_index =\\n      (output_channel * kConvHashOc) + (input_channel * kConvHashIc) +\\n      (kernel_row * kConvHashKh) + (kernel_col * kConvHashKw) +\\n      (layer.layer_id * kConvHashLayer);\\n  const int bucket = WrapPositiveMod(raw_index, layer.codebook_size);\\n  const int sign_seed =\\n      (output_channel * kSignHashA) + (input_channel * kSignHashB) +\\n      (kernel_row * kSignHashC) + (kernel_col * (kSignHashA + kSignHashB)) +\\n      (layer.layer_id * (kSignHashC + 11));\\n  return DequantizeCodebookValue(layer.codebook[bucket], layer.codebook_scale) *\\n         HashSign(layer.signed_hash, sign_seed);\\n}\\n\\nfloat HashDepthwiseWeight(const HashDepthwiseLayerData& layer,\\n                          int channel,\\n                          int kernel_row,\\n                          int kernel_col) {\\n  const int raw_index =\\n      (channel * kDwHashCh) + (kernel_row * kDwHashKh) +\\n      (kernel_col * kDwHashKw) + (layer.layer_id * kDwHashLayer);\\n  const int bucket = WrapPositiveMod(raw_index, layer.codebook_size);\\n  const int sign_seed =\\n      (channel * kSignHashA) + (kernel_row * kSignHashB) +\\n      (kernel_col * kSignHashC) + (layer.layer_id * (kSignHashA + 29));\\n  return DequantizeCodebookValue(layer.codebook[bucket], layer.codebook_scale) *\\n         HashSign(layer.signed_hash, sign_seed);\\n}\\n\\nfloat HashLinearWeight(const HashLinearLayerData& layer, int output_index, int input_index) {\\n  const int raw_index =\\n      (output_index * kLinearHashA) + (input_index * kLinearHashB) +\\n      (layer.layer_id * kLinearHashC);\\n  const int bucket = WrapPositiveMod(raw_index, layer.codebook_size);\\n  const int sign_seed =\\n      (output_index * kSignHashA) + (input_index * kSignHashB) +\\n      (layer.layer_id * kSignHashC);\\n  return DequantizeCodebookValue(layer.codebook[bucket], layer.codebook_scale) *\\n         HashSign(layer.signed_hash, sign_seed);\\n}\\n\\nvoid FillStemKernelWeights(const HashConvLayerData& layer,\\n                           int output_channel,\\n                           float* weights_3x3) {\\n  int index = 0;\\n  for (int kernel_row = 0; kernel_row < 3; ++kernel_row) {\\n    for (int kernel_col = 0; kernel_col < 3; ++kernel_col) {\\n      weights_3x3[index++] =\\n          HashWeight(layer, output_channel, 0, kernel_row, kernel_col);\\n    }\\n  }\\n}\\n\\nvoid FillDepthwiseKernelWeights(const HashDepthwiseLayerData& layer,\\n                                int channel,\\n                                float* weights_3x3) {\\n  int index = 0;\\n  for (int kernel_row = 0; kernel_row < 3; ++kernel_row) {\\n    for (int kernel_col = 0; kernel_col < 3; ++kernel_col) {\\n      weights_3x3[index++] =\\n          HashDepthwiseWeight(layer, channel, kernel_row, kernel_col);\\n    }\\n  }\\n}\\n\\nvoid FillPointwiseWeights(const HashConvLayerData& layer,\\n                          int output_channel,\\n                          float* weights_1x1) {\\n  for (int input_channel = 0; input_channel < layer.in_channels; ++input_channel) {\\n    weights_1x1[input_channel] = HashWeight(layer, output_channel, input_channel, 0, 0);\\n  }\\n}\\n\\nvoid FillLinearWeights(const HashLinearLayerData& layer,\\n                       int output_index,\\n                       float* weights) {\\n  for (int input_index = 0; input_index < layer.in_dim; ++input_index) {\\n    weights[input_index] = HashLinearWeight(layer, output_index, input_index);\\n  }\\n}\\n\\nvoid RunStemConv3x3(const HashConvLayerData& layer,\\n                    const HashActivationQuantParams& quant,\\n                    const int8_t* input,\\n                    int input_rows,\\n                    int input_cols,\\n                    int8_t* output) {\\n  const int output_rows =\\n      OutputDim(input_rows, layer.kernel_h, layer.stride_h, layer.padding_h);\\n  const int output_cols =\\n      OutputDim(input_cols, layer.kernel_w, layer.stride_w, layer.padding_w);\\n\\n  for (int output_channel = 0; output_channel < layer.out_channels; ++output_channel) {\\n    float weights_3x3[9];\\n    FillStemKernelWeights(layer, output_channel, weights_3x3);\\n    const float post_scale = layer.post_scale[output_channel];\\n    const float post_bias = layer.post_bias[output_channel];\\n    for (int output_row = 0; output_row < output_rows; ++output_row) {\\n      for (int output_col = 0; output_col < output_cols; ++output_col) {\\n        float accum = 0.0f;\\n        int weight_index = 0;\\n        for (int kernel_row = 0; kernel_row < 3; ++kernel_row) {\\n          const int input_row =\\n              (output_row * layer.stride_h) + kernel_row - layer.padding_h;\\n          for (int kernel_col = 0; kernel_col < 3; ++kernel_col) {\\n            const int input_col =\\n                (output_col * layer.stride_w) + kernel_col - layer.padding_w;\\n            if ((input_row >= 0) && (input_row < input_rows) &&\\n                (input_col >= 0) && (input_col < input_cols)) {\\n              const int input_index = (input_row * input_cols) + input_col;\\n              const float input_value =\\n                  static_cast<float>(input[input_index]) * quant.input_scale;\\n              accum += input_value * weights_3x3[weight_index];\\n            }\\n            ++weight_index;\\n          }\\n        }\\n        float activated = (post_scale * accum) + post_bias;\\n        if (activated < 0.0f) {\\n          activated = 0.0f;\\n        }\\n        const int output_index =\\n            ((output_channel * output_rows) + output_row) * output_cols + output_col;\\n        output[output_index] = QuantizeToInt8(activated, quant.output_scale);\\n      }\\n    }\\n  }\\n}\\n\\nvoid RunPointwiseConv1x1(const HashConvLayerData& layer,\\n                         const HashActivationQuantParams& quant,\\n                         const int8_t* input,\\n                         int input_rows,\\n                         int input_cols,\\n                         int8_t* output) {\\n  float weights_1x1[kHashMaxChannels];\\n  for (int output_channel = 0; output_channel < layer.out_channels; ++output_channel) {\\n    FillPointwiseWeights(layer, output_channel, weights_1x1);\\n    const float post_scale = layer.post_scale[output_channel];\\n    const float post_bias = layer.post_bias[output_channel];\\n    for (int output_row = 0; output_row < input_rows; ++output_row) {\\n      for (int output_col = 0; output_col < input_cols; ++output_col) {\\n        float accum = 0.0f;\\n        for (int input_channel = 0; input_channel < layer.in_channels; ++input_channel) {\\n          const int input_index =\\n              ((input_channel * input_rows) + output_row) * input_cols + output_col;\\n          const float input_value =\\n              static_cast<float>(input[input_index]) * quant.input_scale;\\n          accum += input_value * weights_1x1[input_channel];\\n        }\\n        float activated = (post_scale * accum) + post_bias;\\n        if (activated < 0.0f) {\\n          activated = 0.0f;\\n        }\\n        const int output_index =\\n            ((output_channel * input_rows) + output_row) * input_cols + output_col;\\n        output[output_index] = QuantizeToInt8(activated, quant.output_scale);\\n      }\\n    }\\n  }\\n}\\n\\nvoid RunPointwiseResidualConv1x1(const HashConvLayerData& layer,\\n                                 const HashActivationQuantParams& quant,\\n                                 const int8_t* input,\\n                                 int input_rows,\\n                                 int input_cols,\\n                                 const int8_t* residual_input,\\n                                 float residual_input_scale,\\n                                 int8_t* output) {\\n  float weights_1x1[kHashMaxChannels];\\n  for (int output_channel = 0; output_channel < layer.out_channels; ++output_channel) {\\n    FillPointwiseWeights(layer, output_channel, weights_1x1);\\n    const float post_scale = layer.post_scale[output_channel];\\n    const float post_bias = layer.post_bias[output_channel];\\n    for (int output_row = 0; output_row < input_rows; ++output_row) {\\n      for (int output_col = 0; output_col < input_cols; ++output_col) {\\n        float accum = 0.0f;\\n        for (int input_channel = 0; input_channel < layer.in_channels; ++input_channel) {\\n          const int input_index =\\n              ((input_channel * input_rows) + output_row) * input_cols + output_col;\\n          const float input_value =\\n              static_cast<float>(input[input_index]) * quant.input_scale;\\n          accum += input_value * weights_1x1[input_channel];\\n        }\\n        float activated = (post_scale * accum) + post_bias;\\n        const int output_index =\\n            ((output_channel * input_rows) + output_row) * input_cols + output_col;\\n        const float residual_value =\\n            static_cast<float>(residual_input[output_index]) * residual_input_scale;\\n        activated += residual_value;\\n        if (activated < 0.0f) {\\n          activated = 0.0f;\\n        }\\n        output[output_index] = QuantizeToInt8(activated, quant.output_scale);\\n      }\\n    }\\n  }\\n}\\n\\nvoid RunHashConv2D(const HashConvLayerData& layer,\\n                   const HashActivationQuantParams& quant,\\n                   const int8_t* input,\\n                   int input_rows,\\n                   int input_cols,\\n                   int8_t* output) {\\n  if ((layer.kernel_h == 3) && (layer.kernel_w == 3) && (layer.in_channels == 1) &&\\n      (layer.out_channels <= kHashMaxChannels)) {\\n    RunStemConv3x3(layer, quant, input, input_rows, input_cols, output);\\n    return;\\n  }\\n  if ((layer.kernel_h == 1) && (layer.kernel_w == 1) && (layer.in_channels <= kHashMaxChannels) &&\\n      (layer.out_channels <= kHashMaxChannels) && (layer.stride_h == 1) &&\\n      (layer.stride_w == 1) && (layer.padding_h == 0) && (layer.padding_w == 0)) {\\n    RunPointwiseConv1x1(layer, quant, input, input_rows, input_cols, output);\\n    return;\\n  }\\n\\n  const int output_rows =\\n      OutputDim(input_rows, layer.kernel_h, layer.stride_h, layer.padding_h);\\n  const int output_cols =\\n      OutputDim(input_cols, layer.kernel_w, layer.stride_w, layer.padding_w);\\n\\n  for (int output_channel = 0; output_channel < layer.out_channels; ++output_channel) {\\n    const float post_scale = layer.post_scale[output_channel];\\n    const float post_bias = layer.post_bias[output_channel];\\n    for (int output_row = 0; output_row < output_rows; ++output_row) {\\n      for (int output_col = 0; output_col < output_cols; ++output_col) {\\n        float accum = 0.0f;\\n        for (int input_channel = 0; input_channel < layer.in_channels; ++input_channel) {\\n          for (int kernel_row = 0; kernel_row < layer.kernel_h; ++kernel_row) {\\n            const int input_row =\\n                (output_row * layer.stride_h) + kernel_row - layer.padding_h;\\n            if ((input_row < 0) || (input_row >= input_rows)) {\\n              continue;\\n            }\\n            for (int kernel_col = 0; kernel_col < layer.kernel_w; ++kernel_col) {\\n              const int input_col =\\n                  (output_col * layer.stride_w) + kernel_col - layer.padding_w;\\n              if ((input_col < 0) || (input_col >= input_cols)) {\\n                continue;\\n              }\\n              const int input_index =\\n                  ((input_channel * input_rows) + input_row) * input_cols + input_col;\\n              const float input_value =\\n                  static_cast<float>(input[input_index]) * quant.input_scale;\\n              accum +=\\n                  input_value *\\n                  HashWeight(layer, output_channel, input_channel, kernel_row, kernel_col);\\n            }\\n          }\\n        }\\n        float activated = (post_scale * accum) + post_bias;\\n        if (activated < 0.0f) {\\n          activated = 0.0f;\\n        }\\n        const int output_index =\\n            ((output_channel * output_rows) + output_row) * output_cols + output_col;\\n        output[output_index] = QuantizeToInt8(activated, quant.output_scale);\\n      }\\n    }\\n  }\\n}\\n\\nvoid RunHashDepthwiseConv2D(const HashDepthwiseLayerData& layer,\\n                            const HashActivationQuantParams& quant,\\n                            const int8_t* input,\\n                            int input_rows,\\n                            int input_cols,\\n                            int8_t* output) {\\n  if ((layer.kernel_h == 3) && (layer.kernel_w == 3) && (layer.channels <= kHashMaxChannels) &&\\n      (layer.stride_h == 1) && (layer.stride_w == 1) && (layer.padding_h == 1) &&\\n      (layer.padding_w == 1)) {\\n    for (int channel = 0; channel < layer.channels; ++channel) {\\n      float weights_3x3[9];\\n      FillDepthwiseKernelWeights(layer, channel, weights_3x3);\\n      const float post_scale = layer.post_scale[channel];\\n      const float post_bias = layer.post_bias[channel];\\n      for (int output_row = 0; output_row < input_rows; ++output_row) {\\n        for (int output_col = 0; output_col < input_cols; ++output_col) {\\n          float accum = 0.0f;\\n          int weight_index = 0;\\n          for (int kernel_row = 0; kernel_row < 3; ++kernel_row) {\\n            const int input_row = output_row + kernel_row - 1;\\n            for (int kernel_col = 0; kernel_col < 3; ++kernel_col) {\\n              const int input_col = output_col + kernel_col - 1;\\n              if ((input_row >= 0) && (input_row < input_rows) &&\\n                  (input_col >= 0) && (input_col < input_cols)) {\\n                const int input_index =\\n                    ((channel * input_rows) + input_row) * input_cols + input_col;\\n                const float input_value =\\n                    static_cast<float>(input[input_index]) * quant.input_scale;\\n                accum += input_value * weights_3x3[weight_index];\\n              }\\n              ++weight_index;\\n            }\\n          }\\n          float activated = (post_scale * accum) + post_bias;\\n          if (activated < 0.0f) {\\n            activated = 0.0f;\\n          }\\n          const int output_index =\\n              ((channel * input_rows) + output_row) * input_cols + output_col;\\n          output[output_index] = QuantizeToInt8(activated, quant.output_scale);\\n        }\\n      }\\n    }\\n    return;\\n  }\\n\\n  const int output_rows =\\n      OutputDim(input_rows, layer.kernel_h, layer.stride_h, layer.padding_h);\\n  const int output_cols =\\n      OutputDim(input_cols, layer.kernel_w, layer.stride_w, layer.padding_w);\\n\\n  for (int channel = 0; channel < layer.channels; ++channel) {\\n    const float post_scale = layer.post_scale[channel];\\n    const float post_bias = layer.post_bias[channel];\\n    for (int output_row = 0; output_row < output_rows; ++output_row) {\\n      for (int output_col = 0; output_col < output_cols; ++output_col) {\\n        float accum = 0.0f;\\n        for (int kernel_row = 0; kernel_row < layer.kernel_h; ++kernel_row) {\\n          const int input_row =\\n              (output_row * layer.stride_h) + kernel_row - layer.padding_h;\\n          if ((input_row < 0) || (input_row >= input_rows)) {\\n            continue;\\n          }\\n          for (int kernel_col = 0; kernel_col < layer.kernel_w; ++kernel_col) {\\n            const int input_col =\\n                (output_col * layer.stride_w) + kernel_col - layer.padding_w;\\n            if ((input_col < 0) || (input_col >= input_cols)) {\\n              continue;\\n            }\\n            const int input_index =\\n                ((channel * input_rows) + input_row) * input_cols + input_col;\\n            const float input_value =\\n                static_cast<float>(input[input_index]) * quant.input_scale;\\n            accum += input_value *\\n                     HashDepthwiseWeight(layer, channel, kernel_row, kernel_col);\\n          }\\n        }\\n        float activated = (post_scale * accum) + post_bias;\\n        if (activated < 0.0f) {\\n          activated = 0.0f;\\n        }\\n        const int output_index =\\n            ((channel * output_rows) + output_row) * output_cols + output_col;\\n        output[output_index] = QuantizeToInt8(activated, quant.output_scale);\\n      }\\n    }\\n  }\\n}\\n\\nvoid RunHashLinear(const HashLinearLayerData& layer,\\n                   const int8_t* input,\\n                   float input_scale,\\n                   float* logits) {\\n  float weights[kHashMaxChannels];\\n  for (int output_index = 0; output_index < layer.out_dim; ++output_index) {\\n    FillLinearWeights(layer, output_index, weights);\\n    float accum = layer.bias[output_index];\\n    for (int input_index = 0; input_index < layer.in_dim; ++input_index) {\\n      accum += (static_cast<float>(input[input_index]) * input_scale) *\\n               weights[input_index];\\n    }\\n    logits[output_index] = accum;\\n  }\\n}\\n\\nvoid AveragePoolChannels(const int8_t* input,\\n                         int channels,\\n                         int rows,\\n                         int cols,\\n                         float input_scale,\\n                         int8_t* pooled_output,\\n                         float pooled_output_scale) {\\n  const int spatial_size = rows * cols;\\n  for (int channel = 0; channel < channels; ++channel) {\\n    int32_t sum = 0;\\n    const int base_index = channel * spatial_size;\\n    for (int i = 0; i < spatial_size; ++i) {\\n      sum += static_cast<int32_t>(input[base_index + i]);\\n    }\\n    const float mean_value = (static_cast<float>(sum) / static_cast<float>(spatial_size)) * input_scale;\\n    pooled_output[channel] = QuantizeToInt8(mean_value, pooled_output_scale);\\n  }\\n}\\n\\nvoid SoftmaxToCenteredInt8(const float* logits, int count, int8_t* output_scores) {\\n  float max_logit = logits[0];\\n  for (int i = 1; i < count; ++i) {\\n    max_logit = std::max(max_logit, logits[i]);\\n  }\\n\\n  float sum = 0.0f;\\n  float probabilities[kHashMaxClasses];\\n  for (int i = 0; i < count; ++i) {\\n    probabilities[i] = std::exp(logits[i] - max_logit);\\n    sum += probabilities[i];\\n  }\\n\\n  for (int i = 0; i < count; ++i) {\\n    const float normalized = probabilities[i] / sum;\\n    int quantized = static_cast<int>(std::lround(normalized * 255.0f)) - 128;\\n    if (quantized < -128) {\\n      quantized = -128;\\n    }\\n    if (quantized > 127) {\\n      quantized = 127;\\n    }\\n    output_scores[i] = static_cast<int8_t>(quantized);\\n  }\\n}\\n\\nbool ValidateModel(const HashDscnnModelData* model) {\\n  if (model == nullptr) {\\n    return false;\\n  }\\n  if (!model->available) {\\n    return false;\\n  }\\n  if (model->input_channels != kHashInputChannels) {\\n    return false;\\n  }\\n  if ((model->num_blocks <= 0) || (model->num_blocks > kHashMaxBlocks)) {\\n    return false;\\n  }\\n  if ((model->stem.out_channels <= 0) || (model->stem.out_channels > kHashMaxChannels)) {\\n    return false;\\n  }\\n  if ((model->classifier.in_dim <= 0) || (model->classifier.in_dim > kHashMaxChannels)) {\\n    return false;\\n  }\\n  if ((model->num_classes <= 0) || (model->num_classes > kHashMaxClasses)) {\\n    return false;\\n  }\\n  for (int block = 0; block < model->num_blocks; ++block) {\\n    if ((model->depthwise[block].channels <= 0) ||\\n        (model->depthwise[block].channels > kHashMaxChannels)) {\\n      return false;\\n    }\\n    if ((model->pointwise[block].in_channels <= 0) ||\\n        (model->pointwise[block].in_channels > kHashMaxChannels) ||\\n        (model->pointwise[block].out_channels <= 0) ||\\n        (model->pointwise[block].out_channels > kHashMaxChannels)) {\\n      return false;\\n    }\\n    if (model->block_residual[block]) {\\n      if ((model->depthwise[block].stride_h != 1) || (model->depthwise[block].stride_w != 1) ||\\n          (model->pointwise[block].kernel_h != 1) || (model->pointwise[block].kernel_w != 1) ||\\n          (model->pointwise[block].stride_h != 1) || (model->pointwise[block].stride_w != 1) ||\\n          (model->pointwise[block].padding_h != 0) || (model->pointwise[block].padding_w != 0) ||\\n          (model->pointwise[block].in_channels != model->pointwise[block].out_channels) ||\\n          (model->depthwise[block].channels != model->pointwise[block].out_channels)) {\\n        return false;\\n      }\\n    }\\n  }\\n  return true;\\n}\\n\\nsize_t MaxActivationElements(const HashDscnnModelData& model) {\\n  int rows = model.input_rows;\\n  int cols = model.input_cols;\\n  size_t max_elements = 0;\\n\\n  rows = OutputDim(rows, model.stem.kernel_h, model.stem.stride_h, model.stem.padding_h);\\n  cols = OutputDim(cols, model.stem.kernel_w, model.stem.stride_w, model.stem.padding_w);\\n  max_elements = std::max(max_elements, static_cast<size_t>(model.stem.out_channels * rows * cols));\\n\\n  for (int block = 0; block < model.num_blocks; ++block) {\\n    rows = OutputDim(rows, model.depthwise[block].kernel_h, model.depthwise[block].stride_h, model.depthwise[block].padding_h);\\n    cols = OutputDim(cols, model.depthwise[block].kernel_w, model.depthwise[block].stride_w, model.depthwise[block].padding_w);\\n    max_elements = std::max(max_elements, static_cast<size_t>(model.depthwise[block].channels * rows * cols));\\n\\n    rows = OutputDim(rows, model.pointwise[block].kernel_h, model.pointwise[block].stride_h, model.pointwise[block].padding_h);\\n    cols = OutputDim(cols, model.pointwise[block].kernel_w, model.pointwise[block].stride_w, model.pointwise[block].padding_w);\\n    max_elements = std::max(max_elements, static_cast<size_t>(model.pointwise[block].out_channels * rows * cols));\\n  }\\n\\n  max_elements = std::max(max_elements, static_cast<size_t>(model.classifier.in_dim));\\n  return max_elements;\\n}\\n\\n}  // namespace\\n\\nbool HashKwsRunner::IsReady() const { return ValidateModel(model_); }\\n\\nint HashKwsRunner::num_classes() const {\\n  return (model_ != nullptr) ? model_->num_classes : 0;\\n}\\n\\nsize_t HashKwsRunner::RequiredSingleScratchBytes() const {\\n  if (model_ == nullptr) {\\n    return 0;\\n  }\\n  return MaxActivationElements(*model_) * sizeof(int8_t);\\n}\\n\\nsize_t HashKwsRunner::RequiredScratchArenaBytes() const {\\n  return 2 * RequiredSingleScratchBytes();\\n}\\n\\nvoid HashKwsRunner::PrepareInputFromMicroFeatures(const int8_t* feature_slices,\\n                                                  int8_t* model_input) const {\\n  if ((feature_slices == nullptr) || (model_input == nullptr)) {\\n    return;\\n  }\\n  for (int time_index = 0; time_index < kHashInputCols; ++time_index) {\\n    for (int freq_index = 0; freq_index < kHashInputRows; ++freq_index) {\\n      const int source_index = (time_index * kHashInputRows) + freq_index;\\n      const int dest_index = (freq_index * kHashInputCols) + time_index;\\n      model_input[dest_index] = feature_slices[source_index];\\n    }\\n  }\\n}\\n\\nbool HashKwsRunner::Invoke(const int8_t* model_input,\\n                           int8_t* scratch_a,\\n                           int8_t* scratch_b,\\n                           int8_t* output_scores) const {\\n  if (!IsReady() || (model_input == nullptr) || (scratch_a == nullptr) ||\\n      (scratch_b == nullptr) || (output_scores == nullptr)) {\\n    return false;\\n  }\\n\\n  int rows = model_->input_rows;\\n  int cols = model_->input_cols;\\n\\n  RunHashConv2D(model_->stem, model_->activations[0], model_input, rows, cols, scratch_a);\\n  rows = OutputDim(rows, model_->stem.kernel_h, model_->stem.stride_h, model_->stem.padding_h);\\n  cols = OutputDim(cols, model_->stem.kernel_w, model_->stem.stride_w, model_->stem.padding_w);\\n\\n  for (int block = 0; block < model_->num_blocks; ++block) {\\n    const int depthwise_stage = 1 + (2 * block);\\n    const int pointwise_stage = depthwise_stage + 1;\\n    RunHashDepthwiseConv2D(model_->depthwise[block],\\n                           model_->activations[depthwise_stage],\\n                           scratch_a,\\n                           rows,\\n                           cols,\\n                           scratch_b);\\n    rows = OutputDim(rows, model_->depthwise[block].kernel_h,\\n                     model_->depthwise[block].stride_h,\\n                     model_->depthwise[block].padding_h);\\n    cols = OutputDim(cols, model_->depthwise[block].kernel_w,\\n                     model_->depthwise[block].stride_w,\\n                     model_->depthwise[block].padding_w);\\n\\n    if (model_->block_residual[block]) {\\n      RunPointwiseResidualConv1x1(model_->pointwise[block],\\n                                  model_->activations[pointwise_stage],\\n                                  scratch_b,\\n                                  rows,\\n                                  cols,\\n                                  scratch_a,\\n                                  model_->activations[depthwise_stage].input_scale,\\n                                  scratch_a);\\n    } else {\\n      RunHashConv2D(model_->pointwise[block],\\n                    model_->activations[pointwise_stage],\\n                    scratch_b,\\n                    rows,\\n                    cols,\\n                    scratch_a);\\n    }\\n    rows = OutputDim(rows, model_->pointwise[block].kernel_h,\\n                     model_->pointwise[block].stride_h,\\n                     model_->pointwise[block].padding_h);\\n    cols = OutputDim(cols, model_->pointwise[block].kernel_w,\\n                     model_->pointwise[block].stride_w,\\n                     model_->pointwise[block].padding_w);\\n  }\\n\\n  const float pooled_scale = model_->activations[2 * model_->num_blocks].output_scale;\\n  AveragePoolChannels(scratch_a,\\n                      model_->classifier.in_dim,\\n                      rows,\\n                      cols,\\n                      pooled_scale,\\n                      scratch_b,\\n                      pooled_scale);\\n\\n  float logits[kHashMaxClasses];\\n  RunHashLinear(model_->classifier, scratch_b, pooled_scale, logits);\\n  SoftmaxToCenteredInt8(logits, model_->num_classes, output_scores);\\n  return true;\\n}\\n\\n}  // namespace hash_kws\\n", "code/firmware/hash_kws_runtime/hash_micro_speech.cpp": "#include <TensorFlowLite_ESP32.h>\\n\\n#include <cstdint>\\n#include <cstdlib>\\n\\n#include \\"../micro_speech_sim/audio_provider.h\\"\\n#include \\"../micro_speech_sim/command_responder.h\\"\\n#include \\"../micro_speech_sim/feature_provider.h\\"\\n#include \\"../micro_speech_sim/micro_model_settings.h\\"\\n#include \\"hash_kws_runner.h\\"\\n#include \\"hash_model_data.h\\"\\n#include \\"hash_recognize_commands.h\\"\\n#include \\"tensorflow/lite/micro/micro_error_reporter.h\\"\\n#include \\"tensorflow/lite/micro/system_setup.h\\"\\n\\nnamespace {\\n\\ntflite::ErrorReporter* error_reporter = nullptr;\\nFeatureProvider* feature_provider = nullptr;\\nhash_kws::HashKwsRunner* runner = nullptr;\\nhash_kws::HashRecognizeCommands* recognizer = nullptr;\\n\\nint32_t previous_time = 0;\\nint8_t feature_buffer[kFeatureElementCount];\\nint8_t* model_input_buffer = nullptr;\\nint8_t* scratch_a = nullptr;\\nint8_t* scratch_b = nullptr;\\nint8_t output_scores[hash_kws::kCategoryCount];\\n\\n}  // namespace\\n\\nvoid setup() {\\n  static tflite::MicroErrorReporter micro_error_reporter;\\n  error_reporter = &micro_error_reporter;\\n\\n  if (!hash_kws::g_hash_model.available) {\\n    TF_LITE_REPORT_ERROR(error_reporter,\\n                         \\"hash_kws model is not available. Export hash_model_data.cpp first.\\");\\n    return;\\n  }\\n\\n  static FeatureProvider static_feature_provider(kFeatureElementCount, feature_buffer);\\n  feature_provider = &static_feature_provider;\\n\\n  static hash_kws::HashKwsRunner static_runner(&hash_kws::g_hash_model);\\n  runner = &static_runner;\\n\\n  if (!runner->IsReady()) {\\n    TF_LITE_REPORT_ERROR(error_reporter, \\"hash_kws runtime model validation failed.\\");\\n    return;\\n  }\\n\\n  static hash_kws::HashRecognizeCommands static_recognizer(error_reporter);\\n  recognizer = &static_recognizer;\\n\\n  const size_t input_bytes =\\n      static_cast<size_t>(hash_kws::g_hash_model.input_rows) *\\n      static_cast<size_t>(hash_kws::g_hash_model.input_cols) *\\n      static_cast<size_t>(hash_kws::g_hash_model.input_channels);\\n  const size_t scratch_bytes = runner->RequiredSingleScratchBytes();\\n\\n  model_input_buffer = static_cast<int8_t*>(std::malloc(input_bytes));\\n  scratch_a = static_cast<int8_t*>(std::malloc(scratch_bytes));\\n  scratch_b = static_cast<int8_t*>(std::malloc(scratch_bytes));\\n  if ((model_input_buffer == nullptr) || (scratch_a == nullptr) || (scratch_b == nullptr)) {\\n    TF_LITE_REPORT_ERROR(error_reporter,\\n                         \\"hash_kws allocation failed: input=%d scratch=%d\\",\\n                         static_cast<int>(input_bytes),\\n                         static_cast<int>(scratch_bytes));\\n    return;\\n  }\\n\\n  TF_LITE_REPORT_ERROR(\\n      error_reporter,\\n      \\"hash_kws ready: classes=%d input=%dx%d scratch_total=%d bytes\\",\\n      runner->num_classes(), hash_kws::g_hash_model.input_rows,\\n      hash_kws::g_hash_model.input_cols,\\n      static_cast<int>(runner->RequiredScratchArenaBytes()));\\n\\n  previous_time = 0;\\n}\\n\\nvoid loop() {\\n  if ((feature_provider == nullptr) || (runner == nullptr) || (recognizer == nullptr) ||\\n      (model_input_buffer == nullptr) || (scratch_a == nullptr) || (scratch_b == nullptr)) {\\n    return;\\n  }\\n\\n  const int32_t current_time = LatestAudioTimestamp();\\n  int how_many_new_slices = 0;\\n  TfLiteStatus feature_status = feature_provider->PopulateFeatureData(\\n      error_reporter, previous_time, current_time, &how_many_new_slices);\\n  if (feature_status != kTfLiteOk) {\\n    TF_LITE_REPORT_ERROR(error_reporter, \\"Feature generation failed\\");\\n    return;\\n  }\\n  previous_time = current_time;\\n  if (how_many_new_slices == 0) {\\n    return;\\n  }\\n\\n  runner->PrepareInputFromMicroFeatures(feature_buffer, model_input_buffer);\\n  if (!runner->Invoke(model_input_buffer, scratch_a, scratch_b, output_scores)) {\\n    TF_LITE_REPORT_ERROR(error_reporter, \\"hash_kws Invoke failed\\");\\n    return;\\n  }\\n\\n  const char* found_command = nullptr;\\n  uint8_t score = 0;\\n  bool is_new_command = false;\\n  TfLiteStatus process_status = recognizer->ProcessLatestResults(\\n      output_scores, hash_kws::kCategoryCount, current_time, &found_command,\\n      &score, &is_new_command);\\n  if (process_status != kTfLiteOk) {\\n    TF_LITE_REPORT_ERROR(error_reporter,\\n                         \\"HashRecognizeCommands::ProcessLatestResults() failed\\");\\n    return;\\n  }\\n\\n  RespondToCommand(error_reporter, current_time, found_command, score, is_new_command);\\n}\\n"}')

def ensure_runtime_files(root: Path, payloads: dict[str, str], overwrite: bool = False):
    created = []
    skipped = []
    for relative_path, content in payloads.items():
        target_path = root / relative_path
        target_path.parent.mkdir(parents=True, exist_ok=True)
        if target_path.exists() and not overwrite:
            skipped.append(relative_path)
            continue
        target_path.write_text(content, encoding="utf-8")
        created.append(relative_path)
    return created, skipped

created_files, skipped_files = ensure_runtime_files(
    PROJECT_ROOT,
    FILE_PAYLOADS,
    overwrite=FORCE_SYNC_RUNTIME_FILES,
)

for path in (TRAINING_ROOT, SCRIPTS_ROOT, HASHEDNET95_ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

importlib.invalidate_caches()
for module_name in list(sys.modules):
    if (
        module_name == "hash_kws_lab"
        or module_name.startswith("hash_kws_lab.")
        or module_name == "hashednet95"
        or module_name.startswith("hashednet95.")
    ):
        del sys.modules[module_name]

local_speechcommands_root = PROJECT_ROOT / "data"
drive_speechcommands_root = DRIVE_CACHE_ROOT / "speechcommands_v2"
speechcommands_root = (
    drive_speechcommands_root
    if DRIVE_CACHE_ACTIVE and CACHE_SPEECHCOMMANDS_ON_DRIVE
    else local_speechcommands_root
)
feature_cache_root = DRIVE_CACHE_ROOT / "hash_feature_cache"
teacher_logits_cache_root = DRIVE_CACHE_ROOT / "teacher_logits_cache"

speechcommands_root.mkdir(parents=True, exist_ok=True)
os.environ["SPEECHCOMMANDS_DATA_ROOT"] = str(speechcommands_root)

if DRIVE_CACHE_ACTIVE:
    DRIVE_CACHE_ROOT.mkdir(parents=True, exist_ok=True)
    feature_cache_root.mkdir(parents=True, exist_ok=True)
    teacher_logits_cache_root.mkdir(parents=True, exist_ok=True)
    os.environ["HASH_KWS_FEATURE_CACHE_ROOT"] = str(feature_cache_root)
    os.environ["HASH_KWS_TEACHER_LOGITS_CACHE_ROOT"] = str(teacher_logits_cache_root)
else:
    feature_cache_root = PROJECT_ROOT / "data" / "hash_feature_cache"
    teacher_logits_cache_root = PROJECT_ROOT / "data" / "teacher_logits_cache"
    feature_cache_root.mkdir(parents=True, exist_ok=True)
    teacher_logits_cache_root.mkdir(parents=True, exist_ok=True)
    os.environ["HASH_KWS_FEATURE_CACHE_ROOT"] = str(feature_cache_root)
    os.environ["HASH_KWS_TEACHER_LOGITS_CACHE_ROOT"] = str(teacher_logits_cache_root)

os.chdir(PROJECT_ROOT)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("SPEECHCOMMANDS_DATA_ROOT:", speechcommands_root)
print("HASH_KWS_FEATURE_CACHE_ROOT:", feature_cache_root)
print("HASH_KWS_TEACHER_LOGITS_CACHE_ROOT:", teacher_logits_cache_root)
print("Runtime files written:", len(created_files))
print("Runtime files skipped:", len(skipped_files))
print("DRIVE_CACHE_ROOT:", DRIVE_CACHE_ROOT if DRIVE_CACHE_ACTIVE else "disabled")


In [ ]:
import json
import shutil
import time
from dataclasses import replace
from pathlib import Path

import torch
from torchaudio.datasets import SPEECHCOMMANDS

from hash_kws_lab.config import make_experiment
from hash_kws_lab.data import ensure_torchaudio_available, prepare_dataloaders
from hash_kws_lab.export import export_model_bundle
from hash_kws_lab.models import build_student_model, build_teacher_model, summarize_model
from hash_kws_lab.reporting import (
    add_note,
    initialize_run_state,
    record_export_artifacts,
    save_history,
    save_history_plots,
    save_json_artifact,
    save_metrics,
    save_model_summary,
    save_text_artifact,
    update_stage_state,
    write_run_summary,
)
from hash_kws_lab.trainer import evaluate, load_model_checkpoint, train_student, train_teacher
from hashednet95_recipes import (
    build_hashednet95_recipe_book,
    describe_hashednet95_recipe,
    with_drive_cache_paths,
)
import export_hash_kws_firmware as firmware_exporter

ensure_torchaudio_available()
torch.manual_seed(13)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(13)
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


In [ ]:
SPEECHCOMMANDS_REQUIRED_FILES = (
    "validation_list.txt",
    "testing_list.txt",
)
SPEECHCOMMANDS_ARCHIVE = "speech_commands_v0.02.tar.gz"
SPEECHCOMMANDS_REQUIRED_DIRS = (
    "_background_noise_",
    "yes",
    "no",
    "up",
    "down",
    "left",
    "right",
    "on",
    "off",
    "stop",
    "go",
)

def speechcommands_extracted_dir(root: Path) -> Path:
    return root / "SpeechCommands" / "speech_commands_v0.02"

def speechcommands_is_complete(root: Path) -> bool:
    extracted = speechcommands_extracted_dir(root)
    return (
        extracted.is_dir()
        and all((extracted / name).is_file() for name in SPEECHCOMMANDS_REQUIRED_FILES)
        and all((extracted / name).is_dir() for name in SPEECHCOMMANDS_REQUIRED_DIRS)
    )

def reset_incomplete_speechcommands(root: Path) -> None:
    if speechcommands_is_complete(root):
        return
    extracted_parent = root / "SpeechCommands"
    if extracted_parent.exists():
        print("Removing incomplete Speech Commands extraction:", extracted_parent)
        shutil.rmtree(extracted_parent)

def remove_incomplete_speechcommands_archive(root: Path) -> None:
    archive_path = root / SPEECHCOMMANDS_ARCHIVE
    if archive_path.exists():
        print("Removing incomplete Speech Commands archive:", archive_path)
        archive_path.unlink()

def ensure_speechcommands_downloaded(root: Path) -> dict[str, int]:
    root.mkdir(parents=True, exist_ok=True)
    reset_incomplete_speechcommands(root)
    if not speechcommands_is_complete(root):
        print("Downloading Speech Commands v0.02 into:", root)
        try:
            SPEECHCOMMANDS(root=str(root), download=True, subset="validation")
        except Exception:
            reset_incomplete_speechcommands(root)
            remove_incomplete_speechcommands_archive(root)
            SPEECHCOMMANDS(root=str(root), download=True, subset="validation")

    if not speechcommands_is_complete(root):
        raise RuntimeError(
            "Speech Commands download/extraction did not produce validation_list.txt, "
            "testing_list.txt, and required label directories under "
            f"{speechcommands_extracted_dir(root)}"
        )

    split_sizes = {}
    for subset in ("training", "validation", "testing"):
        dataset = SPEECHCOMMANDS(root=str(root), download=False, subset=subset)
        split_sizes[subset] = len(dataset)
    return split_sizes

speechcommands_counts = ensure_speechcommands_downloaded(Path(os.environ["SPEECHCOMMANDS_DATA_ROOT"]))
print("Speech Commands split sizes:", speechcommands_counts)


In [ ]:
base = make_experiment(tag="hash_kws12_iterlab_v1", vocabulary_preset="kws12")
recipes = build_hashednet95_recipe_book(base)
print("Recipes:")
for name in recipes:
    print(" -", name)

experiment = recipes[SELECTED_RECIPE]
if DRIVE_CACHE_ACTIVE:
    experiment = with_drive_cache_paths(experiment, DRIVE_CACHE_ROOT)

if SMOKE_MODE:
    experiment = replace(
        experiment,
        dataset=replace(
            experiment.dataset,
            train_limit=2048,
            val_limit=512,
            test_limit=512,
        ),
        train=replace(
            experiment.train,
            teacher_epochs=min(experiment.train.teacher_epochs, 1),
            student_pretrain_epochs=min(experiment.train.student_pretrain_epochs, 1),
            student_epochs=1,
            student_polish_epochs=0,
        ),
    )

torch.manual_seed(experiment.train.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(experiment.train.seed)

description = describe_hashednet95_recipe(experiment)
print(json.dumps({k: v for k, v in description.items() if k != "student_summary"}, indent=2))
print("Student summary:")
print(json.dumps(description["student_summary"], indent=2))


In [ ]:
t0 = time.perf_counter()
bundle = prepare_dataloaders(PROJECT_ROOT, experiment, device=device)
loaders = bundle["loaders"]

run_dir = initialize_run_state(
    PROJECT_ROOT,
    experiment,
    recipe_name=SELECTED_RECIPE,
    dataset_summary=bundle["summary"],
)
save_json_artifact(run_dir, "experiment.json", experiment.to_dict())
save_json_artifact(run_dir, "hashednet95_description.json", description)
save_json_artifact(run_dir, "dataset_summary.json", bundle["summary"])
save_text_artifact(run_dir, "selected_recipe.txt", SELECTED_RECIPE + "\n")
add_note(
    run_dir,
    "HashedNet95 setup",
    "Drive cache is used for Speech Commands and exact feature cache."
    if USE_GOOGLE_DRIVE_CACHE else "Drive cache disabled.",
)

print("Data root:", bundle["data_root"])
print("Run dir:", run_dir)
print("Dataset summary:")
print(json.dumps(bundle["summary"], indent=2))
print("Cache summary:")
print(json.dumps(bundle.get("cache_summary", {}), indent=2))
print("Feature preview shape:", bundle["feature_preview_shape"])
print("Data/cache prepare seconds:", round(time.perf_counter() - t0, 1))


In [ ]:
teacher = build_teacher_model(experiment)
student = build_student_model(experiment)

student_summary = summarize_model(student, experiment)
student_summary_path = save_model_summary(
    run_dir,
    "student",
    str(student) + "\n\n" + json.dumps(student_summary, indent=2),
)
save_json_artifact(run_dir, "student_model_inventory.json", student_summary)
print("Student params:", {
    "trainable": student_summary["trainable_parameters"],
    "compact": student_summary["hash_compact_parameters"],
    "virtual": student_summary["virtual_dense_parameters"],
    "maccs": student_summary["maccs_rough"],
})

teacher_summary_path = None
teacher_reuse_info = {}
if teacher is not None:
    if TEACHER_CHECKPOINT_PATH:
        teacher_reuse_info = load_model_checkpoint(teacher, TEACHER_CHECKPOINT_PATH, device=device)
        add_note(
            run_dir,
            "Teacher checkpoint reuse",
            json.dumps(teacher_reuse_info, indent=2),
        )
    teacher_summary = summarize_model(teacher, experiment)
    teacher_summary_path = save_model_summary(
        run_dir,
        "teacher",
        str(teacher) + "\n\n" + json.dumps(teacher_summary, indent=2),
    )
    save_json_artifact(run_dir, "teacher_model_inventory.json", teacher_summary)
    print("Teacher enabled.")
else:
    print("Teacher disabled.")


In [ ]:
teacher_test_metrics = {}
if teacher is not None:
    if teacher_reuse_info and not FORCE_TEACHER_RETRAIN:
        teacher_result = {
            "history": [],
            "best_state": {k: v.detach().clone() for k, v in teacher.state_dict().items()},
            "best_val_accuracy": None,
            "best_epoch": None,
            "elapsed_sec": 0.0,
        }
        teacher_history_path = save_history(run_dir, "teacher", [])
        teacher_plot_paths = []
    else:
        teacher_result = train_teacher(teacher, loaders=loaders, experiment=experiment, device=device)
        teacher.load_state_dict(teacher_result["best_state"], strict=True)
        teacher_history_path = save_history(run_dir, "teacher", teacher_result["history"])
        teacher_plot_paths = save_history_plots(run_dir, "teacher", teacher_result["history"])
    torch.save(
        {
            "experiment": experiment.to_dict(),
            "state_dict": {k: v.detach().cpu().clone() for k, v in teacher.state_dict().items()},
            "result": teacher_result,
        },
        run_dir / "teacher_best.pt",
    )
    teacher_test_metrics = evaluate(
        teacher,
        loaders["test"],
        device=device,
        top_k=experiment.train.top_k,
        use_amp=experiment.train.use_amp,
        desc="teacher | test",
    )
    save_metrics(run_dir, "teacher", teacher_test_metrics)
    update_stage_state(
        run_dir,
        "teacher",
        metrics=teacher_test_metrics,
        history_path=str(teacher_history_path),
        plot_paths=teacher_plot_paths,
        summary_path=str(teacher_summary_path) if teacher_summary_path else "",
        extra={
            "best_val_accuracy": teacher_result["best_val_accuracy"],
            "best_epoch": teacher_result["best_epoch"],
            "elapsed_sec": teacher_result["elapsed_sec"],
            "checkpoint_reuse": teacher_reuse_info,
            "force_teacher_retrain": FORCE_TEACHER_RETRAIN,
        },
    )
    print("Teacher test:", teacher_test_metrics)


In [ ]:
student_result = train_student(
    student,
    loaders=loaders,
    experiment=experiment,
    device=device,
    teacher=teacher,
)
student.load_state_dict(student_result["best_state"], strict=True)
torch.save(
    {
        "experiment": experiment.to_dict(),
        "state_dict": {k: v.detach().cpu().clone() for k, v in student.state_dict().items()},
        "result": student_result,
    },
    run_dir / "student_best.pt",
)

student_history_path = save_history(run_dir, "student", student_result["history"])
student_plot_paths = save_history_plots(run_dir, "student", student_result["history"])
save_metrics(run_dir, "student", student_result["test_metrics"])
update_stage_state(
    run_dir,
    "student",
    metrics=student_result["test_metrics"],
    history_path=str(student_history_path),
    plot_paths=student_plot_paths,
    summary_path=str(student_summary_path),
    extra={"stage_summaries": student_result["stage_summaries"]},
)
print("Student test:", student_result["test_metrics"])


In [ ]:
export_metadata = export_model_bundle(student, experiment=experiment, stage_name="student")
record_export_artifacts(run_dir, export_metadata)
save_json_artifact(run_dir, "export_metadata_snapshot.json", export_metadata)

firmware_export = firmware_exporter.export_bundle_to_firmware(
    bundle_path=Path(export_metadata["bundle"]["path"]),
    output_dir=PROJECT_ROOT / "code" / "firmware" / "hash_kws_runtime",
    project_root=PROJECT_ROOT,
    device=device,
    calibration_split="validation",
    calibration_batches=8,
)
record_export_artifacts(run_dir, {"firmware_export": firmware_export})
save_json_artifact(run_dir, "firmware_export_snapshot.json", firmware_export)
print(json.dumps(firmware_export, indent=2))


In [ ]:
summary_path = write_run_summary(run_dir)
archive_path = shutil.make_archive(str(run_dir.parent / f"{run_dir.name}_bundle"), "zip", root_dir=run_dir)
print("Summary:", summary_path)
print("Archive:", archive_path)

if DRIVE_CACHE_ACTIVE:
    drive_run_dir = DRIVE_CACHE_ROOT / "runs"
    drive_run_dir.mkdir(parents=True, exist_ok=True)
    target = drive_run_dir / Path(archive_path).name
    shutil.copy2(archive_path, target)
    print("Copied run archive to Drive:", target)


## Recommended Sweep Order

1. `hn95_kd128_cached_schedule` - primary next run: current 128-channel student plus cached logits and cosine KD schedule.
2. `hn95_kd128_hard_polish_cached` - same deploy shape, lower CE smoothing and longer hard-label polish.
3. `hn95_kd128_pw2048_budget_cached_schedule` - same MACs, larger pointwise codebooks to test collision pressure.
4. `hn95_kd112_latency_balanced_cached_schedule` - lower-latency guardrail if 128-channel firmware timing is too high.
5. `hn95_kd128_rich_teacher160_cached_schedule` - training-only richer teacher, same 128-channel deploy student.
6. `hn95_kd128_cached_schedule_s29` and `hn95_kd128_cached_schedule_s47` - training-seed repeat validation for the best candidate.
